# SurfaceGuard AI — KAGGLE V18: Frequency-Detail Model + Evidence-Gated Hybrid

**Goal:** build a stronger candidate against the current best public result, **0.65777 (user-reported Kaggle Version 11)**. The target of 0.68 remains aspirational; no local validation result can guarantee a higher public score.

## Models and inputs

The four proven base checkpoints are V4 U-Net/ResNet34, V5 SegFormer-B0, V6 DeepLabV3+/ResNet50, and MiT-B4 SegFormer. The four auxiliary Colab V7 checkpoints are also required for the eight-model hybrid:

- `model_a_scratch.pt` — ResNet34 U-Net trained from scratch + image-level classifier
- `model_b_pretrained.pt` — ImageNet-pretrained ResNet34 U-Net + image-level classifier
- `model_c_segformer.pt` — SegFormer MiT-B0 + pooled-logit classifier
- `model_d_deeplabv3plus.pt` — DeepLabV3+/ResNet50 + image-level classifier

Attach the competition dataset, `shrutikarthikeyan/v4-v5-v6-mit`, and the Kaggle dataset containing all four Colab V7 checkpoint files. Google Drive itself is not mounted by Kaggle. The notebook checks the exact checkpoint files before expensive inference.

## V18 changes from the executed V17 candidate
Keeps V17's metric-audited eight-model hybrid and expanded beta/gate plus probability/logit fusion search. It keeps the V17 final component filter disabled. In the uploaded V17 run, the component-filter candidate scored 0.724990 on the diagnostic holdout versus 0.724679 without filtering, but the final V17 configuration retained the unfiltered path; V18 does not silently change that prior decision.
Adds a finer 0.05 pairwise-transfer search for the four proven base-model weights, centered on the two strongest global candidates from the existing search. The resulting candidates are also eligible for the conservative class-aware weight selection.
Uses four-way horizontal/vertical flip TTA when a usable Kaggle GPU is enabled. CPU fallback remains supported, but intentionally runs one pass and is not equivalent to the full TTA run.
Updates the comparison baseline to the current best public result: 0.65777 (user-reported Kaggle Version 11).
Preserves all eight existing checkpoints unchanged and adds one optional, separately trained frequency/detail model.
## Fusion formulas

For class k, the base four-model probability is

\[P_{base,k}=w_{4,k}P_{V4,k}+w_{5,k}P_{V5,k}+w_{6,k}P_{V6,k}+w_{M,k}P_{MiT,k},\quad \sum_m w_{m,k}=1.\]

The retained V17 hybrid uses arithmetic probability fusion:

\[P_k=(1-\beta_k)P_{base,k}+\beta_kP_{aux,k}\]

and a local logit-pooling alternative:

\[P_k=\sigma((1-\beta_k)\operatorname{logit}(P_{base,k})+\beta_k\operatorname{logit}(P_{aux,k})).\]

The final mask is

\[\hat M_k=\mathbf 1[P_k\geq\tau_k]\;\mathbf 1[q_{aux,k}\geq\gamma_k],\]

where \(\tau_k\) is the pixel threshold and \(\gamma_k\) is the optional image-level classifier gate. The search keeps an explicit no-auxiliary baseline (beta 0, no gate).

## Competition metric

\[D=\operatorname{mean}_{(i,k):|G_{ik}|>0}\frac{2|P_{ik}\cap G_{ik}|}{|P_{ik}|+|G_{ik}|}\]

\[S_{FP}=1-\frac{\sum_{(i,k):|G_{ik}|=0}|P_{ik}|}{N_{empty}HW},\qquad A_{empty}=\operatorname{mean}_{(i,k):|G_{ik}|=0}\mathbf 1[|P_{ik}|=0]\]

\[Score=0.70D+0.20S_{FP}+0.10A_{empty}.\]

All candidate comparisons use the global pair denominators. The CAL/HOLDOUT split remains stratified by the four-class presence combination. Parameters are selected only on CAL_IDX; HOLDOUT_IDX is used for the diagnostic comparison. The full-validation threshold refit is an in-sample calibration score, not an independent estimate.

## Runtime and submission

- CUDA is used when available with at least 6 GiB free; otherwise CPU fallback is automatic.
- GPU mode uses four-way flip TTA; CPU mode uses one pass to avoid an impractical fourfold CPU runtime.
- If CUDA runs out of memory, the notebook moves models to CPU and continues.
- The notebook generates `/kaggle/working/submission.csv`, a versioned backup, the hybrid configuration, blend/gate/component search CSVs, a validation report, and submission/RLE checks.
- Use a Kaggle GPU accelerator for the expanded search when possible. CPU mode is supported but can take hours.

**Keep the Version 11 result submission (0.65777) unchanged.** V18 creates a separate candidate and must be judged by its new Kaggle public score; a stronger local validation score is not a guarantee.


## Run settings that matter
1. In Kaggle notebook settings, set **Accelerator = GPU** before `Save Version → Save & Run All` to activate four-way TTA. The uploaded V17 execution ran on a Tesla T4 and used four-way flip TTA. CPU fallback uses one pass.
2. Keep the competition data, `v4-v5-v6-mit`, and `surfaceguard-colab-v7-checkpoints` datasets attached.
3. V18 writes a separate versioned candidate and validation report. Do not replace the Version 11 result submission (0.65777) unless the new public score is higher.


## New V18 model: frequency/detail branch

V18 trains one additional four-class segmentation model on the original training partition only. It combines a compact multi-scale encoder-decoder with a fixed Laplacian/Sobel response branch, learned detail features, and an auxiliary image-level class head. This is a custom frequency-aware design inspired by the general industrial-defect literature; it is not a reproduction of any one paper or pretrained foundation model.

The new model is not automatically added to the eight-model blend. Its class-specific blend weights and thresholds are selected on CAL_IDX only. The chosen candidate is then checked on HOLDOUT_IDX. The new model is accepted only when the total holdout objective improves by a minimum margin and defect_b foreground Dice remains within the configured guardrail. Otherwise V18 retains the original V17 hybrid predictions.

The frequency model trains only when a suitable CUDA GPU is available. CPU-only or failed training falls back to the original eight-model V17 inference path so a submission can still be generated. Training time and public score are not guaranteed.


## Execution order and safe fallback

1. Attach the competition data, `v4-v5-v6-mit`, and `surfaceguard-colab-v7-checkpoints` datasets as before.
2. Enable a Kaggle GPU (Tesla T4 or better). The frequency model is skipped on CPU; the V17 ensemble fallback remains available.
3. Run all cells from the top. V18 first trains on a fit/monitor split made exclusively from the original 80% training partition, then caches validation predictions.
4. The eight saved checkpoints are loaded and calibrated exactly as in V17. V18 then selects the new model's per-class blend on CAL_IDX and checks the resulting choice on HOLDOUT_IDX.
5. Check the printed `V18 frequency blend active` decision before submitting. The produced `/kaggle/working/submission.csv` is a new candidate; it does not replace the previous Kaggle submission.
6. Compare its public score with **0.65777**. Keep the 0.65777 submission as the best result unless Kaggle reports a higher score.


For the optional V18 detail model, the final candidate map is

\[P_{V18,k}=(1-\alpha_k)P_{V17,k}+\alpha_kP_{detail,k}.\]

The `alpha` search is restricted to 0.05–0.30; `alpha=0` preserves the previous V17 result. The class thresholds are selected using CAL_IDX and then evaluated once on HOLDOUT_IDX before the new model is allowed into test inference.


## Research basis and implementation boundary

The frequency/detail direction is motivated by recent industrial-segmentation research: FDHNet (9 October 2026) studies wavelet-based frequency decomposition and high-order spatial relationships; HCFNet (20 July 2026) combines a Laplacian-enhanced branch with cross-branch multi-scale fusion; FH-Net (16 September 2026) studies frequency-domain and hierarchical enhancement for few-shot surface-defect segmentation. V18 adopts only the broad design idea of making local high-frequency structure explicit. Its fixed Laplacian/Sobel branch and compact U-Net are a lightweight, original implementation for this 128×800 grayscale competition, not a reproduction of those papers. It does not use SAM2, hypergraphs, learnable wavelet banks, or their reported benchmark scores.

- FDHNet, *Knowledge-Based Systems*, DOI: [10.1016/j.knosys.2026.116853](https://doi.org/10.1016/j.knosys.2026.116853)
- HCFNet, *Sensors*, DOI: [10.3390/s26144597](https://doi.org/10.3390/s26144597)
- FH-Net, *PLOS ONE*, DOI: [10.1371/journal.pone.0358329](https://doi.org/10.1371/journal.pone.0358329)


In [1]:
# Install SMP without resolving/replacing Kaggle's preinstalled PyTorch/CUDA stack.
# Kaggle GPU images already provide torch/torchvision; resolving dependencies can
# otherwise replace that build with a CPU wheel.
%pip install -q --disable-pip-version-check --no-deps segmentation-models-pytorch==0.6.0

from importlib.metadata import version
import torch

smp_version = version('segmentation-models-pytorch')
print('Installed SMP version:', smp_version)
if smp_version != '0.6.0':
    print(f'NOTE: SMP version is {smp_version}; continuing because the checkpoint architecture is loaded strictly below.')
print('PyTorch:', torch.__version__)
print('PyTorch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())

import segmentation_models_pytorch as smp
print('SMP import successful.')

if torch.cuda.is_available():
    try:
        print('GPU:', torch.cuda.get_device_name(0))
    except Exception as exc:
        print('GPU probe warning:', type(exc).__name__, str(exc)[:200])
else:
    print('GPU unavailable at startup; the notebook will use CPU mode automatically.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.6/154.6 kB 1.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Installed SMP version: 0.6.0
PyTorch: 2.11.0+cu128
PyTorch CUDA build: 12.8
CUDA available: True
SMP import successful.
GPU: Tesla T4


In [2]:
import os
import gc
import json
import time
import math
import shutil
import zipfile
import hashlib
import warnings
from pathlib import Path, PurePosixPath
from contextlib import nullcontext

import numpy as np
import pandas as pd
from scipy import ndimage
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import segmentation_models_pytorch as smp
from transformers import SegformerConfig, SegformerForSemanticSegmentation

SEED = 42
CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
HEIGHT, WIDTH = 128, 800
N_CLASSES = len(CLASSES)
N_PIXELS = HEIGHT * WIDTH
BEST_PUBLIC_SCORE_BASELINE = 0.65777  # User-reported Kaggle Version 11
LOCAL_VALIDATION_TARGET = 0.675  # Historical reference only; not a public-score guarantee
PUBLIC_SCORE_TARGET = 0.68

random_state = np.random.RandomState(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Select CUDA only when it is genuinely usable and has enough free memory for
# four loaded segmentation models plus inference activations. Otherwise use CPU.
MIN_FREE_GPU_GB = 6.0
DEVICE = torch.device("cpu")
GPU_NAME = "CPU fallback"
GPU_FREE_GB = 0.0
GPU_TOTAL_GB = 0.0

if torch.cuda.is_available():
    try:
        torch.cuda.init()
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        GPU_FREE_GB = free_bytes / (1024 ** 3)
        GPU_TOTAL_GB = total_bytes / (1024 ** 3)
        if GPU_FREE_GB >= MIN_FREE_GPU_GB:
            _probe = torch.empty((1,), device="cuda:0")
            torch.cuda.synchronize()
            del _probe
            DEVICE = torch.device("cuda:0")
            GPU_NAME = torch.cuda.get_device_name(0)
        else:
            print(
                f"CUDA detected but only {GPU_FREE_GB:.2f} GiB is free; "
                f"minimum is {MIN_FREE_GPU_GB:.1f} GiB. Selecting CPU fallback."
            )
    except Exception as exc:
        DEVICE = torch.device("cpu")
        GPU_NAME = "CPU fallback (CUDA initialisation failed)"
        print("CUDA initialisation failed; selecting CPU fallback:", type(exc).__name__, str(exc)[:250])

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    # Avoid cuDNN's benchmark workspace spikes; fixed input dimensions do not
    # require benchmark mode for reliable inference.
    torch.backends.cudnn.benchmark = False
    MAIN_BATCH_SIZE = 1
    MIT_BATCH_SIZE = 1
    TEST_BATCH_SIZE = 1
    NUM_WORKERS = 2
else:
    # CPU mode uses single-image batches and no worker processes for reliability.
    MAIN_BATCH_SIZE = 1
    MIT_BATCH_SIZE = 1
    TEST_BATCH_SIZE = 1
    NUM_WORKERS = 0
    try:
        torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass  # PyTorch may already have initialised its thread pools.


def _is_cuda_oom_error(exc):
    oom_type = getattr(torch.cuda, "OutOfMemoryError", None)
    if oom_type is not None and isinstance(exc, oom_type):
        return True
    message = str(exc).lower()
    return ("out of memory" in message and ("cuda" in message or "gpu" in message)) or "cuda error: out of memory" in message


def _rebuild_loaders_for_cpu():
    """Switch existing, already-created DataLoaders to safe CPU settings."""
    global val_loader_main, val_loader_mit, test_loader_main, test_loader_mit
    global MAIN_BATCH_SIZE, MIT_BATCH_SIZE, TEST_BATCH_SIZE, NUM_WORKERS
    MAIN_BATCH_SIZE = MIT_BATCH_SIZE = TEST_BATCH_SIZE = 1
    NUM_WORKERS = 0
    loader_specs = [
        ("val_dataset_main", "val_loader_main", MAIN_BATCH_SIZE),
        ("val_dataset_mit", "val_loader_mit", MIT_BATCH_SIZE),
        ("test_dataset_main", "test_loader_main", TEST_BATCH_SIZE),
        ("test_dataset_mit", "test_loader_mit", TEST_BATCH_SIZE),
    ]
    for dataset_name, loader_name, batch_size in loader_specs:
        dataset = globals().get(dataset_name)
        if dataset is not None:
            globals()[loader_name] = DataLoader(
                dataset, batch_size=batch_size, shuffle=False,
                num_workers=0, pin_memory=False, drop_last=False
            )


def switch_to_cpu_fallback(reason, extra_models=()):
    """Move every loaded model to CPU after a CUDA OOM; return whether switched."""
    global DEVICE, GPU_NAME
    if DEVICE.type != "cuda":
        return False
    print("\nCUDA memory/runtime allocation failed. Switching all models to CPU.")
    print("Reason:", str(reason)[:300])
    candidates = list(extra_models)
    for model_name in (
        "model_v4", "model_v5", "model_v6", "model_mit",
        "model_colab_a", "model_colab_b", "model_colab_c", "model_colab_d",
        "frequency_test_model"
    ):
        candidate = globals().get(model_name)
        if isinstance(candidate, torch.nn.Module):
            candidates.append(candidate)
    seen = set()
    for model in candidates:
        if model is None or id(model) in seen:
            continue
        seen.add(id(model))
        try:
            model.to(torch.device("cpu"))
            model.eval()
        except Exception as move_exc:
            raise RuntimeError(
                "CUDA fallback was triggered, but a loaded model could not be moved to CPU."
            ) from move_exc
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    DEVICE = torch.device("cpu")
    GPU_NAME = "CPU fallback (automatic CUDA OOM recovery)"
    try:
        torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass
    _rebuild_loaders_for_cpu()
    print("Fallback complete. All available model objects are on CPU.")
    return True

OUTPUT_DIR = Path("/kaggle/working/surfaceguard_version18_frequency_detail_hybrid")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Validation prediction caches are temporary disk-backed arrays. They are not loaded
# fully into RAM and are not retained as large notebook output artifacts.
CACHE_DIR = Path("/kaggle/temp/surfaceguard_version18_frequency_detail_hybrid_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("SurfaceGuard V18 frequency-detail + hybrid inference pipeline")
print("GPU:", GPU_NAME)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("SMP:", smp.__version__)
print("Classes:", CLASSES)
print("Image size:", (HEIGHT, WIDTH))
print("Local validation reference:", LOCAL_VALIDATION_TARGET)
print("Current best public baseline:", BEST_PUBLIC_SCORE_BASELINE)
print("Public leaderboard goal:", PUBLIC_SCORE_TARGET)
print("Execution device:", DEVICE, "|", GPU_NAME)
if DEVICE.type == "cuda":
    print(f"GPU memory at startup: {GPU_FREE_GB:.2f}/{GPU_TOTAL_GB:.2f} GiB free/total")
    print("GPU mode: mixed-precision inference with conservative batch size 1.")
else:
    print("CPU fallback mode: safe batch size 1, zero DataLoader workers; TTA is reduced to one pass to keep runtime manageable.")
    print("NOTE: for the full V18 four-way TTA candidate, enable a Kaggle GPU accelerator before running all cells.")
print("Output directory:", OUTPUT_DIR)
print("Optional Colab V7 auxiliary models will be included only when all four checkpoint files are attached.")
print("The optional V18 frequency/detail model trains before the eight existing checkpoints are loaded. The established V17 hybrid remains the safe fallback.")


SurfaceGuard V18 frequency-detail + hybrid inference pipeline
GPU: Tesla T4
PyTorch: 2.11.0+cu128
PyTorch CUDA build: 12.8
SMP: 0.6.0
Classes: ['defect_a', 'defect_b', 'defect_c', 'defect_d']
Image size: (128, 800)
Local validation reference: 0.675
Current best public baseline: 0.65777
Public leaderboard goal: 0.68
Execution device: cuda:0 | Tesla T4
GPU memory at startup: 14.46/14.56 GiB free/total
GPU mode: mixed-precision inference with conservative batch size 1.
Output directory: /kaggle/working/surfaceguard_version18_frequency_detail_hybrid
Optional Colab V7 auxiliary models will be included only when all four checkpoint files are attached.
The optional V18 frequency/detail model trains before the eight existing checkpoints are loaded. The established V17 hybrid remains the safe fallback.


In [3]:
INPUT_ROOT = Path("/kaggle/input")
MODEL_CACHE_DIR = Path("/kaggle/temp/surfaceguard_model_inputs")
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_ROOT.exists():
    raise FileNotFoundError("/kaggle/input does not exist. Attach the required Kaggle inputs.")

REQUIRED_ARTIFACTS = {
    "best_v6_deeplabv3plus.pt",
    "best_mit_b4.pt",
    "mit_b4_holdout_report.json",
    "best_v4.pt",
    "best_v5.pt",
    # Optional Colab V7 auxiliary-classifier models. Extracted only when present.
    "model_a_scratch.pt",
    "model_b_pretrained.pt",
    "model_c_segformer.pt",
    "model_d_deeplabv3plus.pt",
}
ARCHIVE_MARKERS = {
    "data.pkl", "version", "byteorder", ".format_version", ".storage_alignment"
}


def _safe_zip_parts(member_name):
    p = PurePosixPath(member_name)
    if p.is_absolute() or ".." in p.parts:
        return None
    return p.parts


def _copy_zip_member(zf, member, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with zf.open(member, "r") as source, open(destination, "wb") as target:
        shutil.copyfileobj(source, target)


# Extract direct model/report files and split V4/V5 torch archives from attached ZIPs.
zip_extractions = []
for zip_path in sorted(INPUT_ROOT.rglob("*.zip")):
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            zip_identity = hashlib.sha256(
                str(zip_path.resolve()).encode("utf-8")
            ).hexdigest()[:12]
            stage_root = (
                MODEL_CACHE_DIR / "zip_extracted" / f"{zip_path.stem}_{zip_identity}"
            )
            for info in zf.infolist():
                if info.is_dir():
                    continue
                parts = _safe_zip_parts(info.filename)
                if not parts:
                    continue
                basename = parts[-1]

                if basename in REQUIRED_ARTIFACTS:
                    # Preserve the member path: two different artifacts with the same
                    # basename must be surfaced as duplicates, never silently ignored.
                    target = stage_root / "artifacts" / Path(*parts)
                    if not target.exists():
                        _copy_zip_member(zf, info, target)
                        zip_extractions.append(str(target))
                    continue

                # A PyTorch split archive has files such as best_v4/data.pkl,
                # best_v4/version, best_v4/data/0, best_v4/.data/serialization_id.
                for model_name in ("best_v4", "best_v5"):
                    matching = [i for i, part in enumerate(parts) if part == model_name]
                    if not matching:
                        continue
                    root_index = matching[-1]  # use the innermost best_v4/best_v5 folder
                    relative_parts = parts[root_index:]
                    if len(relative_parts) < 2:
                        break
                    relative_path = PurePosixPath(*relative_parts)
                    target = stage_root / "archives" / Path(*relative_path.parts)
                    if not target.exists():
                        _copy_zip_member(zf, info, target)
                        zip_extractions.append(str(target))
                    break
    except zipfile.BadZipFile:
        print("Skipping non-ZIP or damaged ZIP:", zip_path)

print("Extracted checkpoint artifacts:", len(zip_extractions))


def _sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_unique_file(filename, required=True):
    hits = []
    for root in (INPUT_ROOT, MODEL_CACHE_DIR):
        if root.exists():
            hits.extend(p for p in root.rglob(filename) if p.is_file())
    hits = list({str(p.resolve()): p for p in hits}.values())
    if not hits:
        if required:
            raise FileNotFoundError(
                f"Required artifact '{filename}' was not found under /kaggle/input. "
                "Attach the correct Notebook Output Files or a ZIP containing this exact file."
            )
        return None
    if len(hits) == 1:
        return hits[0]

    sizes = {p.stat().st_size for p in hits}
    if len(sizes) == 1:
        hashes = {_sha256(p) for p in hits}
        if len(hashes) == 1:
            return sorted(hits, key=lambda p: (len(str(p)), str(p)))[0]

    listing = "\n".join(f"{p} ({p.stat().st_size:,} bytes)" for p in hits)
    raise RuntimeError(
        f"Multiple different files named {filename} were found. Keep only the intended "
        f"checkpoint version attached. Candidates:\n{listing}"
    )


def reconstruct_v4_v5_archive(model_name):
    """Return direct .pt checkpoint or reconstruct it from a split PyTorch archive folder."""
    direct = find_unique_file(f"{model_name}.pt", required=False)
    if direct is not None:
        return direct

    roots = []
    for search_root in (INPUT_ROOT, MODEL_CACHE_DIR):
        if search_root.exists():
            for data_pickle in search_root.rglob("data.pkl"):
                archive_root = data_pickle.parent
                if archive_root.name == model_name and (archive_root / "version").is_file():
                    roots.append(archive_root)
    roots = list({str(p.resolve()): p for p in roots}.values())

    if not roots:
        raise FileNotFoundError(
            f"Could not find {model_name}.pt or its split archive folder. "
            "Attach surfaceguard-v4-v5-checkpoints or a ZIP containing "
            f"{model_name}/data.pkl, {model_name}/version and {model_name}/data/*."
        )

    if len(roots) > 1:
        preferred = [p for p in roots if "surfaceguard-v4-v5-checkpoints" in str(p)]
        if len(preferred) == 1:
            source = preferred[0]
        else:
            raise RuntimeError(
                f"Multiple candidate split archives were found for {model_name}: {roots}. "
                "Keep only the intended checkpoint dataset/ZIP attached."
            )
    else:
        source = roots[0]

    output_path = MODEL_CACHE_DIR / f"{model_name}.pt"
    with zipfile.ZipFile(output_path, "w", compression=zipfile.ZIP_STORED) as zf:
        for file_path in source.rglob("*"):
            if file_path.is_file():
                archive_name = (Path(model_name) / file_path.relative_to(source)).as_posix()
                zf.write(file_path, archive_name)

    if not zipfile.is_zipfile(output_path):
        raise RuntimeError(f"Reconstructed checkpoint is not a valid ZIP archive: {output_path}")
    return output_path


V4_PATH = reconstruct_v4_v5_archive("best_v4")
V5_PATH = reconstruct_v4_v5_archive("best_v5")
V6_PATH = find_unique_file("best_v6_deeplabv3plus.pt")
MIT_PATH = find_unique_file("best_mit_b4.pt")
MIT_REPORT_PATH = find_unique_file("mit_b4_holdout_report.json")


# V17: all four Colab V7 auxiliary checkpoints are REQUIRED for this hybrid run.
# Kaggle cannot read files directly from Google Drive. Create a Kaggle Dataset
# containing the four .pt files, attach it to this notebook, and rerun from the top.
COLAB_V7_FILENAMES = {
    "a_scratch": "model_a_scratch.pt",
    "b_pretrained": "model_b_pretrained.pt",
    "c_segformer_b0": "model_c_segformer.pt",
    "d_deeplabv3plus": "model_d_deeplabv3plus.pt",
}
COLAB_V7_PATHS = {
    key: find_unique_file(filename, required=False)
    for key, filename in COLAB_V7_FILENAMES.items()
}
COLAB_V7_MISSING = [
    COLAB_V7_FILENAMES[key] for key, path in COLAB_V7_PATHS.items() if path is None
]
COLAB_V7_AVAILABLE = len(COLAB_V7_MISSING) == 0

print("Colab V7 auxiliary checkpoint preflight")
for key, filename in COLAB_V7_FILENAMES.items():
    path = COLAB_V7_PATHS[key]
    if path is None:
        print(f"  MISSING: {filename}")
    else:
        print(f"  FOUND:   {filename} | {path} | {path.stat().st_size / (1024**2):.2f} MB")

if COLAB_V7_MISSING:
    discovered = sorted({p.name for p in INPUT_ROOT.rglob("*.pt")}) if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        "V17 requires all four Colab V7 auxiliary checkpoints for the eight-model hybrid.\\n"
        f"Missing files: {COLAB_V7_MISSING}\\n"
        "Kaggle cannot access your Google Drive directly. Create a Kaggle Dataset containing "
        "model_a_scratch.pt, model_b_pretrained.pt, model_c_segformer.pt and "
        "model_d_deeplabv3plus.pt; attach it to this notebook; then choose Save Version -> Save & Run All.\\n"
        f".pt files currently discovered under /kaggle/input: {discovered}"
    )

print("All four Colab V7 checkpoints found. Eight-model hybrid is enabled.")

preferred_data_root = Path(
    "/kaggle/input/competitions/industrial-surface-defect-detection/public"
)
if (preferred_data_root / "train.csv").is_file():
    DATA_ROOT = preferred_data_root
else:
    candidates = []
    for train_csv in INPUT_ROOT.rglob("train.csv"):
        parent = train_csv.parent
        if (parent / "train").is_dir() and (parent / "test").is_dir() and (parent / "sample_submission.csv").is_file():
            candidates.append(parent)
    candidates = list({str(p.resolve()): p for p in candidates}.values())
    if len(candidates) != 1:
        raise FileNotFoundError(
            "Could not uniquely locate competition data. Attach industrial-surface-defect-detection. "
            f"Candidates found: {candidates}"
        )
    DATA_ROOT = candidates[0]

print("\nINPUT PREFLIGHT PASSED")
for label, path in [
    ("V4", V4_PATH), ("V5", V5_PATH), ("V6", V6_PATH),
    ("MiT-B4", MIT_PATH), ("MiT report", MIT_REPORT_PATH),
    ("Competition data", DATA_ROOT)
]:
    print(f"{label:16s}: {path}")
    if Path(path).is_file():
        print(" " * 18, f"{Path(path).stat().st_size / (1024**2):.2f} MB")


Extracted checkpoint artifacts: 0
Colab V7 auxiliary checkpoint preflight
  FOUND:   model_a_scratch.pt | /kaggle/input/datasets/shrutikarthikeyan/surfaceguard-colab-v7-checkpoints/model_a_scratch.pt | 93.37 MB
  FOUND:   model_b_pretrained.pt | /kaggle/input/datasets/shrutikarthikeyan/surfaceguard-colab-v7-checkpoints/model_b_pretrained.pt | 93.37 MB
  FOUND:   model_c_segformer.pt | /kaggle/input/datasets/shrutikarthikeyan/surfaceguard-colab-v7-checkpoints/model_c_segformer.pt | 14.26 MB
  FOUND:   model_d_deeplabv3plus.pt | /kaggle/input/datasets/shrutikarthikeyan/surfaceguard-colab-v7-checkpoints/model_d_deeplabv3plus.pt | 102.12 MB
All four Colab V7 checkpoints found. Eight-model hybrid is enabled.

INPUT PREFLIGHT PASSED
V4              : /kaggle/input/datasets/shrutikarthikeyan/v4-v5-v6-mit/SurfaceGuard_V4_V5_Ensemble_output/reconstructed_checkpoints/best_v4.pt
                   279.81 MB
V5              : /kaggle/input/datasets/shrutikarthikeyan/v4-v5-v6-mit/SurfaceGuard_V4_V5

In [4]:
train_df = pd.read_csv(
    DATA_ROOT / "train.csv",
    dtype={"image_id": str, "defect_type": str}
)

required_columns = {"image_id", "defect_type", "mask_rle"}
if not required_columns.issubset(train_df.columns):
    raise ValueError(f"train.csv is missing required columns: {required_columns - set(train_df.columns)}")

train_df["image_id"] = train_df["image_id"].astype(str)
train_df["defect_type"] = train_df["defect_type"].astype(str)
train_df["mask_rle"] = train_df["mask_rle"].fillna("").astype(str)
train_df["has_defect"] = train_df["mask_rle"].str.strip().ne("")

presence = train_df.pivot_table(
    index="image_id",
    columns="defect_type",
    values="has_defect",
    aggfunc="max",
    fill_value=False
).reindex(columns=CLASSES, fill_value=False).astype(bool)

all_image_ids = presence.index.astype(str).to_numpy()
label_combo = presence.astype(int).astype(str).agg("".join, axis=1)

train_ids, val_ids = train_test_split(
    all_image_ids,
    test_size=0.20,
    random_state=SEED,
    stratify=label_combo.loc[all_image_ids]
)
train_ids = np.asarray(train_ids, dtype=str)
val_ids = np.asarray(val_ids, dtype=str)

if len(all_image_ids) != 11393 or len(train_ids) != 9114 or len(val_ids) != 2279:
    raise RuntimeError(
        f"Unexpected data split: total={len(all_image_ids)}, train={len(train_ids)}, val={len(val_ids)}. "
        "This is not the expected seed-42 80/20 split used by the saved models."
    )
if set(train_ids).intersection(set(val_ids)):
    raise RuntimeError("Training/validation image overlap detected.")

# Recreate the MiT-B4 V2 internal calibration/holdout split.
val_combo = label_combo.loc[val_ids]
try:
    calibration_ids, holdout_ids = train_test_split(
        val_ids,
        test_size=0.40,
        random_state=2026,
        stratify=val_combo.loc[val_ids]
    )
    INTERNAL_SPLIT_NOTE = "stratified by four-class presence combination"
except ValueError as exc:
    warnings.warn(f"Could not stratify internal validation split: {exc}. Using deterministic fallback.")
    calibration_ids, holdout_ids = train_test_split(
        val_ids,
        test_size=0.40,
        random_state=2026
    )
    INTERNAL_SPLIT_NOTE = "deterministic random fallback"

calibration_ids = np.asarray(calibration_ids, dtype=str)
holdout_ids = np.asarray(holdout_ids, dtype=str)
if set(calibration_ids).intersection(set(holdout_ids)):
    raise RuntimeError("Calibration and holdout IDs overlap.")
if set(calibration_ids).union(set(holdout_ids)) != set(val_ids):
    raise RuntimeError("Calibration/holdout split does not cover the full validation set.")

val_index_by_id = {image_id: index for index, image_id in enumerate(val_ids)}
CAL_IDX = np.asarray([val_index_by_id[x] for x in calibration_ids], dtype=np.int64)
HOLDOUT_IDX = np.asarray([val_index_by_id[x] for x in holdout_ids], dtype=np.int64)
ALL_VAL_IDX = np.arange(len(val_ids), dtype=np.int64)

val_positive_by_class = presence.loc[val_ids, CLASSES].sum(axis=0).astype(int).to_dict()
print("Competition data root:", DATA_ROOT)
print("Unique images:", len(all_image_ids))
print("Train / validation:", len(train_ids), "/", len(val_ids))
print("Calibration / holdout:", len(CAL_IDX), "/", len(HOLDOUT_IDX))
print("Internal split:", INTERNAL_SPLIT_NOTE)
print("Validation positive pairs by class:", val_positive_by_class)
print("Validation image-class pairs:", len(val_ids) * N_CLASSES)

Competition data root: /kaggle/input/competitions/industrial-surface-defect-detection/public
Unique images: 11393
Train / validation: 9114 / 2279
Calibration / holdout: 1367 / 912
Internal split: stratified by four-class presence combination
Validation positive pairs by class: {'defect_a': 216, 'defect_b': 63, 'defect_c': 820, 'defect_d': 342}
Validation image-class pairs: 9116


In [5]:
def rle_decode(rle, height=HEIGHT, width=WIDTH):
    flat = np.zeros(height * width, dtype=np.uint8)
    if rle is None or str(rle).strip() == "":
        return flat.reshape((height, width), order="F")
    values = np.asarray(str(rle).split(), dtype=np.int64)
    if len(values) % 2:
        raise ValueError("RLE must contain start-length pairs.")
    starts = values[0::2] - 1
    lengths = values[1::2]
    for start, length in zip(starts, lengths):
        if start < 0 or length <= 0 or start + length > height * width:
            raise ValueError("RLE run is outside the 128x800 image.")
        flat[start:start + length] = 1
    return flat.reshape((height, width), order="F")

mask_lookup = {
    (row.image_id, row.defect_type): row.mask_rle
    for row in train_df[["image_id", "defect_type", "mask_rle"]].itertuples(index=False)
}

# Match the original training preprocessing for every saved model family.
# V4, V5, V6 and MiT-B4 training datasets all used bilinear resizing.
class SurfaceGuardValidationDataset(Dataset):
    def __init__(self, image_ids, data_root, resample):
        self.image_ids = list(map(str, image_ids))
        self.data_root = Path(data_root)
        self.resample = resample

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_id = self.image_ids[index]
        image_path = self.data_root / "train" / f"{image_id}.jpg"
        with Image.open(image_path) as source:
            image = source.convert("L").resize((WIDTH, HEIGHT), resample=self.resample)
            image = np.asarray(image, dtype=np.float32) / 255.0

        mean = float(image.mean())
        std = max(float(image.std()), 0.05)
        image = np.ascontiguousarray((image - mean) / std, dtype=np.float32)

        masks = np.stack([
            rle_decode(mask_lookup.get((image_id, class_name), ""))
            for class_name in CLASSES
        ], axis=0).astype(np.uint8)

        return torch.from_numpy(image).unsqueeze(0), torch.from_numpy(masks), image_id


MAIN_RESAMPLE = Image.Resampling.BILINEAR
MIT_RESAMPLE = Image.Resampling.BILINEAR

val_dataset_main = SurfaceGuardValidationDataset(
    val_ids, DATA_ROOT, MAIN_RESAMPLE
)
val_dataset_mit = SurfaceGuardValidationDataset(
    val_ids, DATA_ROOT, MIT_RESAMPLE
)

val_loader_main = DataLoader(
    val_dataset_main,
    batch_size=MAIN_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=False
)
val_loader_mit = DataLoader(
    val_dataset_mit,
    batch_size=MIT_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=False
)

print("Main validation images:", len(val_dataset_main))
print("MiT validation images:", len(val_dataset_mit))
print("Main batches:", len(val_loader_main))
print("MiT batches:", len(val_loader_mit))

Main validation images: 2279
MiT validation images: 2279
Main batches: 2279
MiT batches: 2279


In [6]:
# V18 OPTIONAL TRAINING: frequency/detail-aware segmentation candidate.
# This cell runs before loading the eight existing models to keep GPU memory predictable.
# If there is no CUDA GPU, training fails, or the checkpoint cannot be validated, the notebook
# continues with the established V17 eight-model hybrid instead of blocking submission creation.

import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split as _train_test_split

FREQUENCY_MODEL_AVAILABLE = False
FREQUENCY_BLEND_ACTIVE = False
FREQUENCY_MODEL_PATH = OUTPUT_DIR / "v18_frequency_detail_model.pt"
FREQUENCY_VAL_PROBS_PATH = CACHE_DIR / "v18_frequency_val_probs.npy"
FREQUENCY_VAL_CLS_PATH = CACHE_DIR / "v18_frequency_val_cls.npy"
FREQUENCY_ALPHA_BY_CLASS = {name: 0.0 for name in CLASSES}
FREQUENCY_SELECTED_THRESHOLDS = None
FREQUENCY_HOLDOUT_SCORE = None
FREQUENCY_BASELINE_HOLDOUT_SCORE = None
FREQUENCY_TRAINING_NOTE = "not_started"


class _ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        groups = min(8, out_ch)
        while out_ch % groups != 0:
            groups -= 1
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride=stride, padding=pad, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)


class _ResidualGNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        groups = min(8, out_ch)
        while out_ch % groups != 0:
            groups -= 1
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.norm1 = nn.GroupNorm(groups, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(groups, out_ch)
        self.skip = (
            nn.Identity() if in_ch == out_ch and stride == 1 else
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), nn.GroupNorm(groups, out_ch))
        )
        self.act = nn.SiLU(inplace=True)
    def forward(self, x):
        residual = self.skip(x)
        x = self.act(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        return self.act(x + residual)


class _ConvFusionBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            _ConvGNAct(in_ch, out_ch),
            _ConvGNAct(out_ch, out_ch),
        )
    def forward(self, x):
        return self.block(x)


class SurfaceGuardFrequencyDetailNet(nn.Module):
    """Compact U-Net with a fixed high-frequency response branch and learned fusion."""
    def __init__(self, num_classes=4):
        super().__init__()
        self.register_buffer("edge_kernels", torch.tensor([
            [[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]],
            [[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]],
            [[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]],
        ], dtype=torch.float32).unsqueeze(1))

        # Main appearance/context stream.
        self.e0 = _ResidualGNBlock(1, 24, 1)
        self.e1 = _ResidualGNBlock(24, 40, 2)
        self.e2 = _ResidualGNBlock(40, 64, 2)
        self.e3 = _ResidualGNBlock(64, 96, 2)
        self.e4 = _ResidualGNBlock(96, 128, 2)

        # Fixed Laplacian/Sobel responses become four channels with the raw image.
        self.f0 = _ConvGNAct(4, 16)
        self.f1 = _ResidualGNBlock(16, 24, 2)
        self.f2 = _ResidualGNBlock(24, 32, 2)
        self.f3 = _ResidualGNBlock(32, 48, 2)
        self.f4 = _ResidualGNBlock(48, 64, 2)

        # Multi-scale detail-to-context fusion.
        self.x0 = _ConvFusionBlock(24 + 16, 24)
        self.x1 = _ConvFusionBlock(40 + 24, 40)
        self.x2 = _ConvFusionBlock(64 + 32, 64)
        self.x3 = _ConvFusionBlock(96 + 48, 96)
        self.x4 = _ConvFusionBlock(128 + 64, 128)

        # U-Net decoder.
        self.d3 = _ConvFusionBlock(128 + 96, 96)
        self.d2 = _ConvFusionBlock(96 + 64, 64)
        self.d1 = _ConvFusionBlock(64 + 40, 40)
        self.d0 = _ConvFusionBlock(40 + 24, 24)
        self.segmentation_head = nn.Conv2d(24, num_classes, kernel_size=1)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.SiLU(inplace=True), nn.Dropout(0.10),
            nn.Linear(64, num_classes)
        )

    def forward(self, image):
        edge = F.conv2d(image, self.edge_kernels.to(dtype=image.dtype), padding=1)
        detail = torch.cat([image, edge[:, 0:1].abs(), edge[:, 1:2].abs(), edge[:, 2:3].abs()], dim=1)
        f0 = self.f0(detail)
        f1 = self.f1(f0)
        f2 = self.f2(f1)
        f3 = self.f3(f2)
        f4 = self.f4(f3)

        e0 = self.x0(torch.cat([self.e0(image), f0], dim=1))
        e1 = self.x1(torch.cat([self.e1(e0), f1], dim=1))
        e2 = self.x2(torch.cat([self.e2(e1), f2], dim=1))
        e3 = self.x3(torch.cat([self.e3(e2), f3], dim=1))
        e4 = self.x4(torch.cat([self.e4(e3), f4], dim=1))

        d3 = F.interpolate(e4, size=e3.shape[-2:], mode="bilinear", align_corners=False)
        d3 = self.d3(torch.cat([d3, e3], dim=1))
        d2 = F.interpolate(d3, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.d2(torch.cat([d2, e2], dim=1))
        d1 = F.interpolate(d2, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.d1(torch.cat([d1, e1], dim=1))
        d0 = F.interpolate(d1, size=e0.shape[-2:], mode="bilinear", align_corners=False)
        d0 = self.d0(torch.cat([d0, e0], dim=1))
        logits = self.segmentation_head(d0)
        class_logits = self.classifier(F.adaptive_avg_pool2d(e4, 1).flatten(1))
        return logits, class_logits


class SurfaceGuardFrequencyDataset(Dataset):
    def __init__(self, image_ids, augment=False):
        self.image_ids = list(map(str, image_ids))
        self.augment = bool(augment)
    def __len__(self):
        return len(self.image_ids)
    def __getitem__(self, index):
        image_id = self.image_ids[index]
        image_path = DATA_ROOT / "train" / f"{image_id}.jpg"
        with Image.open(image_path) as source:
            image = source.convert("L").resize((WIDTH, HEIGHT), resample=Image.Resampling.BILINEAR)
            image = np.asarray(image, dtype=np.float32) / 255.0
        mean = float(image.mean())
        std = max(float(image.std()), 0.05)
        image = np.ascontiguousarray((image - mean) / std, dtype=np.float32)
        masks = np.stack([
            rle_decode(mask_lookup.get((image_id, class_name), ""))
            for class_name in CLASSES
        ], axis=0).astype(np.float32)
        if self.augment:
            if np.random.rand() < 0.50:
                image = np.flip(image, axis=1).copy()
                masks = np.flip(masks, axis=2).copy()
            if np.random.rand() < 0.50:
                image = np.flip(image, axis=0).copy()
                masks = np.flip(masks, axis=1).copy()
        presence = (masks.reshape(N_CLASSES, -1).sum(axis=1) > 0).astype(np.float32)
        return (
            torch.from_numpy(image).unsqueeze(0),
            torch.from_numpy(masks),
            torch.from_numpy(presence),
        )


def _frequency_predict_batch(model, images, device, use_tta=True):
    images = images.to(device, non_blocking=(device.type == "cuda"))
    transforms = [(False, False)]
    if use_tta:
        transforms = [(False, False), (True, False), (False, True), (True, True)]
    probabilities = []
    class_probabilities = []
    amp_enabled = device.type == "cuda"
    for flip_h, flip_v in transforms:
        x = images
        dims = []
        if flip_h: dims.append(3)
        if flip_v: dims.append(2)
        if dims: x = torch.flip(x, dims=dims)
        with torch.cuda.amp.autocast(enabled=amp_enabled):
            seg_logits, cls_logits = model(x)
        seg_probs = torch.sigmoid(seg_logits.float())
        cls_probs = torch.sigmoid(cls_logits.float())
        if dims:
            seg_probs = torch.flip(seg_probs, dims=dims)
        probabilities.append(seg_probs)
        class_probabilities.append(cls_probs)
    return torch.stack(probabilities).mean(0), torch.stack(class_probabilities).mean(0)


if DEVICE.type != "cuda":
    FREQUENCY_TRAINING_NOTE = "skipped: CUDA unavailable; established V17 hybrid fallback will run"
    print("V18 frequency/detail training skipped because CUDA is unavailable.")
else:
    try:
        # Never silently reuse a checkpoint from an earlier execution of this cell.
        if FREQUENCY_MODEL_PATH.exists():
            FREQUENCY_MODEL_PATH.unlink()
        torch.cuda.empty_cache()
        # Reserve a small monitor split from TRAIN_IDS only. CAL_IDX/HOLDOUT_IDX remain untouched.
        try:
            fit_ids, monitor_ids = _train_test_split(
                train_ids,
                test_size=0.10,
                random_state=SEED + 100,
                stratify=label_combo.loc[train_ids],
            )
            FREQUENCY_MONITOR_SPLIT_NOTE = "stratified by class-presence combination"
        except ValueError as split_error:
            warnings.warn(
                f"Could not stratify the small monitor split ({split_error}); "
                "using a deterministic random 90/10 split within TRAIN_IDS."
            )
            fit_ids, monitor_ids = _train_test_split(
                train_ids, test_size=0.10, random_state=SEED + 100
            )
            FREQUENCY_MONITOR_SPLIT_NOTE = "deterministic random fallback"
        fit_ids = np.asarray(fit_ids, dtype=str)
        monitor_ids = np.asarray(monitor_ids, dtype=str)
        if set(fit_ids) & set(monitor_ids) or set(fit_ids) & set(val_ids) or set(monitor_ids) & set(val_ids):
            raise RuntimeError("Frequency model training/monitor/validation IDs overlap.")

        FREQUENCY_BATCH_SIZE = 2
        FREQUENCY_EPOCHS = 14
        FREQUENCY_PATIENCE = 4
        FREQUENCY_LR = 3e-4
        FREQUENCY_WORKERS = 2
        train_freq_loader = DataLoader(
            SurfaceGuardFrequencyDataset(fit_ids, augment=True),
            batch_size=FREQUENCY_BATCH_SIZE, shuffle=True,
            num_workers=FREQUENCY_WORKERS, pin_memory=True, drop_last=False,
        )
        monitor_freq_loader = DataLoader(
            SurfaceGuardFrequencyDataset(monitor_ids, augment=False),
            batch_size=FREQUENCY_BATCH_SIZE, shuffle=False,
            num_workers=FREQUENCY_WORKERS, pin_memory=True, drop_last=False,
        )
        val_freq_loader = DataLoader(
            SurfaceGuardFrequencyDataset(val_ids, augment=False),
            batch_size=1, shuffle=False, num_workers=NUM_WORKERS,
            pin_memory=True, drop_last=False,
        )

        # Estimate class positive-pixel imbalance from the fitting partition only.
        positive_pixel_counts = np.zeros(N_CLASSES, dtype=np.float64)
        for row in train_df[train_df["image_id"].isin(fit_ids)].itertuples(index=False):
            if str(row.mask_rle).strip():
                class_index = CLASSES.index(str(row.defect_type))
                values = np.asarray(str(row.mask_rle).split(), dtype=np.int64)
                if len(values) % 2:
                    raise ValueError("Odd RLE value count while estimating positive-pixel weights.")
                positive_pixel_counts[class_index] += float(values[1::2].sum())
        total_fit_pixels = max(len(fit_ids) * N_PIXELS, 1)
        negative_pixel_counts = total_fit_pixels - positive_pixel_counts
        pos_weights = np.sqrt(
            negative_pixel_counts / np.maximum(positive_pixel_counts, 1.0)
        ).clip(1.0, 8.0).astype(np.float32)
        pos_weight_tensor = torch.as_tensor(pos_weights, device=DEVICE).view(1, N_CLASSES, 1, 1)
        print("V18 fit/monitor images:", len(fit_ids), "/", len(monitor_ids), "| split:", FREQUENCY_MONITOR_SPLIT_NOTE)
        print("Frequency model positive-pixel weights:", dict(zip(CLASSES, pos_weights.round(3).tolist())))

        frequency_model = SurfaceGuardFrequencyDetailNet(N_CLASSES).to(DEVICE)
        optimizer_frequency = optim.AdamW(
            frequency_model.parameters(), lr=FREQUENCY_LR, weight_decay=1e-4
        )
        scheduler_frequency = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer_frequency, mode="min", factor=0.5, patience=2, min_lr=2e-5
        )
        scaler_frequency = torch.cuda.amp.GradScaler(enabled=True)
        best_monitor_loss = float("inf")
        best_epoch = 0
        epochs_without_improvement = 0
        frequency_history = []

        def _frequency_loss(seg_logits, class_logits, targets, presence_targets):
            bce = F.binary_cross_entropy_with_logits(
                seg_logits, targets, pos_weight=pos_weight_tensor
            )
            probs = torch.sigmoid(seg_logits.float())
            target_float = targets.float()
            intersection = (probs * target_float).sum(dim=(2, 3))
            denominator = probs.sum(dim=(2, 3)) + target_float.sum(dim=(2, 3))
            dice = (2.0 * intersection + 1.0) / (denominator + 1.0)
            positive_pairs = target_float.sum(dim=(2, 3)) > 0
            dice_loss = (1.0 - dice[positive_pairs]).mean() if positive_pairs.any() else bce.new_zeros(())
            class_loss = F.binary_cross_entropy_with_logits(class_logits.float(), presence_targets.float())
            return bce + 0.60 * dice_loss + 0.20 * class_loss

        for epoch in range(1, FREQUENCY_EPOCHS + 1):
            frequency_model.train()
            train_loss_sum = 0.0
            train_batches = 0
            optimizer_frequency.zero_grad(set_to_none=True)
            for batch_index, (images, masks, presence_targets) in enumerate(tqdm(
                train_freq_loader, desc=f"V18 frequency training {epoch}/{FREQUENCY_EPOCHS}"
            )):
                images = images.to(DEVICE, non_blocking=True)
                masks = masks.to(DEVICE, non_blocking=True)
                presence_targets = presence_targets.to(DEVICE, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=True):
                    seg_logits, class_logits = frequency_model(images)
                    loss = _frequency_loss(seg_logits, class_logits, masks, presence_targets)
                if not torch.isfinite(loss):
                    raise RuntimeError(f"Non-finite training loss in epoch {epoch}.")
                scaler_frequency.scale(loss).backward()
                scaler_frequency.unscale_(optimizer_frequency)
                torch.nn.utils.clip_grad_norm_(frequency_model.parameters(), max_norm=1.0)
                scaler_frequency.step(optimizer_frequency)
                scaler_frequency.update()
                optimizer_frequency.zero_grad(set_to_none=True)
                train_loss_sum += float(loss.detach().cpu())
                train_batches += 1

            frequency_model.eval()
            monitor_loss_sum = 0.0
            monitor_batches = 0
            with torch.inference_mode():
                for images, masks, presence_targets in tqdm(
                    monitor_freq_loader, desc=f"V18 monitor {epoch}/{FREQUENCY_EPOCHS}"
                ):
                    images = images.to(DEVICE, non_blocking=True)
                    masks = masks.to(DEVICE, non_blocking=True)
                    presence_targets = presence_targets.to(DEVICE, non_blocking=True)
                    with torch.cuda.amp.autocast(enabled=True):
                        seg_logits, class_logits = frequency_model(images)
                        loss = _frequency_loss(seg_logits, class_logits, masks, presence_targets)
                    monitor_loss_sum += float(loss.detach().cpu())
                    monitor_batches += 1
            mean_train_loss = train_loss_sum / max(train_batches, 1)
            mean_monitor_loss = monitor_loss_sum / max(monitor_batches, 1)
            scheduler_frequency.step(mean_monitor_loss)
            frequency_history.append({
                "epoch": epoch, "train_loss": mean_train_loss,
                "monitor_loss": mean_monitor_loss,
                "learning_rate": float(optimizer_frequency.param_groups[0]["lr"]),
            })
            print(
                f"V18 epoch {epoch}: train_loss={mean_train_loss:.6f} "
                f"monitor_loss={mean_monitor_loss:.6f} "
                f"lr={optimizer_frequency.param_groups[0]['lr']:.2e}"
            )
            if mean_monitor_loss < best_monitor_loss - 1e-5:
                best_monitor_loss = mean_monitor_loss
                best_epoch = epoch
                epochs_without_improvement = 0
                torch.save({
                    "model_state_dict": frequency_model.state_dict(),
                    "classes": CLASSES, "height": HEIGHT, "width": WIDTH,
                    "model_id": "surfaceguard_v18_frequency_detail_custom",
                    "epoch": epoch, "monitor_loss": mean_monitor_loss,
                    "positive_pixel_weights": pos_weights.tolist(),
                }, FREQUENCY_MODEL_PATH)
            else:
                epochs_without_improvement += 1
            if epochs_without_improvement >= FREQUENCY_PATIENCE:
                print("V18 early stopping: monitor loss did not improve.")
                break

        if not FREQUENCY_MODEL_PATH.is_file():
            raise FileNotFoundError("V18 frequency model checkpoint was not saved.")
        del frequency_model, optimizer_frequency, scheduler_frequency, scaler_frequency
        gc.collect()
        torch.cuda.empty_cache()

        # Reload the best monitor-selected checkpoint and produce validation maps.
        frequency_checkpoint = torch.load(
            FREQUENCY_MODEL_PATH, map_location="cpu", weights_only=False
        )
        frequency_model = SurfaceGuardFrequencyDetailNet(N_CLASSES)
        frequency_model.load_state_dict(frequency_checkpoint["model_state_dict"], strict=True)
        frequency_model = frequency_model.to(DEVICE).eval()
        del frequency_checkpoint

        frequency_cache_bytes = int(len(val_ids) * N_CLASSES * HEIGHT * WIDTH * 2 + len(val_ids) * N_CLASSES * 2)
        frequency_cache_free = shutil.disk_usage(CACHE_DIR).free
        if frequency_cache_free < int(frequency_cache_bytes * 1.20):
            raise RuntimeError(
                f"Not enough free space for V18 validation caches: "
                f"need about {frequency_cache_bytes / (1024 ** 3):.2f} GiB, "
                f"free {frequency_cache_free / (1024 ** 3):.2f} GiB."
            )
        if FREQUENCY_VAL_PROBS_PATH.exists(): FREQUENCY_VAL_PROBS_PATH.unlink()
        if FREQUENCY_VAL_CLS_PATH.exists(): FREQUENCY_VAL_CLS_PATH.unlink()
        val_frequency_probs_map = np.lib.format.open_memmap(
            FREQUENCY_VAL_PROBS_PATH, mode="w+", dtype=np.float16,
            shape=(len(val_ids), N_CLASSES, HEIGHT, WIDTH)
        )
        val_frequency_cls_map = np.lib.format.open_memmap(
            FREQUENCY_VAL_CLS_PATH, mode="w+", dtype=np.float16,
            shape=(len(val_ids), N_CLASSES)
        )
        cursor = 0
        with torch.inference_mode():
            for images, masks, presence_targets in tqdm(val_freq_loader, desc="V18 frequency validation inference"):
                seg_probs, cls_probs = _frequency_predict_batch(
                    frequency_model, images, DEVICE, use_tta=(DEVICE.type == "cuda")
                )
                end = cursor + len(images)
                val_frequency_probs_map[cursor:end] = seg_probs.detach().cpu().numpy().astype(np.float16)
                val_frequency_cls_map[cursor:end] = cls_probs.detach().cpu().numpy().astype(np.float16)
                cursor = end
        if cursor != len(val_ids):
            raise RuntimeError(f"V18 validation prediction count mismatch: {cursor} != {len(val_ids)}")
        val_frequency_probs_map.flush(); val_frequency_cls_map.flush()
        del val_frequency_probs_map, val_frequency_cls_map, frequency_model
        gc.collect(); torch.cuda.empty_cache()
        val_frequency_probs = np.load(FREQUENCY_VAL_PROBS_PATH, mmap_mode="r")
        val_frequency_cls = np.load(FREQUENCY_VAL_CLS_PATH, mmap_mode="r")
        if val_frequency_probs.shape != (len(val_ids), N_CLASSES, HEIGHT, WIDTH):
            raise RuntimeError("V18 frequency validation probability shape mismatch.")
        if not np.isfinite(val_frequency_probs[0]).all() or not np.isfinite(val_frequency_cls).all():
            raise RuntimeError("V18 frequency validation cache contains non-finite values.")
        FREQUENCY_MODEL_AVAILABLE = True
        FREQUENCY_TRAINING_NOTE = f"trained; best_epoch={best_epoch}; best_monitor_loss={best_monitor_loss:.6f}"
        pd.DataFrame(frequency_history).to_csv(OUTPUT_DIR / "v18_frequency_training_history.csv", index=False)
        print("V18 frequency/detail model available:", FREQUENCY_MODEL_AVAILABLE)
        print("Best checkpoint epoch:", best_epoch, "| monitor loss:", round(best_monitor_loss, 6))
        print("V18 validation cache:", FREQUENCY_VAL_PROBS_PATH)
    except Exception as frequency_error:
        FREQUENCY_MODEL_AVAILABLE = False
        FREQUENCY_TRAINING_NOTE = f"failed/skipped safely: {type(frequency_error).__name__}: {str(frequency_error)[:240]}"
        print("\nV18 frequency/detail candidate could not be completed.")
        print(FREQUENCY_TRAINING_NOTE)
        print("Continuing with the established V17 eight-model hybrid fallback.")
        for _name in ["frequency_model", "optimizer_frequency", "scheduler_frequency", "scaler_frequency"]:
            if _name in globals():
                try: del globals()[_name]
                except Exception: pass
        gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

print("V18 frequency candidate status:", FREQUENCY_TRAINING_NOTE)


V18 fit/monitor images: 8202 / 912 | split: stratified by class-presence combination
Frequency model positive-pixel weights: {'defect_a': 8.0, 'defect_b': 8.0, 'defect_c': 7.114999771118164, 'defect_d': 8.0}


/tmp/ipykernel_23/3692865413.py:278: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_frequency = torch.cuda.amp.GradScaler(enabled=True)


V18 frequency training 1/14:   0%|          | 0/4101 [00:00<?, ?it/s]

/tmp/ipykernel_23/3692865413.py:309: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True):


V18 monitor 1/14:   0%|          | 0/456 [00:00<?, ?it/s]

/tmp/ipykernel_23/3692865413.py:333: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True):


... (48 outputs left out)

V18 epoch 14: train_loss=0.278162 monitor_loss=0.321226 lr=1.50e-04


V18 frequency validation inference:   0%|          | 0/2279 [00:00<?, ?it/s]

/tmp/ipykernel_23/3692865413.py:189: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):


V18 frequency/detail model available: True
Best checkpoint epoch: 14 | monitor loss: 0.321226
V18 validation cache: /kaggle/temp/surfaceguard_version18_frequency_detail_hybrid_cache/v18_frequency_val_probs.npy
V18 frequency candidate status: trained; best_epoch=14; best_monitor_loss=0.321226


In [7]:
# Inference-only checkpoint loading. No optimizer, scheduler, loss or training loop is created.
# Checkpoints are loaded one at a time; optimizer/scaler states are discarded immediately.

def torch_load_checkpoint(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:  # compatibility with older Kaggle PyTorch versions
        return torch.load(path, map_location="cpu")


def safe_model_to_device(model, label):
    """Place a model on the selected device, falling back on CUDA OOM."""
    try:
        model = model.to(DEVICE)
    except RuntimeError as exc:
        if DEVICE.type == "cuda" and _is_cuda_oom_error(exc):
            switch_to_cpu_fallback(exc, extra_models=(model,))
            model = model.to(torch.device("cpu"))
        else:
            raise
    model.eval()
    return model


def verify_checkpoint_metadata(checkpoint, label):
    saved_classes = checkpoint.get("classes")
    if saved_classes is not None and list(saved_classes) != CLASSES:
        raise ValueError(
            f"{label} checkpoint class order mismatch: {list(saved_classes)} != {CLASSES}"
        )
    saved_h, saved_w = checkpoint.get("height"), checkpoint.get("width")
    if saved_h is not None and int(saved_h) != HEIGHT:
        raise ValueError(f"{label} checkpoint height mismatch: {saved_h}")
    if saved_w is not None and int(saved_w) != WIDTH:
        raise ValueError(f"{label} checkpoint width mismatch: {saved_w}")
    if "model_state_dict" not in checkpoint:
        raise KeyError(f"{label} checkpoint has no model_state_dict.")


def load_smp_checkpoint(path, model, label):
    checkpoint = torch_load_checkpoint(path)
    verify_checkpoint_metadata(checkpoint, label)
    metadata = {
        "epoch": checkpoint.get("epoch"),
        "best_val_loss": checkpoint.get("best_val_loss"),
        "mean_dice": checkpoint.get("mean_dice")
    }
    result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    if result.missing_keys or result.unexpected_keys:
        raise RuntimeError(f"{label} strict checkpoint load was not clean: {result}")
    del checkpoint
    gc.collect()
    model = safe_model_to_device(model, label)
    
    print(f"{label} loaded | epoch={metadata['epoch']} | parameters={sum(p.numel() for p in model.parameters()):,}")
    return model, metadata


# V4 — U-Net with ResNet34 encoder.
model_v4 = smp.Unet(
    encoder_name="resnet34", encoder_weights=None,
    in_channels=1, classes=N_CLASSES, activation=None
)
model_v4, META_V4 = load_smp_checkpoint(V4_PATH, model_v4, "V4 U-Net/ResNet34")

# V5 — SegFormer-B0; fetch config only, then strictly load the attached four-class weights.
V5_MODEL_ID = "nvidia/segformer-b0-finetuned-ade-512-512"
config_v5 = SegformerConfig.from_pretrained(V5_MODEL_ID)
config_v5.num_labels = N_CLASSES
config_v5.id2label = {i: name for i, name in enumerate(CLASSES)}
config_v5.label2id = {name: i for i, name in enumerate(CLASSES)}
model_v5 = SegformerForSemanticSegmentation(config_v5)
checkpoint_v5 = torch_load_checkpoint(V5_PATH)
verify_checkpoint_metadata(checkpoint_v5, "V5 SegFormer-B0")
META_V5 = {"epoch": checkpoint_v5.get("epoch"), "mean_dice": checkpoint_v5.get("mean_dice")}
result_v5 = model_v5.load_state_dict(checkpoint_v5["model_state_dict"], strict=True)
if result_v5.missing_keys or result_v5.unexpected_keys:
    raise RuntimeError(f"V5 strict checkpoint load was not clean: {result_v5}")
del checkpoint_v5
model_v5 = safe_model_to_device(model_v5, "V5 SegFormer-B0")
gc.collect()

print(f"V5 SegFormer-B0 loaded | epoch={META_V5['epoch']} | parameters={sum(p.numel() for p in model_v5.parameters()):,}")

# V6 — DeepLabV3+ with ResNet50 encoder.
model_v6 = smp.DeepLabV3Plus(
    encoder_name="resnet50", encoder_weights=None,
    in_channels=1, classes=N_CLASSES, activation=None
)
model_v6, META_V6 = load_smp_checkpoint(
    V6_PATH, model_v6, "V6 DeepLabV3+/ResNet50"
)

# MiT-B4 V2 checkpoint and report determine the model ID and saved inference mode.
with open(MIT_REPORT_PATH, "r") as f:
    mit_report = json.load(f)
MIT_MODEL_ID = mit_report.get(
    "model", "nvidia/segformer-b4-finetuned-ade-512-512"
)
MIT_INFERENCE_MODE = mit_report.get("selected_inference_mode")
if MIT_INFERENCE_MODE not in {"four_way_flip_tta", "no_tta"}:
    raise ValueError(
        "MiT-B4 report must have selected_inference_mode equal to "
        "'four_way_flip_tta' or 'no_tta'."
    )
MIT_USE_TTA = MIT_INFERENCE_MODE == "four_way_flip_tta"

checkpoint_mit = torch_load_checkpoint(MIT_PATH)
verify_checkpoint_metadata(checkpoint_mit, "MiT-B4")
checkpoint_model_id = checkpoint_mit.get("model_id")
if checkpoint_model_id and checkpoint_model_id != MIT_MODEL_ID:
    raise ValueError(
        f"MiT model ID mismatch: checkpoint={checkpoint_model_id}, report={MIT_MODEL_ID}"
    )
META_MIT = {"epoch": checkpoint_mit.get("epoch"), "model_id": MIT_MODEL_ID}
config_mit = SegformerConfig.from_pretrained(MIT_MODEL_ID)
config_mit.num_labels = N_CLASSES
config_mit.id2label = {i: name for i, name in enumerate(CLASSES)}
config_mit.label2id = {name: i for i, name in enumerate(CLASSES)}
model_mit = SegformerForSemanticSegmentation(config_mit)
result_mit = model_mit.load_state_dict(
    checkpoint_mit["model_state_dict"], strict=True
)
if result_mit.missing_keys or result_mit.unexpected_keys:
    raise RuntimeError(f"MiT-B4 strict checkpoint load was not clean: {result_mit}")
del checkpoint_mit
model_mit = safe_model_to_device(model_mit, "MiT-B4")
gc.collect()



# Optional Colab V7 auxiliary-classifier models. Their four checkpoints were
# created by the executed Colab V7 notebook and saved as model_state_dict bundles.
COLAB_V7_GROUP_WEIGHTS = {
    "model_colab_a": 0.8,
    "model_colab_b": 1.0,
    "model_colab_c": 1.2,
    "model_colab_d": 1.0,
}
COLAB_V7_MODEL_SPECS = []

if COLAB_V7_AVAILABLE:
    def build_colab_aux_unet(encoder_name, num_classes):
        return smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=None,  # All trained weights are loaded strictly below.
            in_channels=1,
            classes=num_classes,
            activation=None,
            aux_params=dict(pooling="avg", dropout=0.3, activation=None, classes=num_classes),
        )

    def build_colab_aux_deeplab(num_classes):
        return smp.DeepLabV3Plus(
            encoder_name="resnet50",
            encoder_weights=None,
            in_channels=1,
            classes=num_classes,
            activation=None,
            aux_params=dict(pooling="avg", dropout=0.3, activation=None, classes=num_classes),
        )

    class ColabV7SegFormerWithClassifier(torch.nn.Module):
        def __init__(self, model_name, num_classes):
            super().__init__()
            config = SegformerConfig.from_pretrained(model_name)
            config.num_labels = num_classes
            config.id2label = {i: name for i, name in enumerate(CLASSES)}
            config.label2id = {name: i for i, name in enumerate(CLASSES)}
            self.base = SegformerForSemanticSegmentation(config)
            self.classifier_head = torch.nn.Sequential(
                torch.nn.Linear(num_classes, 64),
                torch.nn.ReLU(inplace=True),
                torch.nn.Linear(64, num_classes),
            )

        def forward(self, pixel_values, target_size):
            seg_logits = self.base(pixel_values=pixel_values).logits
            seg_logits = F.interpolate(
                seg_logits.float(), size=target_size, mode="bilinear", align_corners=False
            )
            pooled = torch.amax(seg_logits, dim=(-2, -1))
            cls_logits = self.classifier_head(pooled)
            return seg_logits, cls_logits

    def colab_v7_unet_forward(model, images):
        seg_logits, cls_logits = model(images)
        return seg_logits, cls_logits

    def colab_v7_segformer_forward(model, images):
        rgb_images = images.repeat(1, 3, 1, 1)
        return model(rgb_images, target_size=(HEIGHT, WIDTH))

    def load_colab_v7_checkpoint(path, model, label):
        checkpoint = torch_load_checkpoint(path)
        verify_checkpoint_metadata(checkpoint, label)
        result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        if result.missing_keys or result.unexpected_keys:
            raise RuntimeError(f"{label} strict checkpoint load failed: {result}")
        metadata = {
            "weighted_score": checkpoint.get("weighted_score"),
            "run_name": checkpoint.get("run_name"),
        }
        del checkpoint
        gc.collect()
        model = safe_model_to_device(model, label)
        print(f"{label} loaded | metadata={metadata} | parameters={sum(p.numel() for p in model.parameters()):,}")
        return model

    # Build without downloading ImageNet tensors: strict checkpoint loading replaces
    # every trained parameter and preserves the Colab V7 architecture.
    model_colab_a = load_colab_v7_checkpoint(
        COLAB_V7_PATHS["a_scratch"],
        build_colab_aux_unet("resnet34", N_CLASSES),
        "Colab V7 Model A — scratch ResNet34 U-Net + classifier",
    )
    model_colab_b = load_colab_v7_checkpoint(
        COLAB_V7_PATHS["b_pretrained"],
        build_colab_aux_unet("resnet34", N_CLASSES),
        "Colab V7 Model B — pretrained ResNet34 U-Net + classifier",
    )
    model_colab_c = load_colab_v7_checkpoint(
        COLAB_V7_PATHS["c_segformer_b0"],
        ColabV7SegFormerWithClassifier(
            "nvidia/segformer-b0-finetuned-ade-512-512", N_CLASSES
        ),
        "Colab V7 Model C — SegFormer MiT-B0 + classifier",
    )
    model_colab_d = load_colab_v7_checkpoint(
        COLAB_V7_PATHS["d_deeplabv3plus"],
        build_colab_aux_deeplab(N_CLASSES),
        "Colab V7 Model D — DeepLabV3+/ResNet50 + classifier",
    )

    COLAB_V7_MODEL_SPECS = [
        (colab_v7_unet_forward, model_colab_a, COLAB_V7_GROUP_WEIGHTS["model_colab_a"]),
        (colab_v7_unet_forward, model_colab_b, COLAB_V7_GROUP_WEIGHTS["model_colab_b"]),
        (colab_v7_segformer_forward, model_colab_c, COLAB_V7_GROUP_WEIGHTS["model_colab_c"]),
        (colab_v7_unet_forward, model_colab_d, COLAB_V7_GROUP_WEIGHTS["model_colab_d"]),
    ]
    print("\nCOLAB V7 AUXILIARY CHECKPOINTS LOADED — INFERENCE ONLY")
else:
    print("\nColab V7 auxiliary models skipped; V13 four-model fallback remains active.")

models = {
    "V4_U-Net_ResNet34": model_v4,
    "V5_SegFormer_B0": model_v5,
    "V6_DeepLabV3Plus_ResNet50": model_v6,
    "MiT_B4_SegFormer": model_mit
}
if COLAB_V7_AVAILABLE:
    models.update({
        "ColabV7_A_Scratch_UNet": model_colab_a,
        "ColabV7_B_Pretrained_UNet": model_colab_b,
        "ColabV7_C_SegFormer_B0": model_colab_c,
        "ColabV7_D_DeepLabV3Plus": model_colab_d,
    })

print("\nFOUR CHECKPOINTS LOADED SUCCESSFULLY — INFERENCE ONLY")
print("V4 metadata:", META_V4)
print("V5 metadata:", META_V5)
print("V6 metadata:", META_V6)
print("MiT-B4 metadata:", META_MIT)
print("MiT-B4 inference mode recorded in saved V2 report:", MIT_INFERENCE_MODE)
print("The effective runtime mode is resolved in the next cell according to the selected device.")
print("All models on:", DEVICE)


V4 U-Net/ResNet34 loaded | epoch=8 | parameters=24,430,532


config.json:   0%|          | 0.00/6.88k [00:00<?, ?B/s]

V5 SegFormer-B0 loaded | epoch=7 | parameters=3,715,172
V6 DeepLabV3+/ResNet50 loaded | epoch=7 | parameters=26,672,084


config.json:   0%|          | 0.00/6.89k [00:00<?, ?B/s]

Colab V7 Model A — scratch ResNet34 U-Net + classifier loaded | metadata={'weighted_score': 0.3559638991426173, 'run_name': 'model-a-resnet34-scratch'} | parameters=24,432,584
Colab V7 Model B — pretrained ResNet34 U-Net + classifier loaded | metadata={'weighted_score': 0.4270792529553629, 'run_name': 'model-b-resnet34-imagenet'} | parameters=24,432,584
Colab V7 Model C — SegFormer MiT-B0 + classifier loaded | metadata={'weighted_score': 0.30469109030428143, 'run_name': 'model-c-segformer-b0'} | parameters=3,715,752
Colab V7 Model D — DeepLabV3+/ResNet50 + classifier loaded | metadata={'weighted_score': 0.4217395673339297, 'run_name': 'model-d-deeplabv3plus-resnet50'} | parameters=26,680,280

COLAB V7 AUXILIARY CHECKPOINTS LOADED — INFERENCE ONLY

FOUR CHECKPOINTS LOADED SUCCESSFULLY — INFERENCE ONLY
V4 metadata: {'epoch': 8, 'best_val_loss': None, 'mean_dice': 0.4486256566377196}
V5 metadata: {'epoch': 7, 'mean_dice': 0.5034020691225941}
V6 metadata: {'epoch': 7, 'best_val_loss': 0.41

In [8]:
# GPU mode uses the full four-way flip TTA experiment. CPU-only mode uses one
# unflipped pass for all models to keep a fallback run practical. Validation and
# test inference always use the same selected mode within a run.
if DEVICE.type == "cuda":
    MAIN_INFERENCE_MODE = "four_way_flip_tta"
    MAIN_TTA_TRANSFORMS = [(False, False), (True, False), (False, True), (True, True)]
else:
    MAIN_INFERENCE_MODE = "no_tta_cpu_fallback"
    MAIN_TTA_TRANSFORMS = [(False, False)]
    MIT_USE_TTA = False
    MIT_INFERENCE_MODE = "no_tta_cpu_fallback"


def autocast_context():
    if DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def resize_segformer_logits(logits):
    if logits.shape[-2:] != (HEIGHT, WIDTH):
        logits = F.interpolate(
            logits.float(), size=(HEIGHT, WIDTH), mode="bilinear", align_corners=False
        )
    return logits.float()


def _predict_main_models_impl(images):
    images = images.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
    sums = [None, None, None]
    for hflip, vflip in MAIN_TTA_TRANSFORMS:
        transformed = images
        if hflip:
            transformed = torch.flip(transformed, dims=[3])
        if vflip:
            transformed = torch.flip(transformed, dims=[2])
        with autocast_context():
            logits_v4 = model_v4(transformed)
            probs_v4 = torch.sigmoid(logits_v4.float())
            del logits_v4
            rgb_images = transformed.repeat(1, 3, 1, 1)
            logits_v5 = model_v5(pixel_values=rgb_images).logits
            logits_v5 = resize_segformer_logits(logits_v5)
            probs_v5 = torch.sigmoid(logits_v5.float())
            del logits_v5, rgb_images
            logits_v6 = model_v6(transformed)
            probs_v6 = torch.sigmoid(logits_v6.float())
            del logits_v6
        current = [probs_v4, probs_v5, probs_v6]
        if vflip:
            current = [torch.flip(p, dims=[2]) for p in current]
        if hflip:
            current = [torch.flip(p, dims=[3]) for p in current]
        for model_index, probabilities in enumerate(current):
            sums[model_index] = probabilities if sums[model_index] is None else sums[model_index] + probabilities
        del transformed, probs_v4, probs_v5, probs_v6, current
    return tuple(total / len(MAIN_TTA_TRANSFORMS) for total in sums)


@torch.inference_mode()
def predict_main_models(images):
    try:
        return _predict_main_models_impl(images)
    except RuntimeError as exc:
        if DEVICE.type == "cuda" and _is_cuda_oom_error(exc):
            switch_to_cpu_fallback(exc)
            return _predict_main_models_impl(images)
        raise


def _predict_mit_model_impl(images):
    images = images.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
    def one_pass(x):
        rgb = x.repeat(1, 3, 1, 1)
        with autocast_context():
            logits = model_mit(pixel_values=rgb).logits
            logits = resize_segformer_logits(logits)
            return torch.sigmoid(logits.float())
    if not MIT_USE_TTA:
        return one_pass(images)
    total = None
    transforms = [(False, False), (True, False), (False, True), (True, True)]
    for hflip, vflip in transforms:
        transformed = images
        if hflip:
            transformed = torch.flip(transformed, dims=[3])
        if vflip:
            transformed = torch.flip(transformed, dims=[2])
        probs = one_pass(transformed)
        if vflip:
            probs = torch.flip(probs, dims=[2])
        if hflip:
            probs = torch.flip(probs, dims=[3])
        total = probs if total is None else total + probs
    return total / len(transforms)


@torch.inference_mode()
def predict_mit_model(images):
    try:
        return _predict_mit_model_impl(images)
    except RuntimeError as exc:
        if DEVICE.type == "cuda" and _is_cuda_oom_error(exc):
            switch_to_cpu_fallback(exc)
            return _predict_mit_model_impl(images)
        raise




@torch.inference_mode()
def _predict_colab_v7_group_impl(images):
    """Return weighted mean segmentation and image-level class probabilities."""
    if not COLAB_V7_AVAILABLE:
        return None, None
    images = images.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
    seg_total = None
    cls_total = None
    weight_total = 0.0
    transforms = MAIN_TTA_TRANSFORMS

    for forward_fn, model, model_weight in COLAB_V7_MODEL_SPECS:
        model_seg_total = None
        model_cls_total = None
        for hflip, vflip in transforms:
            transformed = images
            if hflip:
                transformed = torch.flip(transformed, dims=[3])
            if vflip:
                transformed = torch.flip(transformed, dims=[2])
            with autocast_context():
                seg_logits, cls_logits = forward_fn(model, transformed)
                if seg_logits.shape[-2:] != (HEIGHT, WIDTH):
                    seg_logits = F.interpolate(
                        seg_logits.float(), size=(HEIGHT, WIDTH), mode="bilinear", align_corners=False
                    )
                seg_probs = torch.sigmoid(seg_logits.float())
                cls_probs = torch.sigmoid(cls_logits.float())
            if vflip:
                seg_probs = torch.flip(seg_probs, dims=[2])
            if hflip:
                seg_probs = torch.flip(seg_probs, dims=[3])
            model_seg_total = seg_probs if model_seg_total is None else model_seg_total + seg_probs
            model_cls_total = cls_probs if model_cls_total is None else model_cls_total + cls_probs
            del transformed, seg_logits, cls_logits, seg_probs, cls_probs
        model_seg = model_seg_total / len(transforms)
        model_cls = model_cls_total / len(transforms)
        seg_total = model_weight * model_seg if seg_total is None else seg_total + model_weight * model_seg
        cls_total = model_weight * model_cls if cls_total is None else cls_total + model_weight * model_cls
        weight_total += model_weight
        del model_seg_total, model_cls_total, model_seg, model_cls
    return seg_total / weight_total, cls_total / weight_total


@torch.inference_mode()
def predict_colab_v7_group(images):
    if not COLAB_V7_AVAILABLE:
        return None, None
    try:
        seg, cls = _predict_colab_v7_group_impl(images)
    except RuntimeError as exc:
        if DEVICE.type == "cuda" and _is_cuda_oom_error(exc):
            switch_to_cpu_fallback(
                exc,
                extra_models=[spec[1] for spec in COLAB_V7_MODEL_SPECS],
            )
            seg, cls = _predict_colab_v7_group_impl(images)
        else:
            raise
    if not torch.isfinite(seg).all() or not torch.isfinite(cls).all():
        raise RuntimeError("Colab V7 auxiliary prediction contains non-finite values.")
    if seg.min().item() < 0 or seg.max().item() > 1 or cls.min().item() < 0 or cls.max().item() > 1:
        raise RuntimeError("Colab V7 probabilities are outside [0, 1].")
    return seg, cls

# One-batch inference check before generating any large prediction caches.
smoke_images, smoke_masks, smoke_ids = next(iter(val_loader_main))
smoke_v4, smoke_v5, smoke_v6 = predict_main_models(smoke_images)
smoke_mit_images, _, _ = next(iter(val_loader_mit))
smoke_mit = predict_mit_model(smoke_mit_images)

for name, tensor in [
    ("V4", smoke_v4), ("V5", smoke_v5), ("V6", smoke_v6), ("MiT-B4", smoke_mit)
]:
    expected = (len(tensor), N_CLASSES, HEIGHT, WIDTH)
    if tuple(tensor.shape) != expected:
        raise RuntimeError(f"{name} output shape is invalid: {tuple(tensor.shape)} != {expected}")
    if not torch.isfinite(tensor).all():
        raise RuntimeError(f"{name} output contains non-finite values.")
    if tensor.min().item() < 0 or tensor.max().item() > 1:
        raise RuntimeError(f"{name} probabilities are outside [0, 1].")
    print(name, tuple(tensor.shape), "range:", float(tensor.min()), float(tensor.max()))

print("Inference smoke test: PASSED")

if COLAB_V7_AVAILABLE:
    smoke_colab_seg, smoke_colab_cls = predict_colab_v7_group(smoke_images)
    if tuple(smoke_colab_seg.shape) != (len(smoke_images), N_CLASSES, HEIGHT, WIDTH):
        raise RuntimeError(f"Colab V7 segmentation shape is invalid: {tuple(smoke_colab_seg.shape)}")
    if tuple(smoke_colab_cls.shape) != (len(smoke_images), N_CLASSES):
        raise RuntimeError(f"Colab V7 classifier shape is invalid: {tuple(smoke_colab_cls.shape)}")
    print("Colab V7 auxiliary group smoke test: PASSED")
    del smoke_colab_seg, smoke_colab_cls
print("Execution device:", DEVICE, "|", GPU_NAME)
print("Main inference mode:", MAIN_INFERENCE_MODE, "| TTA passes:", len(MAIN_TTA_TRANSFORMS))
print("MiT inference mode:", MIT_INFERENCE_MODE)
del smoke_images, smoke_masks, smoke_ids, smoke_v4, smoke_v5, smoke_v6, smoke_mit_images, smoke_mit
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


V4 (1, 4, 128, 800) range: 5.0217416003306425e-08 0.10394755750894547
V5 (1, 4, 128, 800) range: 1.5352584981087603e-08 0.27396005392074585
V6 (1, 4, 128, 800) range: 1.182569042690318e-10 0.3969636857509613
MiT-B4 (1, 4, 128, 800) range: 3.8301113391980834e-09 0.18422581255435944
Inference smoke test: PASSED
Colab V7 auxiliary group smoke test: PASSED
Execution device: cuda:0 | Tesla T4
Main inference mode: four_way_flip_tta | TTA passes: 4
MiT inference mode: four_way_flip_tta


In [9]:
N_VAL = len(val_ids)
VAL_SHAPE = (N_VAL, N_CLASSES, HEIGHT, WIDTH)

val_v4_path = CACHE_DIR / "val_v4.npy"
val_v5_path = CACHE_DIR / "val_v5.npy"
val_v6_path = CACHE_DIR / "val_v6.npy"
val_mit_path = CACHE_DIR / "val_mit_b4.npy"
val_masks_path = CACHE_DIR / "val_masks.npy"
val_colab_seg_path = CACHE_DIR / "val_colab_v7_seg.npy"
val_colab_cls_path = CACHE_DIR / "val_colab_v7_cls.npy"

# Four baseline segmentation maps + masks; optional Colab branch adds one ensemble map + classifier scores.
cache_factor_bytes = (2 * 4 + 1) + (2 + 2 / (HEIGHT * WIDTH) if COLAB_V7_AVAILABLE else 0)
cache_required_bytes = int(np.prod(VAL_SHAPE) * cache_factor_bytes)
cache_free_bytes = shutil.disk_usage(CACHE_DIR).free
print("Estimated validation cache:", round(cache_required_bytes / (1024 ** 3), 2), "GiB")
print("Free space at cache location:", round(cache_free_bytes / (1024 ** 3), 2), "GiB")
if cache_free_bytes < int(cache_required_bytes * 1.15):
    raise RuntimeError(
        "Insufficient free space for validation probability caches. "
        "Clear temporary Kaggle files or use a runtime with more disk space before proceeding."
    )

# Fresh V17-specific cache files prevent stale V13/V14 predictions being reused.
for stale in [val_v4_path, val_v5_path, val_v6_path, val_mit_path, val_masks_path,
              val_colab_seg_path, val_colab_cls_path]:
    if stale.exists():
        stale.unlink()

val_v4_probs = np.lib.format.open_memmap(val_v4_path, mode="w+", dtype=np.float16, shape=VAL_SHAPE)
val_v5_probs = np.lib.format.open_memmap(val_v5_path, mode="w+", dtype=np.float16, shape=VAL_SHAPE)
val_v6_probs = np.lib.format.open_memmap(val_v6_path, mode="w+", dtype=np.float16, shape=VAL_SHAPE)
val_mit_probs = np.lib.format.open_memmap(val_mit_path, mode="w+", dtype=np.float16, shape=VAL_SHAPE)
val_masks = np.lib.format.open_memmap(val_masks_path, mode="w+", dtype=np.uint8, shape=VAL_SHAPE)
if COLAB_V7_AVAILABLE:
    val_colab_seg_probs = np.lib.format.open_memmap(
        val_colab_seg_path, mode="w+", dtype=np.float16, shape=VAL_SHAPE
    )
    val_colab_cls_probs = np.lib.format.open_memmap(
        val_colab_cls_path, mode="w+", dtype=np.float16, shape=(N_VAL, N_CLASSES)
    )

print("Generating validation predictions for the proven Kaggle group...")
if COLAB_V7_AVAILABLE:
    print("Also generating the four-model Colab V7 auxiliary segmentation + classifier ensemble...")
cursor = 0
with torch.inference_mode():
    for images, masks, image_ids in tqdm(val_loader_main, desc="Validation: Kaggle + optional Colab V7"):
        p4_batch, p5_batch, p6_batch = predict_main_models(images)
        batch_size = len(image_ids)
        end = cursor + batch_size
        val_v4_probs[cursor:end] = p4_batch.detach().cpu().numpy().astype(np.float16)
        val_v5_probs[cursor:end] = p5_batch.detach().cpu().numpy().astype(np.float16)
        val_v6_probs[cursor:end] = p6_batch.detach().cpu().numpy().astype(np.float16)
        val_masks[cursor:end] = masks.numpy().astype(np.uint8)
        if COLAB_V7_AVAILABLE:
            colab_seg_batch, colab_cls_batch = predict_colab_v7_group(images)
            val_colab_seg_probs[cursor:end] = colab_seg_batch.detach().cpu().numpy().astype(np.float16)
            val_colab_cls_probs[cursor:end] = colab_cls_batch.detach().cpu().numpy().astype(np.float16)
            del colab_seg_batch, colab_cls_batch
        cursor = end
        del p4_batch, p5_batch, p6_batch

if cursor != N_VAL:
    raise RuntimeError(f"Main validation prediction count mismatch: {cursor} != {N_VAL}")

for array in [val_v4_probs, val_v5_probs, val_v6_probs, val_masks]:
    array.flush()
if COLAB_V7_AVAILABLE:
    val_colab_seg_probs.flush()
    val_colab_cls_probs.flush()

print("Kaggle V4/V5/V6 validation predictions saved:", cursor)
if COLAB_V7_AVAILABLE:
    print("Colab V7 ensemble segmentation and classifier predictions saved:", cursor)
print("Validation arrays:", VAL_SHAPE)
print("Cache directory:", CACHE_DIR)


Estimated validation cache: 9.56 GiB
Free space at cache location: 965.12 GiB
Generating validation predictions for the proven Kaggle group...
Also generating the four-model Colab V7 auxiliary segmentation + classifier ensemble...


Validation: Kaggle + optional Colab V7:   0%|          | 0/2279 [00:00<?, ?it/s]

Kaggle V4/V5/V6 validation predictions saved: 2279
Colab V7 ensemble segmentation and classifier predictions saved: 2279
Validation arrays: (2279, 4, 128, 800)
Cache directory: /kaggle/temp/surfaceguard_version18_frequency_detail_hybrid_cache


In [10]:
print("MiT-B4 validation inference mode:", MIT_INFERENCE_MODE)
cursor = 0
with torch.inference_mode():
    for images, masks, image_ids in tqdm(val_loader_mit, desc="Validation: MiT-B4"):
        mit_batch = predict_mit_model(images)
        batch_size = len(image_ids)
        end = cursor + batch_size
        val_mit_probs[cursor:end] = mit_batch.detach().cpu().numpy().astype(np.float16)
        cursor = end

if cursor != N_VAL:
    raise RuntimeError(f"MiT validation prediction count mismatch: {cursor} != {N_VAL}")

val_mit_probs.flush()

# Reopen disk-backed arrays read-only. They remain memory mapped, not copied into RAM.
val_v4_probs = np.load(val_v4_path, mmap_mode="r")
val_v5_probs = np.load(val_v5_path, mmap_mode="r")
val_v6_probs = np.load(val_v6_path, mmap_mode="r")
val_mit_probs = np.load(val_mit_path, mmap_mode="r")
val_masks = np.load(val_masks_path, mmap_mode="r")

for name, array in [
    ("V4", val_v4_probs), ("V5", val_v5_probs), ("V6", val_v6_probs),
    ("MiT-B4", val_mit_probs), ("Ground truth", val_masks)
]:
    if array.shape != VAL_SHAPE:
        raise RuntimeError(f"{name} validation array has wrong shape: {array.shape}")
    if name != "Ground truth":
        # Check finite values in small chunks to avoid reading/copying all arrays at once.
        for start in range(0, N_VAL, 16):
            if not np.isfinite(array[start:start + 16]).all():
                raise RuntimeError(f"{name} contains non-finite probabilities.")
    print(name, array.shape, array.dtype)

print("All four validation caches are ready.")

MiT-B4 validation inference mode: four_way_flip_tta


Validation: MiT-B4:   0%|          | 0/2279 [00:00<?, ?it/s]

V4 (2279, 4, 128, 800) float16
V5 (2279, 4, 128, 800) float16
V6 (2279, 4, 128, 800) float16
MiT-B4 (2279, 4, 128, 800) float16
Ground truth (2279, 4, 128, 800) uint8
All four validation caches are ready.


In [11]:
# Threshold grid used for calibration. Histogram bins let us evaluate a finer 0.005 threshold grid
# without repeatedly materialising full-resolution binary masks.
THRESHOLD_GRID = np.round(np.arange(0.01, 0.951, 0.005), 3).astype(np.float32)
N_THRESHOLD_BINS = len(THRESHOLD_GRID) + 1
CAL_CHUNK_SIZE = 8

# A blend is always [V4, V5, V6, MiT-B4] and must sum to one.
def validate_blend_weights(weights):
    weights = tuple(float(x) for x in weights)
    if len(weights) != 4 or any((not np.isfinite(x)) or x < -1e-9 for x in weights):
        raise ValueError(f"Invalid four-model blend weights: {weights}")
    if not np.isclose(sum(weights), 1.0, atol=1e-6):
        raise ValueError(f"Blend weights must sum to one: {weights}")
    return weights


def validate_blend_weight_matrix(matrix):
    """Validate a [class, model] matrix; each class row must sum to one."""
    matrix = np.asarray(matrix, dtype=np.float64)
    expected_shape = (N_CLASSES, 4)
    if matrix.shape != expected_shape:
        raise ValueError(f"Class-weight matrix must have shape {expected_shape}; got {matrix.shape}")
    if not np.isfinite(matrix).all() or np.any(matrix < -1e-9):
        raise ValueError("Class-weight matrix contains invalid or negative weights.")
    row_sums = matrix.sum(axis=1)
    if not np.allclose(row_sums, 1.0, atol=1e-6):
        raise ValueError(f"Each class weight row must sum to one; got {row_sums}")
    return matrix


def combine_cached_probabilities(row_indices, class_idx, weights):
    # Accept either one global [V4,V5,V6,MiT] vector or one [class,model] matrix.
    weight_array = np.asarray(weights, dtype=np.float64)
    if weight_array.ndim == 2:
        weight_array = validate_blend_weight_matrix(weight_array)[class_idx]
    else:
        weight_array = np.asarray(validate_blend_weights(weight_array), dtype=np.float64)
    w4, w5, w6, wm = weight_array
    p4 = np.asarray(val_v4_probs[row_indices, class_idx], dtype=np.float32)
    p5 = np.asarray(val_v5_probs[row_indices, class_idx], dtype=np.float32)
    p6 = np.asarray(val_v6_probs[row_indices, class_idx], dtype=np.float32)
    pm = np.asarray(val_mit_probs[row_indices, class_idx], dtype=np.float32)
    return w4 * p4 + w5 * p5 + w6 * p6 + wm * pm


# Histogram boundary self-test: side="right" plus suffix index j+1 implements p >= t.
for _j in (0, 40, len(THRESHOLD_GRID) - 1):
    _t = THRESHOLD_GRID[_j]
    _toy = np.array([
        np.nextafter(_t, np.float32(0.0)), _t,
        np.nextafter(_t, np.float32(1.0))
    ], dtype=np.float32)
    _bins = np.searchsorted(THRESHOLD_GRID, _toy, side="right")
    _predicted = _bins >= (_j + 1)
    assert _predicted.tolist() == [False, True, True], (float(_t), _toy, _bins, _predicted)
del _j, _t, _toy, _bins, _predicted
print("Threshold histogram >= boundary self-test: PASSED")

# Count ground-truth pixels per image/class once.
gt_counts = np.zeros((N_VAL, N_CLASSES), dtype=np.uint32)
for start in range(0, N_VAL, 16):
    end = min(start + 16, N_VAL)
    block = np.asarray(val_masks[start:end])
    gt_counts[start:end] = np.count_nonzero(block, axis=(2, 3)).astype(np.uint32)
del block
gc.collect()


def optimise_thresholds_for_weights(weights, indices):
    """Optimise per-class thresholds for global weights or a class-specific weight matrix."""
    weight_array = np.asarray(weights, dtype=np.float64)
    if weight_array.ndim == 2:
        weights = validate_blend_weight_matrix(weight_array)
    else:
        weights = validate_blend_weights(weight_array)
    indices = np.asarray(indices, dtype=np.int64)
    local_gt_counts = gt_counts[indices]
    total_positive = int((local_gt_counts > 0).sum())
    total_empty = int((local_gt_counts == 0).sum())
    if total_positive == 0 or total_empty == 0:
        raise ValueError("Calibration subset must contain positive and empty image-class pairs.")
    total_empty_pixels = total_empty * N_PIXELS

    selected_thresholds = {}
    per_class = {}
    total_score = 0.0

    for class_idx, class_name in enumerate(CLASSES):
        n_rows = len(indices)
        hist_all = np.zeros((n_rows, N_THRESHOLD_BINS), dtype=np.uint32)
        hist_positive = np.zeros_like(hist_all)
        class_gt = local_gt_counts[:, class_idx]
        positive = class_gt > 0
        empty = ~positive
        class_empty_count = int(empty.sum())

        for chunk_start in range(0, n_rows, CAL_CHUNK_SIZE):
            chunk_end = min(chunk_start + CAL_CHUNK_SIZE, n_rows)
            row_indices = indices[chunk_start:chunk_end]
            combined = combine_cached_probabilities(row_indices, class_idx, weights)
            targets = np.asarray(val_masks[row_indices, class_idx] > 0)
            combined_flat = combined.reshape(len(row_indices), -1)
            targets_flat = targets.reshape(len(row_indices), -1)
            bins = np.searchsorted(THRESHOLD_GRID, combined_flat, side="right").astype(np.uint8)

            for local_row in range(len(row_indices)):
                hist_all[chunk_start + local_row] = np.bincount(
                    bins[local_row], minlength=N_THRESHOLD_BINS
                ).astype(np.uint32)
                hist_positive[chunk_start + local_row] = np.bincount(
                    bins[local_row][targets_flat[local_row]], minlength=N_THRESHOLD_BINS
                ).astype(np.uint32)

        suffix_all = np.cumsum(hist_all[:, ::-1], axis=1, dtype=np.uint32)[:, ::-1]
        suffix_positive = np.cumsum(hist_positive[:, ::-1], axis=1, dtype=np.uint32)[:, ::-1]
        best_contribution = -1.0
        best_threshold = None
        best_components = None

        for threshold_index, threshold in enumerate(THRESHOLD_GRID):
            predicted_counts = suffix_all[:, threshold_index + 1]
            intersection_counts = suffix_positive[:, threshold_index + 1]

            pair_dice = 2.0 * intersection_counts[positive] / np.maximum(
                predicted_counts[positive] + class_gt[positive], 1
            )
            dice_sum = float(pair_dice.sum(dtype=np.float64))
            class_foreground_dice = dice_sum / int(positive.sum()) if positive.any() else 0.0
            empty_fp_pixels = int(predicted_counts[empty].sum(dtype=np.uint64))
            correct_empty = int((predicted_counts[empty] == 0).sum())
            class_empty_pixels = class_empty_count * N_PIXELS
            contribution = (
                0.70 * dice_sum / total_positive
                + 0.20 * (class_empty_pixels - empty_fp_pixels) / total_empty_pixels
                + 0.10 * correct_empty / total_empty
            )

            if contribution > best_contribution:
                best_contribution = float(contribution)
                best_threshold = float(threshold)
                best_components = {
                    "positive_pairs": int(positive.sum()),
                    "empty_pairs": class_empty_count,
                    "foreground_dice": float(class_foreground_dice),
                    "false_positive_score": float(1.0 - empty_fp_pixels / class_empty_pixels) if class_empty_pixels else float("nan"),
                    "empty_accuracy": float(correct_empty / class_empty_count) if class_empty_count else float("nan"),
                    "global_score_contribution": float(contribution)
                }

        selected_thresholds[class_name] = best_threshold
        per_class[class_name] = {"threshold": best_threshold, **best_components}
        total_score += best_contribution
        del hist_all, hist_positive, suffix_all, suffix_positive
        gc.collect()

    return float(total_score), selected_thresholds, per_class


def evaluate_mix_on_indices(weights, thresholds, indices):
    """Evaluate global or class-specific four-model weights on supplied indices."""
    weight_array = np.asarray(weights, dtype=np.float64)
    if weight_array.ndim == 2:
        weights = validate_blend_weight_matrix(weight_array)
    else:
        weights = validate_blend_weights(weight_array)
    indices = np.asarray(indices, dtype=np.int64)
    positive_dice_sum = 0.0
    positive_pairs = 0
    empty_fp_pixels = 0
    empty_pixels = 0
    correct_empty_pairs = 0
    empty_pairs = 0
    class_metrics = {}

    for class_idx, class_name in enumerate(CLASSES):
        class_dice_sum = 0.0
        class_positive_pairs = 0
        class_empty_fp_pixels = 0
        class_empty_pixels = 0
        class_correct_empty = 0
        class_empty_pairs = 0
        threshold = np.float32(thresholds[class_name])

        for chunk_start in range(0, len(indices), CAL_CHUNK_SIZE):
            row_indices = indices[chunk_start:chunk_start + CAL_CHUNK_SIZE]
            combined = combine_cached_probabilities(row_indices, class_idx, weights)
            prediction = combined >= threshold
            target = np.asarray(val_masks[row_indices, class_idx] > 0)
            target_count = np.count_nonzero(target, axis=(1, 2))
            pred_count = np.count_nonzero(prediction, axis=(1, 2))
            intersection = np.count_nonzero(prediction & target, axis=(1, 2))
            positive = target_count > 0
            empty = ~positive

            if positive.any():
                pair_dice = 2.0 * intersection[positive] / np.maximum(
                    pred_count[positive] + target_count[positive], 1
                )
                class_dice_sum += float(pair_dice.sum())
                class_positive_pairs += int(positive.sum())
            if empty.any():
                class_empty_pairs += int(empty.sum())
                class_empty_fp_pixels += int(pred_count[empty].sum())
                class_empty_pixels += int(empty.sum()) * N_PIXELS
                class_correct_empty += int((pred_count[empty] == 0).sum())

        positive_dice_sum += class_dice_sum
        positive_pairs += class_positive_pairs
        empty_fp_pixels += class_empty_fp_pixels
        empty_pixels += class_empty_pixels
        correct_empty_pairs += class_correct_empty
        empty_pairs += class_empty_pairs
        class_metrics[class_name] = {
            "positive_pairs": class_positive_pairs,
            "empty_pairs": class_empty_pairs,
            "foreground_dice": class_dice_sum / class_positive_pairs if class_positive_pairs else float("nan"),
            "false_positive_score": 1.0 - class_empty_fp_pixels / class_empty_pixels if class_empty_pixels else float("nan"),
            "empty_accuracy": class_correct_empty / class_empty_pairs if class_empty_pairs else float("nan")
        }

    foreground_dice = positive_dice_sum / positive_pairs
    false_positive_score = 1.0 - empty_fp_pixels / empty_pixels
    empty_accuracy = correct_empty_pairs / empty_pairs
    weighted_score = 0.70 * foreground_dice + 0.20 * false_positive_score + 0.10 * empty_accuracy
    return {
        "foreground_dice": float(foreground_dice),
        "false_positive_score": float(false_positive_score),
        "empty_accuracy": float(empty_accuracy),
        "weighted_score": float(weighted_score),
        "positive_pairs": int(positive_pairs),
        "empty_pairs": int(empty_pairs),
        "class_metrics": class_metrics
    }

print("Independent four-model calibration/evaluation functions ready.")


Threshold histogram >= boundary self-test: PASSED
Independent four-model calibration/evaluation functions ready.


In [12]:
# Search all four model weights independently on CAL_IDX only.
# Coarse simplex grid: 0.25 increments (35 combinations), plus explicit known anchors.
from itertools import product

coarse_candidates = [
    tuple(round(x * 0.25, 3) for x in parts)
    for parts in product(range(5), repeat=4)
    if sum(parts) == 4
]

if len(coarse_candidates) != 35:
    raise RuntimeError(f"Unexpected four-model coarse grid size: {len(coarse_candidates)}")

V7_WEIGHT_ANCHOR_WEIGHTS = (0.125, 0.125, 0.125, 0.625)
THREE_MODEL_ANCHOR_WEIGHTS = (0.375, 0.375, 0.25, 0.0)

search_rows = []
coarse_best = None
for candidate in tqdm(coarse_candidates, desc="Coarse four-model simplex search"):
    score, thresholds, details = optimise_thresholds_for_weights(candidate, CAL_IDX)
    row = {
        "stage": "coarse",
        "weights": candidate,
        "v4_weight": candidate[0], "v5_weight": candidate[1],
        "v6_weight": candidate[2], "mit_b4_weight": candidate[3],
        "calibration_score": score, "thresholds": thresholds, "per_class": details
    }
    search_rows.append(row)
    if coarse_best is None or score > coarse_best["calibration_score"]:
        coarse_best = row

# Evaluate anchors explicitly. These are not selected using the holdout.
anchor_rows = {}
for label, weights in [
    ("v7_weight_anchor", V7_WEIGHT_ANCHOR_WEIGHTS),
    ("three_model_anchor", THREE_MODEL_ANCHOR_WEIGHTS)
]:
    score, thresholds, details = optimise_thresholds_for_weights(weights, CAL_IDX)
    row = {
        "stage": label, "weights": weights,
        "v4_weight": weights[0], "v5_weight": weights[1],
        "v6_weight": weights[2], "mit_b4_weight": weights[3],
        "calibration_score": score, "thresholds": thresholds, "per_class": details
    }
    search_rows.append(row)
    anchor_rows[label] = row

# Refine around the best coarse/anchor candidate. Deltas sum to zero, preserving the simplex.
search_center = max(search_rows, key=lambda r: r["calibration_score"])
refine_step = 0.125
refine_candidates = set()
for deltas in product([-refine_step, 0.0, refine_step], repeat=4):
    if not np.isclose(sum(deltas), 0.0):
        continue
    candidate = tuple(round(w + d, 3) for w, d in zip(search_center["weights"], deltas))
    if min(candidate) < 0 or not np.isclose(sum(candidate), 1.0):
        continue
    if candidate not in {tuple(r["weights"]) for r in search_rows}:
        refine_candidates.add(candidate)

for candidate in tqdm(sorted(refine_candidates), desc="Local 0.125 four-model refinement"):
    score, thresholds, details = optimise_thresholds_for_weights(candidate, CAL_IDX)
    row = {
        "stage": "refine", "weights": candidate,
        "v4_weight": candidate[0], "v5_weight": candidate[1],
        "v6_weight": candidate[2], "mit_b4_weight": candidate[3],
        "calibration_score": score, "thresholds": thresholds, "per_class": details
    }
    search_rows.append(row)


# V17 fine refinement: transfer 0.05 weight between model pairs around the two
# strongest distinct global candidates. These candidates remain class-aware
# selection candidates, but all final decisions are still made using CAL_IDX.
top_centers = []
seen_centers = set()
for candidate_row in sorted(search_rows, key=lambda r: r["calibration_score"], reverse=True):
    center = tuple(float(x) for x in candidate_row["weights"])
    if center not in seen_centers:
        seen_centers.add(center)
        top_centers.append(center)
    if len(top_centers) == 2:
        break

fine_refine_candidates = set()
FINE_REFINE_STEP = 0.05
for center in top_centers:
    for donor in range(4):
        for receiver in range(4):
            if donor == receiver:
                continue
            candidate = list(center)
            candidate[donor] -= FINE_REFINE_STEP
            candidate[receiver] += FINE_REFINE_STEP
            if min(candidate) < -1e-9:
                continue
            candidate = tuple(round(max(0.0, float(x)), 3) for x in candidate)
            if not np.isclose(sum(candidate), 1.0, atol=1e-6):
                continue
            if candidate not in {tuple(r["weights"]) for r in search_rows}:
                fine_refine_candidates.add(candidate)

fine_refine_rows = []
for candidate in tqdm(
    sorted(fine_refine_candidates),
    desc="V17 fine 0.05 base-model weight refinement"
):
    score, thresholds, details = optimise_thresholds_for_weights(candidate, CAL_IDX)
    row = {
        "stage": "fine_0.05",
        "weights": candidate,
        "v4_weight": candidate[0], "v5_weight": candidate[1],
        "v6_weight": candidate[2], "mit_b4_weight": candidate[3],
        "calibration_score": score, "thresholds": thresholds, "per_class": details
    }
    search_rows.append(row)
    fine_refine_rows.append(row)

fine_blend_search_df = pd.DataFrame([
    {
        "stage": r["stage"], "v4_weight": r["v4_weight"],
        "v5_weight": r["v5_weight"], "v6_weight": r["v6_weight"],
        "mit_b4_weight": r["mit_b4_weight"],
        "calibration_score": r["calibration_score"]
    }
    for r in fine_refine_rows
]).sort_values("calibration_score", ascending=False).reset_index(drop=True)

print("V17 fine candidates evaluated:", len(fine_refine_rows))
if len(fine_blend_search_df):
    print("Top V17 fine-grid candidates:")
    display(fine_blend_search_df.head(10).round(6))
else:
    print("No additional fine-grid candidates were needed; the coarse candidates already covered the local simplex.")

best_calibration_row = max(search_rows, key=lambda r: r["calibration_score"])
SELECTED_WEIGHTS = validate_blend_weights(best_calibration_row["weights"])
SELECTED_V4_WEIGHT, SELECTED_V5_WEIGHT, SELECTED_V6_WEIGHT, SELECTED_MIT_WEIGHT = SELECTED_WEIGHTS
CALIBRATION_THRESHOLDS = dict(best_calibration_row["thresholds"])
V7_WEIGHT_ANCHOR_CALIBRATION_SCORE = float(anchor_rows["v7_weight_anchor"]["calibration_score"])
THREE_MODEL_ANCHOR_CALIBRATION_SCORE = float(anchor_rows["three_model_anchor"]["calibration_score"])

blend_search_df = pd.DataFrame([
    {
        "stage": r["stage"], "v4_weight": r["v4_weight"],
        "v5_weight": r["v5_weight"], "v6_weight": r["v6_weight"],
        "mit_b4_weight": r["mit_b4_weight"],
        "calibration_score": r["calibration_score"]
    }
    for r in search_rows
]).sort_values("calibration_score", ascending=False).reset_index(drop=True)

print("TOP FOUR-MODEL BLEND CANDIDATES")
display(blend_search_df.head(12).round(6))
print("\nSelected independent weights:")
for name, weight in zip(["V4", "V5", "V6", "MiT-B4"], SELECTED_WEIGHTS):
    print(f"{name}: {weight:.3f}")
print("Weights sum:", sum(SELECTED_WEIGHTS))
print("Selected calibration thresholds:", CALIBRATION_THRESHOLDS)
print("V7 weight-anchor calibration score:", round(V7_WEIGHT_ANCHOR_CALIBRATION_SCORE, 6))
print("Three-model anchor calibration score:", round(THREE_MODEL_ANCHOR_CALIBRATION_SCORE, 6))


Coarse four-model simplex search:   0%|          | 0/35 [00:00<?, ?it/s]

Local 0.125 four-model refinement:   0%|          | 0/12 [00:00<?, ?it/s]

V17 fine 0.05 base-model weight refinement:   0%|          | 0/21 [00:00<?, ?it/s]

V17 fine candidates evaluated: 21
Top V17 fine-grid candidates:


,stage,v4_weight,v5_weight,v6_weight,mit_b4_weight,calibration_score
0,fine_0.05,0.075,0.125,0.25,0.550,0.726201
1,fine_0.05,0.050,0.125,0.25,0.575,0.726168
2,fine_0.05,0.075,0.125,0.30,0.500,0.726052
3,fine_0.05,0.050,0.125,0.20,0.625,0.726021
4,fine_0.05,0.050,0.075,0.25,0.625,0.725928
5,fine_0.05,0.125,0.125,0.20,0.550,0.725596
6,fine_0.05,0.000,0.125,0.30,0.575,0.725381
7,fine_0.05,0.075,0.175,0.25,0.500,0.725238
8,fine_0.05,0.125,0.125,0.30,0.450,0.725230
9,fine_0.05,0.000,0.125,0.20,0.675,0.725181


TOP FOUR-MODEL BLEND CANDIDATES


,stage,v4_weight,v5_weight,v6_weight,mit_b4_weight,calibration_score
0,fine_0.05,0.075,0.125,0.25,0.550,0.726201
1,fine_0.05,0.050,0.125,0.25,0.575,0.726168
2,fine_0.05,0.075,0.125,0.30,0.500,0.726052
3,fine_0.05,0.050,0.125,0.20,0.625,0.726021
4,fine_0.05,0.050,0.075,0.25,0.625,0.725928
5,fine_0.05,0.125,0.125,0.20,0.550,0.725596
6,refine,0.000,0.125,0.25,0.625,0.725590
7,refine,0.125,0.125,0.25,0.500,0.725426
8,fine_0.05,0.000,0.125,0.30,0.575,0.725381
9,fine_0.05,0.075,0.175,0.25,0.500,0.725238



Selected independent weights:
V4: 0.075
V5: 0.125
V6: 0.250
MiT-B4: 0.550
Weights sum: 1.0
Selected calibration thresholds: {'defect_a': 0.2800000011920929, 'defect_b': 0.29499998688697815, 'defect_c': 0.16500000655651093, 'defect_d': 0.20000000298023224}
V7 weight-anchor calibration score: 0.725029
Three-model anchor calibration score: 0.703888


In [13]:
# Build a conservative class-aware weight matrix from the already-computed calibration search.
# This reuses search_rows: no additional model inference or probability-cache generation is needed.
# The V12 global blend remains the fallback for every class.

GLOBAL_V12_WEIGHTS = np.asarray(SELECTED_WEIGHTS, dtype=np.float64)
MAX_CLASS_L1_DISTANCE = 0.50
MIN_CLASS_CALIBRATION_GAIN = {
    "defect_a": 0.00025,
    "defect_b": 0.00075,  # More conservative: this class has only 63 positives in all validation.
    "defect_c": 0.00025,
    "defect_d": 0.00025,
}

CLASS_WEIGHT_MATRIX = np.tile(GLOBAL_V12_WEIGHTS, (N_CLASSES, 1))
CLASS_WEIGHT_SELECTION = {}

for class_idx, class_name in enumerate(CLASSES):
    baseline_contribution = float(
        best_calibration_row["per_class"][class_name]["global_score_contribution"]
    )
    eligible_rows = []
    for row in search_rows:
        candidate_weights = np.asarray(row["weights"], dtype=np.float64)
        distance = float(np.abs(candidate_weights - GLOBAL_V12_WEIGHTS).sum())
        if distance <= MAX_CLASS_L1_DISTANCE + 1e-9:
            eligible_rows.append((row, distance))

    if not eligible_rows:
        raise RuntimeError(f"No conservative candidate weights found for {class_name}.")

    best_row, best_distance = max(
        eligible_rows,
        key=lambda item: float(item[0]["per_class"][class_name]["global_score_contribution"])
    )
    best_contribution = float(
        best_row["per_class"][class_name]["global_score_contribution"]
    )
    gain = best_contribution - baseline_contribution
    required_gain = MIN_CLASS_CALIBRATION_GAIN[class_name]

    if gain >= required_gain:
        CLASS_WEIGHT_MATRIX[class_idx] = np.asarray(best_row["weights"], dtype=np.float64)
        decision = "class-specific candidate accepted"
        selected_stage = best_row["stage"]
    else:
        decision = "retain V12 global weights (gain below conservative minimum)"
        selected_stage = "V12_global_fallback"
        best_distance = 0.0
        best_contribution = baseline_contribution
        gain = 0.0

    CLASS_WEIGHT_SELECTION[class_name] = {
        "decision": decision,
        "selected_stage": selected_stage,
        "weights": CLASS_WEIGHT_MATRIX[class_idx].tolist(),
        "calibration_contribution_gain": float(gain),
        "required_minimum_gain": float(required_gain),
        "l1_distance_from_v12_global": float(best_distance),
    }

CLASS_WEIGHT_MATRIX = validate_blend_weight_matrix(CLASS_WEIGHT_MATRIX).astype(np.float32)

# Re-optimise thresholds against the chosen per-class weight rows on CAL_IDX.
CLASSWISE_CALIBRATION_SCORE, CLASSWISE_CALIBRATION_THRESHOLDS, CLASSWISE_CALIBRATION_DETAILS = (
    optimise_thresholds_for_weights(CLASS_WEIGHT_MATRIX, CAL_IDX)
)

print("CLASS-AWARE WEIGHT SEARCH")
print("=" * 65)
print("V12 global reference:", GLOBAL_V12_WEIGHTS.tolist())
print("Conservative max L1 distance:", MAX_CLASS_L1_DISTANCE)
print("Class-aware calibration score:", round(CLASSWISE_CALIBRATION_SCORE, 6))
print("V12 global calibration score:", round(best_calibration_row["calibration_score"], 6))
print("Calibration score delta:", round(CLASSWISE_CALIBRATION_SCORE - best_calibration_row["calibration_score"], 6))
print("\nWeights by class [V4, V5, V6, MiT-B4]:")
for class_idx, class_name in enumerate(CLASSES):
    print(f"{class_name}: {CLASS_WEIGHT_MATRIX[class_idx].round(3).tolist()} | {CLASS_WEIGHT_SELECTION[class_name]['decision']}")
print("\nCalibration thresholds:", CLASSWISE_CALIBRATION_THRESHOLDS)

if not np.allclose(CLASS_WEIGHT_MATRIX.sum(axis=1), 1.0, atol=1e-6):
    raise RuntimeError("At least one selected class-specific weight row does not sum to one.")


CLASS-AWARE WEIGHT SEARCH
V12 global reference: [0.075, 0.125, 0.25, 0.55]
Conservative max L1 distance: 0.5
Class-aware calibration score: 0.730677
V12 global calibration score: 0.726201
Calibration score delta: 0.004477

Weights by class [V4, V5, V6, MiT-B4]:
defect_a: [0.125, 0.25, 0.125, 0.5] | class-specific candidate accepted
defect_b: [0.25, 0.125, 0.125, 0.5] | class-specific candidate accepted
defect_c: [0.0, 0.07500000298023224, 0.25, 0.675000011920929] | class-specific candidate accepted
defect_d: [0.07500000298023224, 0.125, 0.25, 0.550000011920929] | retain V12 global weights (gain below conservative minimum)

Calibration thresholds: {'defect_a': 0.4000000059604645, 'defect_b': 0.36000001430511475, 'defect_c': 0.12999999523162842, 'defect_d': 0.20000000298023224}


In [14]:
# Compare the V13 class-aware baseline with an expanded Colab V7 hybrid.
# All parameter selection happens on CAL_IDX. HOLDOUT_IDX is diagnostic only.
# The official metric has GLOBAL denominators across all image-class pairs.

HYBRID_BETA_GRID = np.round(np.arange(0.00, 1.001, 0.10), 2).astype(np.float32)
HYBRID_GATE_GRID = [None] + np.round(np.arange(0.05, 0.951, 0.05), 2).tolist()
HYBRID_MIN_GAIN = {
    "defect_a": 0.00040,
    "defect_b": 0.00100,  # Rare class: keep the acceptance rule conservative.
    "defect_c": 0.00040,
    "defect_d": 0.00040,
}
LOGIT_LOCAL_BETA_RADIUS = 0.20
LOGIT_LOCAL_GATE_RADIUS = 0.10


def _fuse_hybrid_probabilities(base_probs, colab_probs, beta, fusion_mode):
    """Combine probability maps by arithmetic probability pooling or logit pooling."""
    beta = float(beta)
    if fusion_mode == "probability":
        return (1.0 - beta) * base_probs + beta * colab_probs
    if fusion_mode == "logit":
        eps = 1e-4
        base_clipped = np.clip(np.asarray(base_probs, dtype=np.float32), eps, 1.0 - eps)
        colab_clipped = np.clip(np.asarray(colab_probs, dtype=np.float32), eps, 1.0 - eps)
        base_logits = np.log(base_clipped) - np.log1p(-base_clipped)
        colab_logits = np.log(colab_clipped) - np.log1p(-colab_clipped)
        mixed_logits = (1.0 - beta) * base_logits + beta * colab_logits
        return (1.0 / (1.0 + np.exp(-mixed_logits))).astype(np.float32, copy=False)
    raise ValueError(f"Unknown fusion mode: {fusion_mode}")


def _hybrid_global_contribution(
    dice_sum, empty_fp_pixels, correct_empty, class_empty_pairs,
    total_positive, total_empty
):
    """Additive per-class contribution to the GLOBAL competition score.

    The 0.20 false-positive baseline is allocated across classes in proportion
    to each class's empty-pair count. It must not be added once per class.
    Summing this function over all classes reproduces the official score.
    """
    total_empty_pixels = int(total_empty) * N_PIXELS
    class_empty_pixels = int(class_empty_pairs) * N_PIXELS
    return (
        0.70 * float(dice_sum) / max(int(total_positive), 1)
        + 0.20 * (class_empty_pixels - int(empty_fp_pixels)) / max(total_empty_pixels, 1)
        + 0.10 * float(correct_empty) / max(int(total_empty), 1)
    )


def _build_hybrid_histograms(class_idx, indices, beta, fusion_mode="probability"):
    indices = np.asarray(indices, dtype=np.int64)
    n_rows = len(indices)
    hist_all = np.zeros((n_rows, N_THRESHOLD_BINS), dtype=np.uint32)
    hist_positive = np.zeros_like(hist_all)
    class_gt = gt_counts[indices, class_idx]
    targets_positive = class_gt > 0

    for chunk_start in range(0, n_rows, CAL_CHUNK_SIZE):
        chunk_end = min(chunk_start + CAL_CHUNK_SIZE, n_rows)
        row_indices = indices[chunk_start:chunk_end]
        base_probs = combine_cached_probabilities(row_indices, class_idx, CLASS_WEIGHT_MATRIX)
        if COLAB_V7_AVAILABLE:
            colab_probs = np.asarray(val_colab_seg_probs[row_indices, class_idx], dtype=np.float32)
            combined = _fuse_hybrid_probabilities(base_probs, colab_probs, beta, fusion_mode)
        else:
            combined = base_probs
        targets = np.asarray(val_masks[row_indices, class_idx] > 0)
        combined_flat = combined.reshape(len(row_indices), -1)
        targets_flat = targets.reshape(len(row_indices), -1)
        bins = np.searchsorted(THRESHOLD_GRID, combined_flat, side="right").astype(np.uint8)
        for local_row in range(len(row_indices)):
            hist_all[chunk_start + local_row] = np.bincount(
                bins[local_row], minlength=N_THRESHOLD_BINS
            ).astype(np.uint32)
            hist_positive[chunk_start + local_row] = np.bincount(
                bins[local_row][targets_flat[local_row]], minlength=N_THRESHOLD_BINS
            ).astype(np.uint32)

    suffix_all = np.cumsum(hist_all[:, ::-1], axis=1, dtype=np.uint32)[:, ::-1]
    suffix_positive = np.cumsum(hist_positive[:, ::-1], axis=1, dtype=np.uint32)[:, ::-1]
    return suffix_all, suffix_positive, class_gt


def _search_one_class_hybrid(
    class_idx, indices, beta_candidates, gate_candidates, fusion_mode="probability"
):
    indices = np.asarray(indices, dtype=np.int64)
    total_positive = int((gt_counts[indices] > 0).sum())
    total_empty = int((gt_counts[indices] == 0).sum())
    if total_positive == 0 or total_empty == 0:
        raise ValueError("Calibration split must include positive and empty pairs for every class.")

    class_name = CLASSES[class_idx]
    classifier_scores = (
        np.asarray(val_colab_cls_probs[indices, class_idx], dtype=np.float32)
        if COLAB_V7_AVAILABLE else None
    )

    best = None
    baseline = None
    for beta in beta_candidates:
        suffix_all, suffix_positive, class_gt = _build_hybrid_histograms(
            class_idx, indices, beta, fusion_mode=fusion_mode
        )
        positive = class_gt > 0
        empty = ~positive
        class_empty_pairs = int(empty.sum())

        for gate_threshold in gate_candidates:
            if gate_threshold is None or not COLAB_V7_AVAILABLE:
                gate_pass = np.ones(len(indices), dtype=bool)
            else:
                gate_pass = classifier_scores >= float(gate_threshold)

            for threshold_index, threshold in enumerate(THRESHOLD_GRID):
                predicted_counts = suffix_all[:, threshold_index + 1]
                intersection_counts = suffix_positive[:, threshold_index + 1]
                gated_predicted = np.where(gate_pass, predicted_counts, 0)
                gated_intersection = np.where(gate_pass, intersection_counts, 0)

                if positive.any():
                    pair_dice = (
                        2.0 * gated_intersection[positive]
                        / np.maximum(gated_predicted[positive] + class_gt[positive], 1)
                    )
                    dice_sum = float(pair_dice.sum(dtype=np.float64))
                else:
                    dice_sum = 0.0

                empty_fp_pixels = int(gated_predicted[empty].sum(dtype=np.uint64))
                correct_empty = int((empty & ((~gate_pass) | (predicted_counts == 0))).sum())
                contribution = _hybrid_global_contribution(
                    dice_sum, empty_fp_pixels, correct_empty, class_empty_pairs,
                    total_positive, total_empty
                )
                row = {
                    "class": class_name,
                    "fusion_mode": fusion_mode,
                    "beta_colab_v7": float(beta),
                    "gate_threshold": None if gate_threshold is None else float(gate_threshold),
                    "mask_threshold": float(threshold),
                    "global_score_contribution": float(contribution),
                    "foreground_dice_class": float(dice_sum / max(int(positive.sum()), 1)),
                    "empty_pairs_class": class_empty_pairs,
                    "positive_pairs_class": int(positive.sum()),
                }
                if best is None or row["global_score_contribution"] > best["global_score_contribution"]:
                    best = row
                if (
                    fusion_mode == "probability" and float(beta) == 0.0
                    and gate_threshold is None
                ):
                    if baseline is None or row["global_score_contribution"] > baseline["global_score_contribution"]:
                        baseline = row
        del suffix_all, suffix_positive
        gc.collect()

    return best, baseline


def _local_logit_search_candidates(probability_best):
    """Small local logit-space search around the best probability-space candidate."""
    center_beta = float(probability_best["beta_colab_v7"])
    beta_values = sorted(set(float(np.clip(center_beta + delta, 0.0, 1.0))
                             for delta in (-LOGIT_LOCAL_BETA_RADIUS, -0.10, 0.0, 0.10, LOGIT_LOCAL_BETA_RADIUS)))
    center_gate = probability_best["gate_threshold"]
    if center_gate is None:
        gate_center = 0.50
        gate_values = [None] + [float(np.clip(gate_center + d, 0.05, 0.95))
                                for d in (-LOGIT_LOCAL_GATE_RADIUS, 0.0, LOGIT_LOCAL_GATE_RADIUS)]
    else:
        gate_center = float(center_gate)
        gate_values = [None] + [float(np.clip(gate_center + d, 0.05, 0.95))
                                for d in (-LOGIT_LOCAL_GATE_RADIUS, -0.05, 0.0, 0.05, LOGIT_LOCAL_GATE_RADIUS)]
    gate_values = list(dict.fromkeys(gate_values))
    return beta_values, gate_values


def optimise_fixed_hybrid_thresholds(beta_by_class, gate_by_class, fusion_by_class, indices):
    """Refit pixel thresholds for fixed class beta/gate/fusion choices."""
    selected_thresholds = {}
    per_class = {}
    total_score = 0.0
    for class_idx, class_name in enumerate(CLASSES):
        beta = float(beta_by_class[class_name])
        gate = gate_by_class[class_name]
        fusion_mode = fusion_by_class[class_name]
        best, _ = _search_one_class_hybrid(
            class_idx, indices, [beta], [gate], fusion_mode=fusion_mode
        )
        selected_thresholds[class_name] = float(best["mask_threshold"])
        per_class[class_name] = best
        total_score += float(best["global_score_contribution"])
    return float(total_score), selected_thresholds, per_class


def remove_small_components(mask, min_area):
    """Remove 8-connected foreground components smaller than min_area pixels."""
    mask = np.asarray(mask, dtype=bool)
    min_area = int(min_area)
    if min_area <= 1 or not mask.any():
        return mask
    labels, _ = ndimage.label(mask, structure=np.ones((3, 3), dtype=np.uint8))
    component_sizes = np.bincount(labels.ravel())
    keep = component_sizes >= min_area
    keep[0] = False
    return keep[labels]


def calibrate_component_min_areas(beta_by_class, gate_by_class, fusion_by_class, thresholds, indices):
    """Tune small-component removal on CAL_IDX after blend/gate/pixel threshold are fixed."""
    indices = np.asarray(indices, dtype=np.int64)
    total_positive = int((gt_counts[indices] > 0).sum())
    total_empty = int((gt_counts[indices] == 0).sum())
    total_empty_pixels = total_empty * N_PIXELS
    area_candidates = [0, 2, 4, 8, 16, 24, 32, 48, 64, 96, 128, 192]
    min_areas = {}
    rows = []
    per_class_details = {}
    for class_idx, class_name in enumerate(CLASSES):
        beta = float(beta_by_class[class_name])
        gate = gate_by_class[class_name]
        fusion_mode = fusion_by_class[class_name]
        threshold = float(thresholds[class_name])
        records = []
        for start in range(0, len(indices), CAL_CHUNK_SIZE):
            rows_idx = indices[start:start + CAL_CHUNK_SIZE]
            base = combine_cached_probabilities(rows_idx, class_idx, CLASS_WEIGHT_MATRIX)
            if COLAB_V7_AVAILABLE:
                aux = np.asarray(val_colab_seg_probs[rows_idx, class_idx], dtype=np.float32)
                combined = _fuse_hybrid_probabilities(base, aux, beta, fusion_mode)
                if gate is None:
                    gate_pass = np.ones(len(rows_idx), dtype=bool)
                else:
                    q = np.asarray(val_colab_cls_probs[rows_idx, class_idx], dtype=np.float32)
                    gate_pass = q >= float(gate)
            else:
                combined = base
                gate_pass = np.ones(len(rows_idx), dtype=bool)
            pred = (combined >= threshold) & gate_pass[:, None, None]
            target = np.asarray(val_masks[rows_idx, class_idx] > 0)
            target_counts = np.count_nonzero(target, axis=(1, 2))
            for local_i in range(len(rows_idx)):
                labels, _ = ndimage.label(pred[local_i], structure=np.ones((3, 3), dtype=np.uint8))
                sizes = np.bincount(labels.ravel())[1:]
                intersections = np.bincount(
                    labels.ravel(), weights=target[local_i].ravel().astype(np.float32)
                )[1:]
                records.append((int(target_counts[local_i]), sizes.astype(np.int64), intersections.astype(np.float64)))

        best_row = None
        baseline_row = None
        for min_area in area_candidates:
            dice_sum = 0.0
            fp_pixels = 0
            correct_empty = 0
            class_positive = 0
            class_empty = 0
            for target_count, sizes, intersections in records:
                keep = sizes >= int(min_area)
                pred_count = int(sizes[keep].sum()) if keep.any() else 0
                intersection = float(intersections[keep].sum()) if keep.any() else 0.0
                if target_count > 0:
                    dice_sum += 2.0 * intersection / max(pred_count + target_count, 1)
                    class_positive += 1
                else:
                    class_empty += 1
                    fp_pixels += pred_count
                    correct_empty += int(pred_count == 0)
            # Additive class contribution excluding the shared +0.20 FP baseline.
            contribution = (
                0.70 * dice_sum / max(total_positive, 1)
                - 0.20 * fp_pixels / max(total_empty_pixels, 1)
                + 0.10 * correct_empty / max(total_empty, 1)
            )
            row = {
                "class": class_name, "min_component_area": int(min_area),
                "global_score_contribution_excluding_shared_fp_baseline": float(contribution),
                "foreground_dice_class": float(dice_sum / max(class_positive, 1)),
                "empty_pairs_class": int(class_empty), "positive_pairs_class": int(class_positive),
                "empty_fp_pixels": int(fp_pixels), "correct_empty_pairs": int(correct_empty),
            }
            rows.append(row)
            if best_row is None or contribution > best_row["global_score_contribution_excluding_shared_fp_baseline"]:
                best_row = row
            if int(min_area) == 0:
                baseline_row = row
        if baseline_row is None or best_row is None:
            raise RuntimeError(f"Component-area search failed for {class_name}.")
        gain = float(best_row["global_score_contribution_excluding_shared_fp_baseline"]
                     - baseline_row["global_score_contribution_excluding_shared_fp_baseline"])
        # Require a small but non-trivial CAL gain before enabling a post-processing rule.
        chosen_area = int(best_row["min_component_area"]) if gain >= 0.00005 else 0
        min_areas[class_name] = chosen_area
        per_class_details[class_name] = {
            "selected_min_component_area": chosen_area,
            "best_candidate_min_component_area": int(best_row["min_component_area"]),
            "calibration_gain": gain,
        }
    return min_areas, pd.DataFrame(rows), per_class_details


def evaluate_hybrid_config(
    beta_by_class, gate_by_class, fusion_by_class, thresholds, indices, component_min_areas=None
):
    """Direct official-metric evaluator; denominators are global across all classes."""
    indices = np.asarray(indices, dtype=np.int64)
    positive_dice_sum = 0.0
    positive_pairs = 0
    empty_fp_pixels = 0
    empty_pixels = 0
    correct_empty_pairs = 0
    empty_pairs = 0
    class_metrics = {}

    for class_idx, class_name in enumerate(CLASSES):
        class_dice_sum = 0.0
        class_positive_pairs = 0
        class_empty_fp = 0
        class_empty_pixel_count = 0
        class_correct_empty = 0
        class_empty_pairs = 0
        beta = float(beta_by_class[class_name])
        gate = gate_by_class[class_name]
        fusion_mode = fusion_by_class[class_name]
        threshold = float(thresholds[class_name])

        for start in range(0, len(indices), CAL_CHUNK_SIZE):
            rows = indices[start:start + CAL_CHUNK_SIZE]
            base = combine_cached_probabilities(rows, class_idx, CLASS_WEIGHT_MATRIX)
            if COLAB_V7_AVAILABLE:
                colab = np.asarray(val_colab_seg_probs[rows, class_idx], dtype=np.float32)
                combined = _fuse_hybrid_probabilities(base, colab, beta, fusion_mode)
                if gate is not None:
                    q = np.asarray(val_colab_cls_probs[rows, class_idx], dtype=np.float32)
                    gate_pass = q >= float(gate)
                else:
                    gate_pass = np.ones(len(rows), dtype=bool)
            else:
                combined = base
                gate_pass = np.ones(len(rows), dtype=bool)
            prediction = (combined >= threshold) & gate_pass[:, None, None]
            min_component_area = 0 if component_min_areas is None else int(component_min_areas.get(class_name, 0))
            if min_component_area > 1:
                prediction = np.stack([
                    remove_small_components(row_mask, min_component_area) for row_mask in prediction
                ], axis=0)
            target = np.asarray(val_masks[rows, class_idx] > 0)
            target_count = np.count_nonzero(target, axis=(1, 2))
            pred_count = np.count_nonzero(prediction, axis=(1, 2))
            intersection = np.count_nonzero(prediction & target, axis=(1, 2))
            positive = target_count > 0
            empty = ~positive
            if positive.any():
                pair_dice = 2.0 * intersection[positive] / np.maximum(
                    pred_count[positive] + target_count[positive], 1
                )
                class_dice_sum += float(pair_dice.sum())
                class_positive_pairs += int(positive.sum())
            if empty.any():
                class_empty_pairs += int(empty.sum())
                class_empty_fp += int(pred_count[empty].sum())
                class_empty_pixel_count += int(empty.sum()) * N_PIXELS
                class_correct_empty += int((pred_count[empty] == 0).sum())

        positive_dice_sum += class_dice_sum
        positive_pairs += class_positive_pairs
        empty_fp_pixels += class_empty_fp
        empty_pixels += class_empty_pixel_count
        correct_empty_pairs += class_correct_empty
        empty_pairs += class_empty_pairs
        class_metrics[class_name] = {
            "positive_pairs": class_positive_pairs,
            "empty_pairs": class_empty_pairs,
            "foreground_dice": class_dice_sum / max(class_positive_pairs, 1),
            "false_positive_score": 1.0 - class_empty_fp / max(class_empty_pixel_count, 1),
            "empty_accuracy": class_correct_empty / max(class_empty_pairs, 1),
            "beta_colab_v7": beta,
            "gate_threshold": gate,
            "fusion_mode": fusion_mode,
            "mask_threshold": threshold,
        }

    foreground_dice = positive_dice_sum / max(positive_pairs, 1)
    false_positive_score = 1.0 - empty_fp_pixels / max(empty_pixels, 1)
    empty_accuracy = correct_empty_pairs / max(empty_pairs, 1)
    score = 0.70 * foreground_dice + 0.20 * false_positive_score + 0.10 * empty_accuracy
    return {
        "foreground_dice": float(foreground_dice),
        "false_positive_score": float(false_positive_score),
        "empty_accuracy": float(empty_accuracy),
        "weighted_score": float(score),
        "positive_pairs": int(positive_pairs),
        "empty_pairs": int(empty_pairs),
        "class_metrics": class_metrics,
    }


if COLAB_V7_AVAILABLE:
    print("CALIBRATING THE EXPANDED COLAB V7 HYBRID ON CAL_IDX ONLY")
    print("Probability beta candidates:", HYBRID_BETA_GRID.tolist())
    print("Classifier gate candidates:", HYBRID_GATE_GRID)
    print("Logit mode: local refinement around each class's best probability blend")
    CLASS_BETA = {}
    CLASS_GATE_THRESHOLDS = {}
    CLASS_FUSION_MODE = {}
    HYBRID_CAL_THRESHOLDS = {}
    HYBRID_SELECTION_ROWS = []
    HYBRID_CLASS_SELECTION = {}

    for class_idx, class_name in enumerate(CLASSES):
        best_probability, baseline_row = _search_one_class_hybrid(
            class_idx, CAL_IDX, HYBRID_BETA_GRID.tolist(), HYBRID_GATE_GRID,
            fusion_mode="probability"
        )
        if baseline_row is None:
            raise RuntimeError(f"Exact no-Colab baseline was not evaluated for {class_name}.")

        logit_beta_candidates, logit_gate_candidates = _local_logit_search_candidates(best_probability)
        best_logit, _ = _search_one_class_hybrid(
            class_idx, CAL_IDX, logit_beta_candidates, logit_gate_candidates,
            fusion_mode="logit"
        )
        best_row = max(
            [best_probability, best_logit],
            key=lambda row: float(row["global_score_contribution"])
        )
        gain = float(best_row["global_score_contribution"] - baseline_row["global_score_contribution"])
        min_gain = float(HYBRID_MIN_GAIN[class_name])
        accepted = gain >= min_gain
        chosen = best_row if accepted else baseline_row

        CLASS_BETA[class_name] = float(chosen["beta_colab_v7"])
        CLASS_GATE_THRESHOLDS[class_name] = chosen["gate_threshold"]
        CLASS_FUSION_MODE[class_name] = str(chosen["fusion_mode"])
        HYBRID_CAL_THRESHOLDS[class_name] = float(chosen["mask_threshold"])
        HYBRID_CLASS_SELECTION[class_name] = {
            "accepted": bool(accepted),
            "decision": "Colab V7 contribution accepted" if accepted else "retain V13 class-aware baseline",
            "baseline_calibration_contribution": float(baseline_row["global_score_contribution"]),
            "best_probability_candidate": best_probability,
            "best_local_logit_candidate": best_logit,
            "best_candidate_calibration_contribution": float(best_row["global_score_contribution"]),
            "observed_gain": gain,
            "required_gain": min_gain,
            "selected_beta_colab_v7": CLASS_BETA[class_name],
            "selected_gate_threshold": CLASS_GATE_THRESHOLDS[class_name],
            "selected_fusion_mode": CLASS_FUSION_MODE[class_name],
            "selected_calibration_mask_threshold": HYBRID_CAL_THRESHOLDS[class_name],
        }
        HYBRID_SELECTION_ROWS.append({
            **best_row,
            "baseline_contribution": baseline_row["global_score_contribution"],
            "gain": gain,
            "accepted": accepted,
            "selected": bool(chosen is best_row),
        })
        print(
            f"{class_name}: fusion={CLASS_FUSION_MODE[class_name]}, "
            f"beta={CLASS_BETA[class_name]:.2f}, gate={CLASS_GATE_THRESHOLDS[class_name]}, "
            f"mask_t={HYBRID_CAL_THRESHOLDS[class_name]:.3f}, "
            f"gain={gain:+.6f}, accepted={accepted}"
        )

    hybrid_calibration_score = float(sum(
        (selection["best_candidate_calibration_contribution"]
         if selection["accepted"] else selection["baseline_calibration_contribution"])
        for selection in HYBRID_CLASS_SELECTION.values()
    ))

    # Refit pixel thresholds on all validation images only after selecting beta/gates/fusion on CAL.
    full_validation_score, FINAL_THRESHOLDS, final_threshold_details = optimise_fixed_hybrid_thresholds(
        CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE, ALL_VAL_IDX
    )
    hybrid_calibration_thresholds = dict(HYBRID_CAL_THRESHOLDS)
else:
    CLASS_BETA = {name: 0.0 for name in CLASSES}
    CLASS_GATE_THRESHOLDS = {name: None for name in CLASSES}
    CLASS_FUSION_MODE = {name: "probability" for name in CLASSES}
    hybrid_calibration_score = float(CLASSWISE_CALIBRATION_SCORE)
    hybrid_calibration_thresholds = dict(CLASSWISE_CALIBRATION_THRESHOLDS)
    full_validation_score, FINAL_THRESHOLDS, final_threshold_details = optimise_thresholds_for_weights(
        CLASS_WEIGHT_MATRIX, ALL_VAL_IDX
    )
    HYBRID_CLASS_SELECTION = {
        name: {"accepted": False, "decision": "Colab V7 checkpoints missing; V13 fallback",
               "selected_beta_colab_v7": 0.0, "selected_gate_threshold": None,
               "selected_fusion_mode": "probability"}
        for name in CLASSES
    }
    HYBRID_SELECTION_ROWS = []

# Small-component removal is a post-processing experiment, calibrated only on CAL_IDX.
# It is applied only when it gives a minimum 0.00005 additive CAL contribution gain.
COMPONENT_MIN_AREAS, component_area_search_df, COMPONENT_AREA_SELECTION = calibrate_component_min_areas(
    CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
    hybrid_calibration_thresholds, CAL_IDX
)
component_calibration_metrics = evaluate_hybrid_config(
    CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
    hybrid_calibration_thresholds, CAL_IDX, component_min_areas=COMPONENT_MIN_AREAS
)
component_area_search_df.to_csv(OUTPUT_DIR / "component_min_area_search.csv", index=False)
print("Selected minimum component areas:", COMPONENT_MIN_AREAS)
print("Component-filtered CAL score:", round(component_calibration_metrics["weighted_score"], 6))

# V17 does not apply the calibrated component filter to final masks. In the V16
# run, it improved CAL but reduced the reserved HOLDOUT score (0.721759 with
# filtering vs 0.722160 without filtering). Keep the candidate areas for audit.
COMPONENT_MIN_AREAS_CALIBRATED = dict(COMPONENT_MIN_AREAS)
COMPONENT_MIN_AREAS = {class_name: 0 for class_name in CLASSES}
print("V17 final component filter (disabled):", COMPONENT_MIN_AREAS)


COLAB_V7_BLEND_ACTIVE = bool(
    COLAB_V7_AVAILABLE and any(
        float(CLASS_BETA[name]) > 0.0 or CLASS_GATE_THRESHOLDS[name] is not None
        for name in CLASSES
    )
)
print("Colab V7 branch selected for final test inference:", COLAB_V7_BLEND_ACTIVE)

# Direct metric-consistency checks: class contributions must add up to the global scorer.
if COLAB_V7_AVAILABLE:
    hybrid_calibration_metrics = evaluate_hybrid_config(
        CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
        hybrid_calibration_thresholds, CAL_IDX
    )
    if not np.isclose(
        hybrid_calibration_score, hybrid_calibration_metrics["weighted_score"],
        atol=2e-5, rtol=0.0
    ):
        raise RuntimeError(
            "Hybrid CAL contribution sum does not match the direct official evaluator: "
            f"{hybrid_calibration_score:.8f} vs {hybrid_calibration_metrics['weighted_score']:.8f}"
        )
    full_validation_metrics = evaluate_hybrid_config(
        CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
        FINAL_THRESHOLDS, ALL_VAL_IDX
    )
    full_validation_metrics_with_components = evaluate_hybrid_config(
        CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
        FINAL_THRESHOLDS, ALL_VAL_IDX, component_min_areas=COMPONENT_MIN_AREAS_CALIBRATED
    )
    if not np.isclose(
        full_validation_score, full_validation_metrics["weighted_score"],
        atol=2e-5, rtol=0.0
    ):
        raise RuntimeError(
            "Full-validation contribution sum does not match the direct official evaluator: "
            f"{full_validation_score:.8f} vs {full_validation_metrics['weighted_score']:.8f}"
        )
else:
    hybrid_calibration_metrics = None
    full_validation_metrics = None
    full_validation_metrics_with_components = None

# Evaluate the selected candidate and the V13 baseline on the same reserved diagnostic split.
hybrid_holdout_metrics_unfiltered = evaluate_hybrid_config(
    CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
    hybrid_calibration_thresholds, HOLDOUT_IDX
) if COLAB_V7_AVAILABLE else evaluate_mix_on_indices(
    CLASS_WEIGHT_MATRIX, CLASSWISE_CALIBRATION_THRESHOLDS, HOLDOUT_IDX
)
hybrid_holdout_metrics_component_candidate = evaluate_hybrid_config(
    CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
    hybrid_calibration_thresholds, HOLDOUT_IDX,
    component_min_areas=COMPONENT_MIN_AREAS_CALIBRATED
) if COLAB_V7_AVAILABLE else hybrid_holdout_metrics_unfiltered
hybrid_holdout_metrics = evaluate_hybrid_config(
    CLASS_BETA, CLASS_GATE_THRESHOLDS, CLASS_FUSION_MODE,
    hybrid_calibration_thresholds, HOLDOUT_IDX, component_min_areas=COMPONENT_MIN_AREAS
) if COLAB_V7_AVAILABLE else hybrid_holdout_metrics_unfiltered
v13_baseline_holdout_metrics = evaluate_mix_on_indices(
    CLASS_WEIGHT_MATRIX, CLASSWISE_CALIBRATION_THRESHOLDS, HOLDOUT_IDX
)

print("\nHYBRID HOLDOUT COMPARISON")
print("=" * 65)
print("V13 baseline holdout:", round(v13_baseline_holdout_metrics["weighted_score"], 6))
print("Hybrid candidate holdout:", round(hybrid_holdout_metrics["weighted_score"], 6))
print("Component candidate holdout:", round(hybrid_holdout_metrics_component_candidate["weighted_score"], 6))
print("Hybrid minus V13:", round(
    hybrid_holdout_metrics["weighted_score"] - v13_baseline_holdout_metrics["weighted_score"], 6
))
print("Selected beta by class:", CLASS_BETA)
print("Selected gate thresholds:", CLASS_GATE_THRESHOLDS)
print("Selected fusion modes:", CLASS_FUSION_MODE)
print("Final mask thresholds:", FINAL_THRESHOLDS)
print("CAL official score:", round(hybrid_calibration_score, 6))
print("Full-validation in-sample score (without component filter):", round(full_validation_score, 6))
if full_validation_metrics_with_components is not None:
    print("Full-validation in-sample score (with component filter):", round(full_validation_metrics_with_components["weighted_score"], 6))
if not COLAB_V7_AVAILABLE:
    print("\nFALLBACK NOTICE: this run did not include Colab V7 checkpoints; it is a V13-equivalent fallback.")
elif all(float(v) == 0.0 and CLASS_GATE_THRESHOLDS[k] is None for k, v in CLASS_BETA.items()):
    print("\nThe calibration search retained the V13 baseline for all classes; no Colab contribution was justified.")

FINAL_BLEND_CONFIG = {
    "notebook_version": "v18_frequency_detail_hybrid_evidence_gated",
    "baseline_public_score": BEST_PUBLIC_SCORE_BASELINE,
    "public_score_goal": PUBLIC_SCORE_TARGET,
    "models": ["V4 U-Net/ResNet34", "V5 SegFormer-B0", "V6 DeepLabV3+/ResNet50", "MiT-B4 SegFormer"],
    "baseline_class_aware_weights": {
        class_name: {
            model_name: float(CLASS_WEIGHT_MATRIX[class_idx, model_idx])
            for model_idx, model_name in enumerate(["v4", "v5", "v6", "mit_b4"])
        }
        for class_idx, class_name in enumerate(CLASSES)
    },
    "optional_colab_v7_models_available": bool(COLAB_V7_AVAILABLE),
    "colab_v7_checkpoint_paths": {k: str(v) if v is not None else None for k, v in COLAB_V7_PATHS.items()},
    "colab_v7_group_weights": COLAB_V7_GROUP_WEIGHTS,
    "colab_v7_beta_by_class": CLASS_BETA,
    "colab_v7_classifier_gate_by_class": CLASS_GATE_THRESHOLDS,
    "colab_v7_fusion_mode_by_class": CLASS_FUSION_MODE,
    "component_min_area_by_class": COMPONENT_MIN_AREAS,
    "calibrated_component_candidate_by_class": COMPONENT_MIN_AREAS_CALIBRATED,
    "component_candidate_holdout_metrics": hybrid_holdout_metrics_component_candidate,
    "component_area_selection": COMPONENT_AREA_SELECTION,
    "component_filtered_calibration_metrics": component_calibration_metrics,
    "hybrid_holdout_metrics_without_component_filter": hybrid_holdout_metrics_unfiltered,
    "calibration_mask_thresholds": hybrid_calibration_thresholds,
    "final_mask_thresholds": FINAL_THRESHOLDS,
    "hybrid_class_selection": HYBRID_CLASS_SELECTION,
    "hybrid_calibration_score": float(hybrid_calibration_score),
    "hybrid_calibration_metrics": hybrid_calibration_metrics,
    "full_validation_calibrated_objective": float(full_validation_score),
    "full_validation_metrics": full_validation_metrics,
    "full_validation_metrics_with_components": full_validation_metrics_with_components,
    "hybrid_holdout_metrics": hybrid_holdout_metrics,
    "v13_baseline_holdout_metrics": v13_baseline_holdout_metrics,
    "beta_search_grid": HYBRID_BETA_GRID.tolist(),
    "gate_search_grid": HYBRID_GATE_GRID,
    "pixel_threshold_step": 0.005,
    "logit_search": "local neighbourhood around best probability-space candidate",
    "inference_mode": MAIN_INFERENCE_MODE,
    "mit_b4_inference_mode": MIT_INFERENCE_MODE,
    "metric": "0.70 foreground Dice + 0.20 false-positive score + 0.10 empty-prediction accuracy",
    "metric_uses_global_pair_denominators": True,
    "metric_contribution_sum_assertion": "passed" if COLAB_V7_AVAILABLE else "not_applicable_fallback",
    "training_performed": False,
    "holdout_caveat": "All checkpoints were selected/trained using the original validation process; the internal holdout is diagnostic, not a fully untouched external estimate.",
    "public_score_caveat": "No local metric guarantees a higher Kaggle public score. Preserve the previous 0.65777 submission as the baseline."
}

CONFIG_PATH = OUTPUT_DIR / "selected_v18_frequency_detail_blend.json"
with open(CONFIG_PATH, "w") as f:
    json.dump(FINAL_BLEND_CONFIG, f, indent=2)

blend_search_df.to_csv(OUTPUT_DIR / "global_four_model_weight_search.csv", index=False)
fine_blend_search_df.to_csv(OUTPUT_DIR / "fine_four_model_weight_search.csv", index=False)
pd.DataFrame(CLASS_WEIGHT_SELECTION).T.to_csv(OUTPUT_DIR / "class_aware_weight_selection.csv")
pd.DataFrame(HYBRID_SELECTION_ROWS).to_csv(OUTPUT_DIR / "hybrid_beta_gate_search.csv", index=False)
pd.DataFrame(HYBRID_CLASS_SELECTION).T.to_csv(OUTPUT_DIR / "hybrid_class_selection.csv")
print("\nV18 retained-base hybrid configuration saved:", CONFIG_PATH)


CALIBRATING THE EXPANDED COLAB V7 HYBRID ON CAL_IDX ONLY
Probability beta candidates: [0.0, 0.10000000149011612, 0.20000000298023224, 0.30000001192092896, 0.4000000059604645, 0.5, 0.6000000238418579, 0.699999988079071, 0.800000011920929, 0.8999999761581421, 1.0]
Classifier gate candidates: [None, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
Logit mode: local refinement around each class's best probability blend
defect_a: fusion=probability, beta=0.30, gate=0.7, mask_t=0.250, gain=+0.004369, accepted=True
defect_b: fusion=probability, beta=0.70, gate=0.8, mask_t=0.370, gain=+0.005368, accepted=True
defect_c: fusion=probability, beta=0.00, gate=0.2, mask_t=0.130, gain=+0.000942, accepted=True
defect_d: fusion=logit, beta=0.20, gate=0.35, mask_t=0.250, gain=+0.000611, accepted=True
Selected minimum component areas: {'defect_a': 24, 'defect_b': 48, 'defect_c': 24, 'defect_d': 192}
Component-filtered CAL score: 0.742578
V17 final c

In [15]:
# V18 EVIDENCE GATE: calibrate the new model on CAL_IDX and diagnose on HOLDOUT_IDX.
# If training did not finish, or if the holdout guardrails fail, retain the V17 hybrid unchanged.

FREQUENCY_BASELINE_HOLDOUT_SCORE = float(hybrid_holdout_metrics["weighted_score"])
FREQUENCY_HOLDOUT_SCORE = FREQUENCY_BASELINE_HOLDOUT_SCORE
FREQUENCY_BASELINE_CAL_SCORE = float(hybrid_calibration_score)
FREQUENCY_CANDIDATE_CAL_SCORE = FREQUENCY_BASELINE_CAL_SCORE
FREQUENCY_ALPHA_BY_CLASS = {name: 0.0 for name in CLASSES}
V17_BASE_THRESHOLDS = dict(FINAL_THRESHOLDS)
FREQUENCY_SELECTED_THRESHOLDS = dict(V17_BASE_THRESHOLDS)
FREQUENCY_BLEND_ACTIVE = False
FREQUENCY_ACCEPTANCE_REASON = "frequency model unavailable; V17 fallback retained"
FREQUENCY_SEARCH_ROWS = []


def _v17_cached_probability(class_idx, row_indices):
    base = combine_cached_probabilities(row_indices, class_idx, CLASS_WEIGHT_MATRIX)
    if COLAB_V7_AVAILABLE:
        aux = np.asarray(val_colab_seg_probs[row_indices, class_idx], dtype=np.float32)
        base = _fuse_hybrid_probabilities(
            base, aux,
            CLASS_BETA[CLASSES[class_idx]],
            CLASS_FUSION_MODE[CLASSES[class_idx]],
        )
    return np.asarray(base, dtype=np.float32)


def _v17_cached_gate(class_idx, row_indices):
    class_name = CLASSES[class_idx]
    gate = CLASS_GATE_THRESHOLDS.get(class_name)
    if gate is None or not COLAB_V7_AVAILABLE:
        return np.ones(len(row_indices), dtype=bool)
    scores = np.asarray(val_colab_cls_probs[row_indices, class_idx], dtype=np.float32)
    return scores >= float(gate)


def _evaluate_v18_frequency_blend(indices, alpha_by_class, thresholds_by_class):
    indices = np.asarray(indices, dtype=np.int64)
    total_positive = int((gt_counts[indices] > 0).sum())
    total_empty = int((gt_counts[indices] == 0).sum())
    total_empty_pixels = total_empty * N_PIXELS
    dice_sum_all = 0.0
    fp_pixels_all = 0
    correct_empty_all = 0
    class_metrics = {}
    for class_idx, class_name in enumerate(CLASSES):
        class_dice_sum = 0.0
        class_positive = 0
        class_empty = 0
        class_fp = 0
        class_correct_empty = 0
        alpha = float(alpha_by_class[class_name])
        threshold = float(thresholds_by_class[class_name])
        for start in range(0, len(indices), CAL_CHUNK_SIZE):
            rows = indices[start:start + CAL_CHUNK_SIZE]
            old_probs = _v17_cached_probability(class_idx, rows)
            if alpha > 0:
                freq_probs = np.asarray(val_frequency_probs[rows, class_idx], dtype=np.float32)
                probs = (1.0 - alpha) * old_probs + alpha * freq_probs
            else:
                probs = old_probs
            gate_pass = _v17_cached_gate(class_idx, rows)
            prediction = (probs >= threshold) & gate_pass[:, None, None]
            target = np.asarray(val_masks[rows, class_idx] > 0)
            target_count = np.count_nonzero(target, axis=(1, 2))
            pred_count = np.count_nonzero(prediction, axis=(1, 2))
            intersection = np.count_nonzero(prediction & target, axis=(1, 2))
            positive = target_count > 0
            empty = ~positive
            if positive.any():
                class_dice_sum += float((2.0 * intersection[positive] / np.maximum(pred_count[positive] + target_count[positive], 1)).sum())
                class_positive += int(positive.sum())
            if empty.any():
                class_empty += int(empty.sum())
                class_fp += int(pred_count[empty].sum())
                class_correct_empty += int((pred_count[empty] == 0).sum())
        dice_sum_all += class_dice_sum
        fp_pixels_all += class_fp
        correct_empty_all += class_correct_empty
        class_metrics[class_name] = {
            "foreground_dice": float(class_dice_sum / max(class_positive, 1)),
            "positive_pairs": int(class_positive),
            "empty_pairs": int(class_empty),
            "empty_fp_pixels": int(class_fp),
            "empty_accuracy": float(class_correct_empty / max(class_empty, 1)),
        }
    score = (
        0.70 * dice_sum_all / max(total_positive, 1)
        + 0.20 * (total_empty_pixels - fp_pixels_all) / max(total_empty_pixels, 1)
        + 0.10 * correct_empty_all / max(total_empty, 1)
    )
    return {
        "weighted_score": float(score),
        "foreground_dice": float(dice_sum_all / max(total_positive, 1)),
        "false_positive_score": float(1.0 - fp_pixels_all / max(total_empty_pixels, 1)),
        "empty_accuracy": float(correct_empty_all / max(total_empty, 1)),
        "class_metrics": class_metrics,
    }


if FREQUENCY_MODEL_AVAILABLE:
    try:
        val_frequency_probs = np.load(FREQUENCY_VAL_PROBS_PATH, mmap_mode="r")
        val_frequency_cls = np.load(FREQUENCY_VAL_CLS_PATH, mmap_mode="r")
        expected_frequency_shape = (len(val_ids), N_CLASSES, HEIGHT, WIDTH)
        if val_frequency_probs.shape != expected_frequency_shape:
            raise ValueError(f"Invalid frequency cache shape: {val_frequency_probs.shape}")
        if not np.isfinite(val_frequency_cls).all():
            raise ValueError("Frequency classifier cache contains NaN or infinity.")

        # Baseline score is recomputed using its exact existing V17 thresholds.
        baseline_cal = _evaluate_v18_frequency_blend(
            CAL_IDX, {name: 0.0 for name in CLASSES}, FINAL_THRESHOLDS
        )
        baseline_holdout = _evaluate_v18_frequency_blend(
            HOLDOUT_IDX, {name: 0.0 for name in CLASSES}, FINAL_THRESHOLDS
        )
        FREQUENCY_BASELINE_CAL_SCORE = baseline_cal["weighted_score"]
        FREQUENCY_BASELINE_HOLDOUT_SCORE = baseline_holdout["weighted_score"]

        # Build candidate threshold histograms once per (class, alpha); gate is applied
        # to image-level counts after threshold lookup, matching V17's metric evaluator.
        alpha_grid = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
        min_cal_gain = {"defect_a": 0.00015, "defect_b": 0.00020, "defect_c": 0.00015, "defect_d": 0.00015}
        n_cal_positive = int((gt_counts[CAL_IDX] > 0).sum())
        n_cal_empty = int((gt_counts[CAL_IDX] == 0).sum())

        for class_idx, class_name in enumerate(CLASSES):
            # The exact V17 threshold and alpha=0 remain a hard fallback.
            best_row = {
                "class": class_name, "alpha": 0.0,
                "threshold": float(FINAL_THRESHOLDS[class_name]),
                "calibration_contribution": None,
                "calibration_gain_over_v17": 0.0,
            }
            class_gt = gt_counts[CAL_IDX, class_idx]
            positive = class_gt > 0
            empty = ~positive
            class_empty_pairs = int(empty.sum())
            base_contribution = None

            # Evaluate alpha=0 contribution using the existing fixed threshold.
            baseline_class_eval = _evaluate_v18_frequency_blend(
                CAL_IDX,
                {name: (0.0 if name != class_name else 0.0) for name in CLASSES},
                FINAL_THRESHOLDS,
            )
            # Contribution is measured directly for this class so the class-wise search
            # uses the global denominators from all CAL image-class pairs.
            rows_all = CAL_IDX
            base_hist_all = np.zeros((len(rows_all), N_THRESHOLD_BINS), dtype=np.uint32)
            base_hist_pos = np.zeros_like(base_hist_all)
            for chunk_start in range(0, len(rows_all), CAL_CHUNK_SIZE):
                chunk_end = min(chunk_start + CAL_CHUNK_SIZE, len(rows_all))
                rows = rows_all[chunk_start:chunk_end]
                probs = _v17_cached_probability(class_idx, rows)
                targets = np.asarray(val_masks[rows, class_idx] > 0)
                flat_probs = probs.reshape(len(rows), -1)
                flat_targets = targets.reshape(len(rows), -1)
                bins = np.searchsorted(THRESHOLD_GRID, flat_probs, side="right").astype(np.uint8)
                for local_row in range(len(rows)):
                    base_hist_all[chunk_start + local_row] = np.bincount(bins[local_row], minlength=N_THRESHOLD_BINS)
                    base_hist_pos[chunk_start + local_row] = np.bincount(bins[local_row][flat_targets[local_row]], minlength=N_THRESHOLD_BINS)
            base_suffix_all = np.cumsum(base_hist_all[:, ::-1], axis=1, dtype=np.uint32)[:, ::-1]
            base_suffix_pos = np.cumsum(base_hist_pos[:, ::-1], axis=1, dtype=np.uint32)[:, ::-1]
            threshold_index = int(np.argmin(np.abs(THRESHOLD_GRID - float(FINAL_THRESHOLDS[class_name]))))
            pred0 = base_suffix_all[:, threshold_index + 1]
            inter0 = base_suffix_pos[:, threshold_index + 1]
            gate0 = np.asarray([_v17_cached_gate(class_idx, np.asarray([idx]))[0] for idx in rows_all], dtype=bool)
            pred0 = np.where(gate0, pred0, 0)
            inter0 = np.where(gate0, inter0, 0)
            dice0 = float((2.0 * inter0[positive] / np.maximum(pred0[positive] + class_gt[positive], 1)).sum())
            fp0 = int(pred0[empty].sum(dtype=np.uint64))
            ce0 = int((empty & ((~gate0) | (pred0 == 0))).sum())
            base_contribution = _hybrid_global_contribution(
                dice0, fp0, ce0, class_empty_pairs, n_cal_positive, n_cal_empty
            )
            best_row["calibration_contribution"] = float(base_contribution)

            for alpha in alpha_grid:
                hist_all = np.zeros((len(rows_all), N_THRESHOLD_BINS), dtype=np.uint32)
                hist_pos = np.zeros_like(hist_all)
                for chunk_start in range(0, len(rows_all), CAL_CHUNK_SIZE):
                    chunk_end = min(chunk_start + CAL_CHUNK_SIZE, len(rows_all))
                    rows = rows_all[chunk_start:chunk_end]
                    old_probs = _v17_cached_probability(class_idx, rows)
                    freq_probs = np.asarray(val_frequency_probs[rows, class_idx], dtype=np.float32)
                    probs = (1.0 - alpha) * old_probs + alpha * freq_probs
                    targets = np.asarray(val_masks[rows, class_idx] > 0)
                    flat_probs = probs.reshape(len(rows), -1)
                    flat_targets = targets.reshape(len(rows), -1)
                    bins = np.searchsorted(THRESHOLD_GRID, flat_probs, side="right").astype(np.uint8)
                    for local_row in range(len(rows)):
                        hist_all[chunk_start + local_row] = np.bincount(bins[local_row], minlength=N_THRESHOLD_BINS)
                        hist_pos[chunk_start + local_row] = np.bincount(bins[local_row][flat_targets[local_row]], minlength=N_THRESHOLD_BINS)
                suffix_all = np.cumsum(hist_all[:, ::-1], axis=1, dtype=np.uint32)[:, ::-1]
                suffix_pos = np.cumsum(hist_pos[:, ::-1], axis=1, dtype=np.uint32)[:, ::-1]
                gate_scores = (
                    np.asarray(val_colab_cls_probs[rows_all, class_idx], dtype=np.float32)
                    if COLAB_V7_AVAILABLE and CLASS_GATE_THRESHOLDS[class_name] is not None
                    else None
                )
                gate_thr = CLASS_GATE_THRESHOLDS.get(class_name)
                gate_pass = np.ones(len(rows_all), dtype=bool) if gate_scores is None else (gate_scores >= float(gate_thr))
                for t_idx, threshold in enumerate(THRESHOLD_GRID):
                    predicted = np.where(gate_pass, suffix_all[:, t_idx + 1], 0)
                    intersection = np.where(gate_pass, suffix_pos[:, t_idx + 1], 0)
                    dice_sum = float((2.0 * intersection[positive] / np.maximum(predicted[positive] + class_gt[positive], 1)).sum())
                    fp = int(predicted[empty].sum(dtype=np.uint64))
                    correct_empty = int((empty & ((~gate_pass) | (predicted == 0))).sum())
                    contribution = _hybrid_global_contribution(
                        dice_sum, fp, correct_empty, class_empty_pairs, n_cal_positive, n_cal_empty
                    )
                    if contribution > best_row["calibration_contribution"]:
                        best_row = {
                            "class": class_name, "alpha": float(alpha),
                            "threshold": float(threshold),
                            "calibration_contribution": float(contribution),
                            "calibration_gain_over_v17": float(contribution - base_contribution),
                        }
                del hist_all, hist_pos, suffix_all, suffix_pos
                gc.collect()

            # Do not activate a weak/noisy per-class addition on the calibration split.
            if best_row["alpha"] > 0 and best_row["calibration_gain_over_v17"] < min_cal_gain[class_name]:
                best_row = {
                    "class": class_name, "alpha": 0.0,
                    "threshold": float(FINAL_THRESHOLDS[class_name]),
                    "calibration_contribution": float(base_contribution),
                    "calibration_gain_over_v17": 0.0,
                }
            FREQUENCY_SEARCH_ROWS.append(best_row)

        candidate_alpha = {row["class"]: float(row["alpha"]) for row in FREQUENCY_SEARCH_ROWS}
        candidate_thresholds = {row["class"]: float(row["threshold"]) for row in FREQUENCY_SEARCH_ROWS}
        candidate_cal = _evaluate_v18_frequency_blend(CAL_IDX, candidate_alpha, candidate_thresholds)
        candidate_holdout = _evaluate_v18_frequency_blend(HOLDOUT_IDX, candidate_alpha, candidate_thresholds)
        baseline_holdout = _evaluate_v18_frequency_blend(
            HOLDOUT_IDX, {name: 0.0 for name in CLASSES}, FINAL_THRESHOLDS
        )
        FREQUENCY_CANDIDATE_CAL_SCORE = float(candidate_cal["weighted_score"])
        FREQUENCY_HOLDOUT_SCORE = float(candidate_holdout["weighted_score"])
        baseline_b = float(baseline_holdout["class_metrics"]["defect_b"]["foreground_dice"])
        candidate_b = float(candidate_holdout["class_metrics"]["defect_b"]["foreground_dice"])
        holdout_gain = FREQUENCY_HOLDOUT_SCORE - FREQUENCY_BASELINE_HOLDOUT_SCORE
        defect_b_delta = candidate_b - baseline_b

        FREQUENCY_SEARCH_DF = pd.DataFrame(FREQUENCY_SEARCH_ROWS)
        FREQUENCY_SEARCH_DF.to_csv(OUTPUT_DIR / "v18_frequency_alpha_threshold_search.csv", index=False)
        print("V18 frequency candidate CAL score:", round(FREQUENCY_CANDIDATE_CAL_SCORE, 6))
        print("V17 baseline CAL score:", round(FREQUENCY_BASELINE_CAL_SCORE, 6))
        print("V18 frequency candidate holdout:", round(FREQUENCY_HOLDOUT_SCORE, 6))
        print("V17 baseline holdout:", round(FREQUENCY_BASELINE_HOLDOUT_SCORE, 6))
        print("Holdout gain:", round(holdout_gain, 6))
        print("defect_b foreground Dice delta:", round(defect_b_delta, 6))
        display(FREQUENCY_SEARCH_DF.round(6))

        # Strict evidence gate: modest total improvement + no material rare-class regression.
        if holdout_gain >= 0.00025 and defect_b_delta >= -0.0025:
            FREQUENCY_BLEND_ACTIVE = True
            FREQUENCY_ALPHA_BY_CLASS = candidate_alpha
            FREQUENCY_SELECTED_THRESHOLDS = candidate_thresholds
            FINAL_THRESHOLDS = dict(candidate_thresholds)
            FREQUENCY_ACCEPTANCE_REASON = (
                f"accepted: holdout gain={holdout_gain:.6f}; defect_b Dice delta={defect_b_delta:.6f}"
            )
        else:
            FREQUENCY_BLEND_ACTIVE = False
            FREQUENCY_ALPHA_BY_CLASS = {name: 0.0 for name in CLASSES}
            FREQUENCY_SELECTED_THRESHOLDS = dict(V17_BASE_THRESHOLDS)
            FINAL_THRESHOLDS = dict(V17_BASE_THRESHOLDS)
            FREQUENCY_ACCEPTANCE_REASON = (
                f"rejected; holdout gain={holdout_gain:.6f}; defect_b Dice delta={defect_b_delta:.6f}; V17 fallback retained"
            )
        print("V18 frequency blend active:", FREQUENCY_BLEND_ACTIVE)
        print("Decision:", FREQUENCY_ACCEPTANCE_REASON)

    except Exception as frequency_calibration_error:
        FREQUENCY_BLEND_ACTIVE = False
        FREQUENCY_ALPHA_BY_CLASS = {name: 0.0 for name in CLASSES}
        FREQUENCY_ACCEPTANCE_REASON = (
            f"calibration failed safely: {type(frequency_calibration_error).__name__}: "
            f"{str(frequency_calibration_error)[:220]}; V17 fallback retained"
        )
        print("V18 frequency calibration failed; retaining V17 blend.")
        print(FREQUENCY_ACCEPTANCE_REASON)
else:
    print("V18 frequency blend not evaluated; retaining V17 hybrid.")

print("V18 final frequency alpha:", FREQUENCY_ALPHA_BY_CLASS)
print("V18 final thresholds:", FINAL_THRESHOLDS)


V18 frequency candidate CAL score: 0.749503
V17 baseline CAL score: 0.741951
V18 frequency candidate holdout: 0.734251
V17 baseline holdout: 0.724721
Holdout gain: 0.00953
defect_b foreground Dice delta: 0.0


,class,alpha,threshold,calibration_contribution,calibration_gain_over_v17
0,defect_a,0.25,0.260,0.129605,0.002909
1,defect_b,0.00,0.370,0.100110,0.000000
2,defect_c,0.15,0.150,0.317118,0.004642
3,defect_d,0.00,0.255,0.202669,0.000000


V18 frequency blend active: True
Decision: accepted: holdout gain=0.009530; defect_b Dice delta=0.000000
V18 final frequency alpha: {'defect_a': 0.25, 'defect_b': 0.0, 'defect_c': 0.15, 'defect_d': 0.0}
V18 final thresholds: {'defect_a': 0.25999999046325684, 'defect_b': 0.3700000047683716, 'defect_c': 0.15000000596046448, 'defect_d': 0.2549999952316284}


In [16]:
SAMPLE_SUBMISSION_PATH = DATA_ROOT / "sample_submission.csv"
sample_submission = pd.read_csv(
    SAMPLE_SUBMISSION_PATH,
    dtype={"ID": str, "image_id": str, "defect_type": str, "mask_rle": str},
    keep_default_na=False
)

EXPECTED_COLUMNS = ["ID", "image_id", "defect_type", "mask_rle"]
if sample_submission.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(f"Unexpected sample submission columns: {sample_submission.columns.tolist()}")

if sample_submission.duplicated(subset=["image_id", "defect_type"]).any():
    raise ValueError("Sample submission contains duplicate image-class pairs.")

if set(sample_submission["defect_type"].unique()) != set(CLASSES):
    raise ValueError("Sample submission class names do not match the model class order.")

TEST_IDS = sample_submission["image_id"].drop_duplicates().astype(str).tolist()
if len(TEST_IDS) != 3000 or len(sample_submission) != 12000:
    raise ValueError(
        f"Unexpected test template dimensions: {len(TEST_IDS)} images, {len(sample_submission)} rows."
    )

class SurfaceGuardTestDataset(Dataset):
    def __init__(self, image_ids, data_root, resample):
        self.image_ids = list(map(str, image_ids))
        self.data_root = Path(data_root)
        self.resample = resample

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_id = self.image_ids[index]
        image_path = self.data_root / "test" / f"{image_id}.jpg"
        if not image_path.is_file():
            raise FileNotFoundError(f"Test image not found: {image_path}")
        with Image.open(image_path) as source:
            image = source.convert("L").resize((WIDTH, HEIGHT), resample=self.resample)
            image = np.asarray(image, dtype=np.float32) / 255.0
        mean = float(image.mean())
        std = max(float(image.std()), 0.05)
        image = np.ascontiguousarray((image - mean) / std, dtype=np.float32)
        return torch.from_numpy(image).unsqueeze(0), image_id


test_dataset_main = SurfaceGuardTestDataset(TEST_IDS, DATA_ROOT, MAIN_RESAMPLE)
test_dataset_mit = SurfaceGuardTestDataset(TEST_IDS, DATA_ROOT, MIT_RESAMPLE)

test_loader_main = DataLoader(
    test_dataset_main,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=False
)
test_loader_mit = DataLoader(
    test_dataset_mit,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=False
)

if len(test_loader_main) != len(test_loader_mit):
    raise RuntimeError("Main and MiT test loaders have different batch counts.")

print("Test images:", len(TEST_IDS))
print("Submission rows:", len(sample_submission))
print("Test batches:", len(test_loader_main))
print("Batch size:", TEST_BATCH_SIZE)
print("Official test template check: PASSED")

Test images: 3000
Submission rows: 12000
Test batches: 3000
Batch size: 1
Official test template check: PASSED


In [17]:
def rle_encode_column_major(mask):
    pixels = np.asarray(mask, dtype=np.uint8).reshape(-1, order="F")
    if not np.any(pixels):
        return " "
    padded = np.concatenate([
        np.array([0], dtype=np.uint8),
        pixels,
        np.array([0], dtype=np.uint8)
    ])
    changes = np.flatnonzero(padded[1:] != padded[:-1]) + 1
    starts = changes[0::2]
    ends = changes[1::2]
    lengths = ends - starts
    encoded = np.empty(starts.size * 2, dtype=np.int64)
    encoded[0::2] = starts
    encoded[1::2] = lengths
    return " ".join(map(str, encoded.tolist()))


def rle_decode_for_check(rle, height=HEIGHT, width=WIDTH):
    flat = np.zeros(height * width, dtype=np.uint8)
    if str(rle).strip() == "":
        return flat.reshape((height, width), order="F")
    values = np.asarray(str(rle).split(), dtype=np.int64)
    if len(values) % 2:
        raise ValueError("RLE does not contain start-length pairs.")
    starts = values[0::2] - 1
    lengths = values[1::2]
    for start, length in zip(starts, lengths):
        if start < 0 or length <= 0 or start + length > height * width:
            raise ValueError("RLE run is outside image bounds.")
        flat[start:start + length] = 1
    return flat.reshape((height, width), order="F")


def validate_rle(rle, height=HEIGHT, width=WIDTH):
    if not isinstance(rle, str):
        return False
    if rle.strip() == "":
        return True
    try:
        values = np.asarray(rle.split(), dtype=np.int64)
        if len(values) % 2:
            return False
        starts = values[0::2]
        lengths = values[1::2]
        if np.any(starts < 1) or np.any(lengths < 1):
            return False
        ends = starts + lengths - 1
        if np.any(ends > height * width):
            return False
        if len(starts) > 1 and np.any(starts[1:] <= ends[:-1]):
            return False
        return True
    except (ValueError, TypeError, OverflowError):
        return False


# Explicit RLE unit test: the second run starts at position 9 in column-major order.
_rle_test = np.zeros((3, 4), dtype=np.uint8)
_rle_test[0, 0] = 1
_rle_test[1, 0] = 1
_rle_test[2, 2] = 1
_rle_text = rle_encode_column_major(_rle_test)
assert _rle_text == "1 2 9 1", f"RLE self-test failed: {_rle_text}"
assert np.array_equal(rle_decode_for_check(_rle_text, 3, 4), _rle_test)
assert validate_rle(_rle_text, 3, 4)
print("Column-major RLE encode/decode self-test: PASSED")

Column-major RLE encode/decode self-test: PASSED


In [18]:
print("FINAL TEST INFERENCE — V18 FREQUENCY-DETAIL HYBRID / V17 FALLBACK")
print("=" * 70)
print("Optional Colab V7 checkpoints available:", COLAB_V7_AVAILABLE)
print("Colab V7 contribution/gate selected:", COLAB_V7_BLEND_ACTIVE)
print("Class-aware V13 weights:", FINAL_BLEND_CONFIG["baseline_class_aware_weights"])
print("Colab V7 beta by class:", CLASS_BETA)
print("Colab V7 classifier gates:", CLASS_GATE_THRESHOLDS)
print("Colab V7 fusion modes:", CLASS_FUSION_MODE)
print("Main inference mode:", MAIN_INFERENCE_MODE)
print("MiT inference mode:", MIT_INFERENCE_MODE)
print("Pixel thresholds:", FINAL_THRESHOLDS)

if not np.allclose(CLASS_WEIGHT_MATRIX.sum(axis=1), 1.0, atol=1e-6):
    raise RuntimeError("One or more class-specific V13 weight rows do not sum to one.")

# Restore the optional V18 frequency model after the eight established checkpoints are loaded.
frequency_test_model = None
if FREQUENCY_BLEND_ACTIVE:
    try:
        frequency_test_checkpoint = torch.load(
            FREQUENCY_MODEL_PATH, map_location="cpu", weights_only=False
        )
        frequency_test_model = SurfaceGuardFrequencyDetailNet(N_CLASSES)
        frequency_test_model.load_state_dict(frequency_test_checkpoint["model_state_dict"], strict=True)
        del frequency_test_checkpoint
        frequency_test_model = frequency_test_model.to(DEVICE).eval()
        print("V18 frequency model loaded for test inference.")
    except Exception as frequency_test_load_error:
        print("V18 frequency model could not be loaded for test; falling back to V17.")
        print(type(frequency_test_load_error).__name__, str(frequency_test_load_error)[:240])
        frequency_test_model = None
        FREQUENCY_BLEND_ACTIVE = False
        FREQUENCY_ALPHA_BY_CLASS = {name: 0.0 for name in CLASSES}
        FINAL_THRESHOLDS = dict(V17_BASE_THRESHOLDS)
        FREQUENCY_ACCEPTANCE_REASON = "test checkpoint load failed; V17 thresholds and blend restored"

prediction_lookup = {}
processed_images = 0
start_time = time.time()
progress_path = OUTPUT_DIR / "test_predictions_progress.csv"


def save_test_progress(predictions, processed_count):
    rows = [
        {"image_id": image_id, "defect_type": class_name,
         "mask_rle": predictions[(image_id, class_name)]}
        for image_id in TEST_IDS[:processed_count]
        for class_name in CLASSES
        if (image_id, class_name) in predictions
    ]
    progress_df = pd.DataFrame(rows, columns=["image_id", "defect_type", "mask_rle"])
    temp_path = progress_path.with_name(progress_path.stem + ".tmp.csv")
    progress_df.to_csv(temp_path, index=False)
    os.replace(temp_path, progress_path)


with torch.inference_mode():
    loader_iterator = zip(test_loader_main, test_loader_mit)
    for (main_images, main_ids), (mit_images, mit_ids) in tqdm(
        loader_iterator, total=len(test_loader_main), desc="Test inference: V18 + V17 hybrid"
    ):
        main_ids = [str(x) for x in main_ids]
        mit_ids = [str(x) for x in mit_ids]
        if main_ids != mit_ids:
            raise RuntimeError("Main and MiT test-loader image order mismatch.")

        # Run the optional auxiliary group first. If it triggers CUDA OOM, the
        # global device is switched before the V13 models are evaluated.
        if COLAB_V7_BLEND_ACTIVE:
            colab_seg, colab_cls = predict_colab_v7_group(main_images)
        else:
            colab_seg, colab_cls = None, None

        p4_probs, p5_probs, p6_probs = predict_main_models(main_images)
        mit_probs = predict_mit_model(mit_images)

        # A later model can still trigger CUDA OOM after an earlier model has
        # returned CUDA tensors. Re-home every probability tensor to the current
        # DEVICE before any arithmetic to avoid CPU/CUDA device-mismatch errors.
        p4_probs = p4_probs.to(DEVICE).half().float()
        p5_probs = p5_probs.to(DEVICE).half().float()
        p6_probs = p6_probs.to(DEVICE).half().float()
        mit_probs = mit_probs.to(DEVICE).half().float()

        # Recreate small tensors inside the loop because DEVICE can change after
        # automatic OOM recovery; tensors created before the fallback become stale.
        class_weight_tensors = torch.as_tensor(
            CLASS_WEIGHT_MATRIX, dtype=torch.float32, device=DEVICE
        )
        baseline_probs = (
            class_weight_tensors[:, 0].view(1, N_CLASSES, 1, 1) * p4_probs
            + class_weight_tensors[:, 1].view(1, N_CLASSES, 1, 1) * p5_probs
            + class_weight_tensors[:, 2].view(1, N_CLASSES, 1, 1) * p6_probs
            + class_weight_tensors[:, 3].view(1, N_CLASSES, 1, 1) * mit_probs
        )
        threshold_tensor = torch.as_tensor(
            [FINAL_THRESHOLDS[name] for name in CLASSES], dtype=torch.float32, device=DEVICE
        ).view(1, N_CLASSES, 1, 1)

        if COLAB_V7_BLEND_ACTIVE:
            colab_seg = colab_seg.to(DEVICE).half().float()
            colab_cls = colab_cls.to(DEVICE).half().float()
            beta_tensor = torch.as_tensor(
                [CLASS_BETA[name] for name in CLASSES], dtype=torch.float32, device=DEVICE
            ).view(1, N_CLASSES, 1, 1)
            gate_values = [
                -1.0 if CLASS_GATE_THRESHOLDS[name] is None else float(CLASS_GATE_THRESHOLDS[name])
                for name in CLASSES
            ]
            gate_threshold_tensor = torch.as_tensor(
                gate_values, dtype=torch.float32, device=DEVICE
            ).view(1, N_CLASSES)
            probability_combined = (1.0 - beta_tensor) * baseline_probs + beta_tensor * colab_seg
            eps = 1e-4
            base_clipped = baseline_probs.clamp(min=eps, max=1.0 - eps)
            colab_clipped = colab_seg.clamp(min=eps, max=1.0 - eps)
            logit_combined = torch.sigmoid(
                (1.0 - beta_tensor) * torch.logit(base_clipped)
                + beta_tensor * torch.logit(colab_clipped)
            )
            logit_mode_tensor = torch.as_tensor(
                [CLASS_FUSION_MODE[name] == "logit" for name in CLASSES],
                dtype=torch.bool, device=DEVICE
            ).view(1, N_CLASSES, 1, 1)
            combined_probs = torch.where(logit_mode_tensor, logit_combined, probability_combined)
            gate_pass = (colab_cls >= gate_threshold_tensor) | (gate_threshold_tensor < 0)
            del colab_seg, colab_cls, beta_tensor, gate_threshold_tensor
            del probability_combined, base_clipped, colab_clipped, logit_combined, logit_mode_tensor
        else:
            combined_probs = baseline_probs
            gate_pass = torch.ones(
                (len(main_ids), N_CLASSES), dtype=torch.bool, device=DEVICE
            )

        if FREQUENCY_BLEND_ACTIVE and frequency_test_model is not None:
            try:
                frequency_probs, _frequency_class_probs = _frequency_predict_batch(
                    frequency_test_model, main_images, DEVICE,
                    use_tta=(DEVICE.type == "cuda")
                )
                frequency_probs = frequency_probs.to(DEVICE).half().float()
                alpha_tensor = torch.as_tensor(
                    [FREQUENCY_ALPHA_BY_CLASS[name] for name in CLASSES],
                    dtype=torch.float32, device=DEVICE
                ).view(1, N_CLASSES, 1, 1)
                combined_probs = (1.0 - alpha_tensor) * combined_probs + alpha_tensor * frequency_probs
                del frequency_probs, _frequency_class_probs, alpha_tensor
            except Exception as frequency_test_error:
                # Never create a partial mixed candidate: a failure forces explicit V17 fallback.
                print("V18 frequency inference failed; switching to V17-only predictions.")
                print(type(frequency_test_error).__name__, str(frequency_test_error)[:220])
                FREQUENCY_BLEND_ACTIVE = False
                FREQUENCY_ALPHA_BY_CLASS = {name: 0.0 for name in CLASSES}
                FINAL_THRESHOLDS = dict(V17_BASE_THRESHOLDS)
                FREQUENCY_ACCEPTANCE_REASON = "test inference failed; rerun cell for clean V17 fallback"
                raise RuntimeError("Rerun this inference cell; FREQUENCY_BLEND_ACTIVE has been disabled so it will regenerate a clean V17 fallback.") from frequency_test_error

        if not torch.isfinite(combined_probs).all():
            raise RuntimeError("Non-finite values in final blended test probabilities.")
        binary_masks = (
            (combined_probs >= threshold_tensor)
            & gate_pass[:, :, None, None]
        ).to(torch.uint8).cpu().numpy()
        for batch_index in range(len(main_ids)):
            for class_index, class_name in enumerate(CLASSES):
                min_area = int(COMPONENT_MIN_AREAS.get(class_name, 0))
                if min_area > 1:
                    binary_masks[batch_index, class_index] = remove_small_components(
                        binary_masks[batch_index, class_index], min_area
                    ).astype(np.uint8)

        for batch_index, image_id in enumerate(main_ids):
            for class_index, class_name in enumerate(CLASSES):
                prediction_lookup[(image_id, class_name)] = rle_encode_column_major(
                    binary_masks[batch_index, class_index]
                )

        processed_images += len(main_ids)
        if processed_images % 100 == 0 or processed_images == len(TEST_IDS):
            save_test_progress(prediction_lookup, processed_images)

        del p4_probs, p5_probs, p6_probs, mit_probs, baseline_probs, combined_probs, binary_masks, gate_pass
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

if processed_images != len(TEST_IDS):
    raise RuntimeError(f"Test inference incomplete: {processed_images}/{len(TEST_IDS)} images.")
expected_pairs = len(TEST_IDS) * N_CLASSES
if len(prediction_lookup) != expected_pairs:
    raise RuntimeError(f"Prediction coverage mismatch: {len(prediction_lookup)} != {expected_pairs}.")

submission = sample_submission.copy()
submission["mask_rle"] = [
    prediction_lookup[(str(image_id), str(class_name))]
    for image_id, class_name in zip(submission["image_id"], submission["defect_type"])
]
submission["mask_rle"] = submission["mask_rle"].replace("", " ")

FINAL_SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
temporary_submission_path = FINAL_SUBMISSION_PATH.with_name("submission.tmp.csv")
submission.to_csv(temporary_submission_path, index=False)
os.replace(temporary_submission_path, FINAL_SUBMISSION_PATH)
backup_path = OUTPUT_DIR / "submission_surfaceguard_v18_frequency_detail_hybrid.csv"
submission.to_csv(backup_path, index=False)

elapsed_minutes = (time.time() - start_time) / 60.0
print("\nTest inference completed.")
print("Images processed:", processed_images)
print("Submission rows:", len(submission))
print("Elapsed minutes:", round(elapsed_minutes, 2))
print("Saved candidate:", FINAL_SUBMISSION_PATH)
print("Progress file:", progress_path)

check = pd.read_csv(
    FINAL_SUBMISSION_PATH,
    dtype={"ID": str, "image_id": str, "defect_type": str, "mask_rle": str},
    keep_default_na=False,
)
if check.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(f"Wrong submission columns: {check.columns.tolist()}")
if check.shape != (12000, 4):
    raise ValueError(f"Wrong submission shape: {check.shape}")
for column in ["ID", "image_id", "defect_type"]:
    if check[column].tolist() != sample_submission[column].astype(str).tolist():
        raise ValueError(f"Submission template order mismatch in {column}.")
if check.isna().any().any():
    raise ValueError("Null values found in saved submission.")
if check.duplicated().any() or check.duplicated(subset=["image_id", "defect_type"]).any():
    raise ValueError("Duplicate rows or image-class pairs found in saved submission.")
if check["ID"].nunique() != 12000 or check["image_id"].nunique() != 3000:
    raise ValueError("Submission ID/image counts are incorrect.")
if not (check.groupby("image_id").size() == N_CLASSES).all():
    raise ValueError("Not every test image has exactly four class rows.")
class_counts = check["defect_type"].value_counts().reindex(CLASSES, fill_value=0)
if not (class_counts == 3000).all():
    raise ValueError(f"Unexpected class counts:\n{class_counts}")
rle_valid = check["mask_rle"].map(validate_rle)
if not rle_valid.all():
    display(check.loc[~rle_valid].head(10))
    raise ValueError(f"Invalid RLE rows: {int((~rle_valid).sum())}")

empty_masks = int(check["mask_rle"].str.strip().eq("").sum())
nonempty_masks = int(check["mask_rle"].str.strip().ne("").sum())
file_size_mb = FINAL_SUBMISSION_PATH.stat().st_size / (1024 ** 2)
for row_index in [0, 1, 2, 100, 1000, 5000, 11999]:
    image_id = str(check.iloc[row_index]["image_id"])
    class_name = str(check.iloc[row_index]["defect_type"])
    rle = prediction_lookup[(image_id, class_name)]
    decoded = rle_decode_for_check(rle)
    if not validate_rle(rle) or decoded.shape != (HEIGHT, WIDTH):
        raise RuntimeError(f"RLE round-trip check failed at row {row_index}.")

FINAL_REPORT = {
    "status": "PASSED",
    "notebook_version": "v18_frequency_detail_hybrid_evidence_gated",
    "best_previous_public_score": BEST_PUBLIC_SCORE_BASELINE,
    "public_score_goal": 0.68,
    "frequency_model_trained": bool(FREQUENCY_MODEL_AVAILABLE),
    "frequency_training_note": FREQUENCY_TRAINING_NOTE,
    "frequency_blend_active": bool(FREQUENCY_BLEND_ACTIVE),
    "frequency_alpha_by_class": FREQUENCY_ALPHA_BY_CLASS,
    "frequency_acceptance_reason": FREQUENCY_ACCEPTANCE_REASON,
    "frequency_holdout_score": FREQUENCY_HOLDOUT_SCORE,
    "frequency_baseline_holdout_score": FREQUENCY_BASELINE_HOLDOUT_SCORE,
    "colab_v7_checkpoints_available": bool(COLAB_V7_AVAILABLE),
    "colab_v7_auxiliary_models_used_in_test": bool(COLAB_V7_BLEND_ACTIVE),
    "colab_v7_missing_checkpoints": COLAB_V7_MISSING,
    "device": str(DEVICE),
    "gpu_name": GPU_NAME,
    "main_inference_mode": MAIN_INFERENCE_MODE,
    "mit_inference_mode": MIT_INFERENCE_MODE,
    "beta_by_class": CLASS_BETA,
    "fusion_mode_by_class": CLASS_FUSION_MODE,
    "component_min_area_by_class": COMPONENT_MIN_AREAS,
    "gate_thresholds": CLASS_GATE_THRESHOLDS,
    "pixel_thresholds": FINAL_THRESHOLDS,
    "holdout_hybrid_score": float(hybrid_holdout_metrics["weighted_score"]),
    "holdout_hybrid_score_without_component_filter": float(hybrid_holdout_metrics_unfiltered["weighted_score"]),
    "holdout_v13_baseline_score": float(v13_baseline_holdout_metrics["weighted_score"]),
    "full_validation_calibrated_objective_without_component_filter": float(full_validation_score),
    "full_validation_score_with_component_filter": (
        float(full_validation_metrics_with_components["weighted_score"])
        if full_validation_metrics_with_components is not None else None
    ),
    "calibration_score": float(hybrid_calibration_score),
    "metric_contribution_sum_assertion": "passed" if COLAB_V7_AVAILABLE else "not_applicable_fallback",
    "submission_file": str(FINAL_SUBMISSION_PATH),
    "rows": int(len(check)),
    "unique_images": int(check["image_id"].nunique()),
    "class_counts": {str(k): int(v) for k, v in class_counts.items()},
    "empty_masks": empty_masks,
    "nonempty_masks": nonempty_masks,
    "invalid_rle_rows": int((~rle_valid).sum()),
    "null_values": int(check.isna().sum().sum()),
    "duplicate_rows": int(check.duplicated().sum()),
    "file_size_mb": round(file_size_mb, 3),
    "metric": "0.70 foreground Dice + 0.20 false-positive score + 0.10 empty-prediction accuracy",
    "metric_uses_global_pair_denominators": True,
    "training_performed": False,
    "holdout_caveat": "The checkpoints were selected/trained using prior validation labels; holdout is diagnostic, not an untouched independent estimate.",
    "public_score_caveat": "The Kaggle public score is only known after submission; local validation does not guarantee improvement over 0.65777.",
}
FINAL_REPORT_PATH = OUTPUT_DIR / "final_submission_validation_report.json"
with open(FINAL_REPORT_PATH, "w") as f:
    json.dump(FINAL_REPORT, f, indent=2)
FINAL_CLASS_ANALYSIS_PATH = OUTPUT_DIR / "final_validation_per_class_metrics.csv"
pd.DataFrame(hybrid_holdout_metrics["class_metrics"]).T.to_csv(FINAL_CLASS_ANALYSIS_PATH)

print("=" * 70)
print("FINAL SUBMISSION VALIDATION: PASSED")
print("=" * 70)
print("Submission:", FINAL_SUBMISSION_PATH)
print("Rows:", len(check), "| Unique images:", check["image_id"].nunique())
print("Class counts:\n", class_counts)
print("Empty masks:", empty_masks, "| Non-empty masks:", nonempty_masks)
print("Invalid RLE rows:", int((~rle_valid).sum()))
print("Device:", DEVICE, "| GPU:", GPU_NAME)
print("Colab V7 checkpoints available:", COLAB_V7_AVAILABLE)
print("Colab V7 contribution used in test:", COLAB_V7_BLEND_ACTIVE)
print("Hybrid holdout:", round(hybrid_holdout_metrics["weighted_score"], 6))
print("V13 baseline holdout:", round(v13_baseline_holdout_metrics["weighted_score"], 6))
print("Full-validation objective:", round(full_validation_score, 6))
print("Validation report:", FINAL_REPORT_PATH)


FINAL TEST INFERENCE — V18 FREQUENCY-DETAIL HYBRID / V17 FALLBACK
Optional Colab V7 checkpoints available: True
Colab V7 contribution/gate selected: True
Class-aware V13 weights: {'defect_a': {'v4': 0.125, 'v5': 0.25, 'v6': 0.125, 'mit_b4': 0.5}, 'defect_b': {'v4': 0.25, 'v5': 0.125, 'v6': 0.125, 'mit_b4': 0.5}, 'defect_c': {'v4': 0.0, 'v5': 0.07500000298023224, 'v6': 0.25, 'mit_b4': 0.675000011920929}, 'defect_d': {'v4': 0.07500000298023224, 'v5': 0.125, 'v6': 0.25, 'mit_b4': 0.550000011920929}}
Colab V7 beta by class: {'defect_a': 0.30000001192092896, 'defect_b': 0.699999988079071, 'defect_c': 0.0, 'defect_d': 0.20000000149011612}
Colab V7 classifier gates: {'defect_a': 0.7, 'defect_b': 0.8, 'defect_c': 0.2, 'defect_d': 0.35}
Colab V7 fusion modes: {'defect_a': 'probability', 'defect_b': 'probability', 'defect_c': 'probability', 'defect_d': 'logit'}
Main inference mode: four_way_flip_tta
MiT inference mode: four_way_flip_tta
Pixel thresholds: {'defect_a': 0.25999999046325684, 'defect

Test inference: V18 + V17 hybrid:   0%|          | 0/3000 [00:00<?, ?it/s]

/tmp/ipykernel_23/3692865413.py:189: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):



Test inference completed.
Images processed: 3000
Submission rows: 12000
Elapsed minutes: 26.2
Saved candidate: /kaggle/working/submission.csv
Progress file: /kaggle/working/surfaceguard_version18_frequency_detail_hybrid/test_predictions_progress.csv
FINAL SUBMISSION VALIDATION: PASSED
Submission: /kaggle/working/submission.csv
Rows: 12000 | Unique images: 3000
Class counts:
 defect_type
defect_a    3000
defect_b    3000
defect_c    3000
defect_d    3000
Name: count, dtype: int64
Empty masks: 7762 | Non-empty masks: 4238
Invalid RLE rows: 0
Device: cuda:0 | GPU: Tesla T4
Colab V7 checkpoints available: True
Colab V7 contribution used in test: True
Hybrid holdout: 0.724645
V13 baseline holdout: 0.716466
Full-validation objective: 0.735028
Validation report: /kaggle/working/surfaceguard_version18_frequency_detail_hybrid/final_submission_validation_report.json


In [19]:
# Existing V4/V5/V6/MiT training runs remain unchanged. This optional run records only
# the final four-model evaluation; missing secrets or W&B connectivity will not fail submission generation.
try:
    import wandb
    from kaggle_secrets import UserSecretsClient

    api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    if api_key and api_key.strip():
        WANDB_HOST = "https://api.forge.coreweave.com"
        WANDB_ENTITY = "23f3003529-dl-genai-project"
        WANDB_PROJECT = "23f3003529-t32026"

        wandb.login(
            key=api_key.strip(),
            host=WANDB_HOST,
            relogin=True
        )
        del api_key

        final_run = wandb.init(
            entity=WANDB_ENTITY,
            project=WANDB_PROJECT,
            name="SurfaceGuard-Version18-Frequency-Detail-Hybrid",
            job_type="frequency-detail-training-and-ensemble-evaluation",
            tags=["SurfaceGuardAI", "nine-model-candidate", "frequency-detail", "evidence-gated-hybrid"],
            config={
                "models": FINAL_BLEND_CONFIG["models"] + (["ColabV7 A/B/C/D"] if COLAB_V7_AVAILABLE else []),
                "class_aware_weights": FINAL_BLEND_CONFIG["baseline_class_aware_weights"],
                "main_inference_mode": MAIN_INFERENCE_MODE,
                "main_tta_passes": len(MAIN_TTA_TRANSFORMS),
                "mit_b4_inference_mode": MIT_INFERENCE_MODE,
                "version": "v18_frequency_detail_hybrid_evidence_gated",
                "frequency_model_trained": bool(FREQUENCY_MODEL_AVAILABLE),
                "frequency_blend_active": bool(FREQUENCY_BLEND_ACTIVE),
                "frequency_alpha_by_class": FREQUENCY_ALPHA_BY_CLASS,
                "frequency_acceptance_reason": FREQUENCY_ACCEPTANCE_REASON,
                "main_resampling": "bilinear",
                "weight_optimisation": "class_aware_v4_v5_v6_mit_b4_with_fine_0.05_pairwise_refinement_plus_expanded_colab_v7_beta_gate_and_local_logit_fusion",
                "colab_v7_available": bool(COLAB_V7_AVAILABLE),
                "colab_v7_beta_by_class": CLASS_BETA,
                "colab_v7_gate_thresholds": CLASS_GATE_THRESHOLDS,
                "colab_v7_fusion_mode_by_class": CLASS_FUSION_MODE,
                "component_min_area_by_class": COMPONENT_MIN_AREAS,
                "component_min_area_candidate_by_class": COMPONENT_MIN_AREAS_CALIBRATED,
                "mit_resampling": "bilinear",
                "execution_device": str(DEVICE),
                "gpu_name": GPU_NAME,
                "previous_public_baseline": BEST_PUBLIC_SCORE_BASELINE,
                "thresholds": FINAL_THRESHOLDS,
                "target_local_validation_score": LOCAL_VALIDATION_TARGET,
                "public_leaderboard_goal": PUBLIC_SCORE_TARGET
            }
        )
        final_run.summary.update({
            "component_candidate_holdout_score": float(hybrid_holdout_metrics_component_candidate["weighted_score"]),
            "calibration_score": float(hybrid_calibration_score),
            "holdout_score": float(hybrid_holdout_metrics["weighted_score"]),
            "holdout_score_without_component_filter": float(hybrid_holdout_metrics_unfiltered["weighted_score"]),
            "calibration_score": float(hybrid_calibration_score),
            "full_validation_calibrated_objective": float(full_validation_score),
            "submission_rows": int(len(check)),
            "valid_rle_rows": int(rle_valid.sum()),
            "empty_masks": empty_masks,
            "submission_validation": "passed"
        })
        final_run.finish()
        print("W&B final evaluation run:", final_run.url)
    else:
        print("W&B skipped: WANDB_API_KEY secret is empty.")
except Exception as exc:
    print("W&B logging skipped; submission generation is unaffected.")
    print(type(exc).__name__ + ":", str(exc)[:300])


wandb: Updated settings file /root/.config/wandb/settings
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: No netrc file found, creating one.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: setting up run euq3s06e
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261110_163844-euq3s06e
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run SurfaceGuard-Version18-Frequency-Detail-Hybrid
wandb: ⭐️ View project at https://app.forge.coreweave.com/23f3003529-dl-genai-project/23f3003529-t32026
wandb: 🚀 View run at https://app.forge.core

W&B final evaluation run: https://app.forge.coreweave.com/23f3003529-dl-genai-project/23f3003529-t32026/runs/euq3s06e


In [20]:
import os
import shutil
from pathlib import Path
from IPython.display import FileLink, display

print("SURFACEGUARD AI — VERSION 18 FINAL ARTIFACTS")
print("=" * 60)

# The inference cell creates this validated candidate inside OUTPUT_DIR.
CANDIDATE_SUBMISSION_PATH = OUTPUT_DIR / "submission_surfaceguard_v18_frequency_detail_hybrid.csv"
KAGGLE_SUBMISSION_PATH = Path("/kaggle/working/submission.csv")

if not CANDIDATE_SUBMISSION_PATH.is_file():
    raise FileNotFoundError(
        "The final candidate submission was not created by the inference cell: "
        f"{CANDIDATE_SUBMISSION_PATH}"
    )

# Copy atomically so Kaggle always sees the exact required filename at the root.
TEMP_SUBMISSION_COPY = KAGGLE_SUBMISSION_PATH.with_name("submission.copy.tmp.csv")
shutil.copy2(CANDIDATE_SUBMISSION_PATH, TEMP_SUBMISSION_COPY)
os.replace(TEMP_SUBMISSION_COPY, KAGGLE_SUBMISSION_PATH)

# Verify the copied root-level file, not just the source candidate.
root_check = pd.read_csv(
    KAGGLE_SUBMISSION_PATH,
    dtype={"ID": str, "image_id": str, "defect_type": str, "mask_rle": str},
    keep_default_na=False
)
candidate_check = pd.read_csv(
    CANDIDATE_SUBMISSION_PATH,
    dtype={"ID": str, "image_id": str, "defect_type": str, "mask_rle": str},
    keep_default_na=False
)

if not root_check.equals(candidate_check):
    raise ValueError("Root submission.csv differs from the validated candidate submission.")

if root_check.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(f"Copied submission has incorrect columns: {root_check.columns.tolist()}")
if root_check.shape != (12000, 4):
    raise ValueError(f"Copied submission has incorrect shape: {root_check.shape}")
if root_check.isna().any().any():
    raise ValueError("Copied submission contains null values.")
if root_check.duplicated(subset=["image_id", "defect_type"]).any():
    raise ValueError("Copied submission contains duplicate image-class pairs.")
if root_check["ID"].tolist() != sample_submission["ID"].astype(str).tolist():
    raise ValueError("Copied submission ID order differs from the sample template.")
if root_check["image_id"].tolist() != sample_submission["image_id"].astype(str).tolist():
    raise ValueError("Copied submission image order differs from the sample template.")
if root_check["defect_type"].tolist() != sample_submission["defect_type"].astype(str).tolist():
    raise ValueError("Copied submission class order differs from the sample template.")
if not root_check["mask_rle"].map(validate_rle).all():
    raise ValueError("Copied submission contains invalid RLE masks.")

print("Kaggle submission file copied and verified:", KAGGLE_SUBMISSION_PATH)
print("Rows:", len(root_check))
print("File size (MB):", round(KAGGLE_SUBMISSION_PATH.stat().st_size / (1024 ** 2), 3))

final_artifacts = [
    KAGGLE_SUBMISSION_PATH,
    CANDIDATE_SUBMISSION_PATH,
    FINAL_REPORT_PATH,
    FINAL_CLASS_ANALYSIS_PATH,
    CONFIG_PATH,
    OUTPUT_DIR / "global_four_model_weight_search.csv",
    OUTPUT_DIR / "fine_four_model_weight_search.csv",
    OUTPUT_DIR / "class_aware_weight_selection.csv",
    OUTPUT_DIR / "hybrid_beta_gate_search.csv",
    OUTPUT_DIR / "hybrid_class_selection.csv",
    OUTPUT_DIR / "component_min_area_search.csv",
    OUTPUT_DIR / "test_predictions_progress.csv",
    FREQUENCY_MODEL_PATH if FREQUENCY_MODEL_AVAILABLE else None,
    OUTPUT_DIR / "v18_frequency_training_history.csv" if FREQUENCY_MODEL_AVAILABLE else None,
    OUTPUT_DIR / "v18_frequency_alpha_threshold_search.csv" if (OUTPUT_DIR / "v18_frequency_alpha_threshold_search.csv").is_file() else None,
]

print("\nAVAILABLE OUTPUT ARTIFACTS")
for artifact in final_artifacts:
    if artifact is None:
        continue
    artifact = Path(artifact)
    if not artifact.is_file():
        raise FileNotFoundError(f"Expected output is missing: {artifact}")
    print("\n", artifact.name, "|", round(artifact.stat().st_size / (1024 ** 2), 3), "MB")
    display(FileLink(str(artifact)))

print("\nV18 FINAL NOTEBOOK PIPELINE COMPLETED.")
print("Colab V7 auxiliary models included:", COLAB_V7_AVAILABLE)
print("V18 frequency model trained:", FREQUENCY_MODEL_AVAILABLE)
print("V18 frequency blend active:", FREQUENCY_BLEND_ACTIVE)
print("V18 acceptance decision:", FREQUENCY_ACCEPTANCE_REASON)
print("If False, this was the V13 fallback rather than the new hybrid experiment.")
print("The V18 frequency/detail model training status is recorded in the validation report. The original eight checkpoints were not modified.")


SURFACEGUARD AI — VERSION 18 FINAL ARTIFACTS
Kaggle submission file copied and verified: /kaggle/working/submission.csv
Rows: 12000
File size (MB): 2.651

AVAILABLE OUTPUT ARTIFACTS

 submission.csv | 2.651 MB


/kaggle/working/submission.csv


 submission_surfaceguard_v18_frequency_detail_hybrid.csv | 2.651 MB


/kaggle/working/surfaceguard_version18_frequency_detail_hybrid/submission_surfaceguard_v18_frequency_detail_hybrid.csv


 final_submission_validation_report.json | 0.003 MB


/kaggle/working/surfaceguard_version18_frequency_detail_hybrid/final_submission_validation_report.json

... (21 outputs left out)

/kaggle/working/surfaceguard_version18_frequency_detail_hybrid/v18_frequency_training_history.csv


 v18_frequency_alpha_threshold_search.csv | 0.0 MB


/kaggle/working/surfaceguard_version18_frequency_detail_hybrid/v18_frequency_alpha_threshold_search.csv


V18 FINAL NOTEBOOK PIPELINE COMPLETED.
Colab V7 auxiliary models included: True
V18 frequency model trained: True
V18 frequency blend active: True
V18 acceptance decision: accepted: holdout gain=0.009530; defect_b Dice delta=0.000000
If False, this was the V13 fallback rather than the new hybrid experiment.
The V18 frequency/detail model training status is recorded in the validation report. The original eight checkpoints were not modified.


In [21]:
import pandas as pd
from pathlib import Path

files_to_repair = [
    Path("/kaggle/working/submission.csv"),
    OUTPUT_DIR / "submission_surfaceguard_v18_frequency_detail_hybrid.csv"
]

for path in files_to_repair:

    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False
    )

    # Same correction used in your previously accepted notebook
    original_empty_count = int(
        df["mask_rle"].eq("").sum()
    )

    df["mask_rle"] = df["mask_rle"].replace("", " ")

    df.to_csv(path, index=False)

    # Verify the saved CSV
    check = pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False
    )

    assert len(check) == 12000
    assert check["mask_rle"].eq("").sum() == 0
    assert check.isna().sum().sum() == 0
    assert check["mask_rle"].map(validate_rle).all()

    print("\nREPAIR PASSED:", path.name)
    print("Previously empty fields:", original_empty_count)
    print(
        "Empty masks preserved:",
        int(check["mask_rle"].str.strip().eq("").sum())
    )
    print("Raw blank fields:", int(check["mask_rle"].eq("").sum()))
    print("Null values:", int(check.isna().sum().sum()))

print("\nV18 SUBMISSION FILES REPAIRED.")



REPAIR PASSED: submission.csv
Previously empty fields: 0
Empty masks preserved: 7762
Raw blank fields: 0
Null values: 0

REPAIR PASSED: submission_surfaceguard_v18_frequency_detail_hybrid.csv
Previously empty fields: 0
Empty masks preserved: 7762
Raw blank fields: 0
Null values: 0

V18 SUBMISSION FILES REPAIRED.
